## EPA Analysis Report and Slideshow
The purpose of this script is to generate the report and slideshow to present to the coaching staff

In [1]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# By default this runs on the synthetic sample in data/TU_Games_synthetic so the
# pipeline can be reproduced from a fresh clone. The real Hudl exports are
# private to Trinity Athletics and are not in the repo.
#
# To run on the real data in Colab:
#   from google.colab import drive; drive.mount('/content/drive')
#   os.environ['TUFB_DATA_DIR']       = '/content/drive/MyDrive/TUFB_EPA/TU_Games'
#   os.environ['TUFB_PROCESSED_FILE'] = '/content/drive/MyDrive/TUFB_EPA/TUFB_EPA_Analysis_FULL_DATA.xlsx'
import os
from pathlib import Path

REPO_ROOT      = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR       = Path(os.environ.get('TUFB_DATA_DIR', REPO_ROOT / 'data' / 'TU_Games_synthetic'))
PROCESSED_FILE = Path(os.environ.get('TUFB_PROCESSED_FILE',
                      REPO_ROOT / 'data' / 'processed' / 'TUFB_EPA_Analysis_FULL_DATA.xlsx'))
FIG_DIR        = Path(os.environ.get('TUFB_FIG_DIR', REPO_ROOT / 'figures'))
DOCS_DIR       = Path(os.environ.get('TUFB_DOCS_DIR', REPO_ROOT / 'docs'))
for _d in (PROCESSED_FILE.parent, FIG_DIR, DOCS_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print(f"Game files : {DATA_DIR}")
print(f"Processed  : {PROCESSED_FILE}")

Game files : /mnt/user-data/uploads/TU_Games
Processed  : /root/work/real/TUFB_EPA_Analysis_FULL_DATA.xlsx


## Report Builder
This builds a report as an html file and an ipynb file

In [2]:
# -*- coding: utf-8 -*-
"""
epa_tufb_report_builder.py
==========================
Run this script in Google Colab to execute the full TU offensive EPA
analysis and render a single self-contained HTML report — the Python
equivalent of R Markdown.

How it works:
  1. REPORT_CELLS defines the report as an ordered list of markdown and
     code cells (identical pattern to a .Rmd file with narrative + chunks).
  2. build_report() assembles those cells into a real .ipynb using nbformat,
     executes it via nbconvert ExecutePreprocessor, then converts the
     executed notebook to a styled, self-contained HTML file.
  3. The HTML is saved to your Google Drive alongside the source data.

Usage:
  Simply run this file in Colab. The last line calls build_report().
  Output: docs/TU_EPA_Offensive_Report.html
"""

# ── Report cell registry ───────────────────────────────────────────────────────
# Each entry is a dict with keys:
#   'type' : 'markdown' | 'code'
#   'source': the cell content as a string
# Markdown cells become formatted narrative in the HTML output.
# Code cells are executed and their outputs (plots, tables, Plotly charts)
# are embedded inline.
# ──────────────────────────────────────────────────────────────────────────────

REPORT_CELLS = []

def md(source):
    """Register a markdown narrative cell."""
    REPORT_CELLS.append({'type': 'markdown', 'source': source})

def code(source):
    """Register an executable code cell."""
    REPORT_CELLS.append({'type': 'code', 'source': source})


# ══════════════════════════════════════════════════════════════════════════════
# COVER & EXECUTIVE SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

md("""
# Trinity University Football
## Offensive EPA Analysis Report

---

## Executive Summary

### What We Set Out to Do

This project brings an advanced analytical framework to Trinity University Football.
The goal is straightforward: **move beyond box score statistics and understand the
true value of every offensive play we run.**

Traditional metrics — total yards, yards per carry, completion percentage — tell us
*what happened*, but strip away the context that determines whether a play was actually
*good*. A 4-yard gain on 3rd-and-3 is a success. A 4-yard gain on 3rd-and-10 is a
failure. Averages cannot tell the difference. EPA can.

---

### Why Expected Points Added (EPA)?

**Expected Points (EP)** is the foundation of this model. At any point in a game,
given the down, distance, and field position, we can estimate how many points a team
is *expected* to score on the current drive. EPA measures how much a single play moves
that expectation — up or down.

> A play with **EPA > 0** helped us. It moved the chains, created a favorable
> down-and-distance, or threatened scoring.
> A play with **EPA < 0** hurt us — it put us behind schedule, forced a tough 3rd
> down, or stalled a drive.

This framework lets us:

- **Compare plays across all situations** on a level playing field
- **Identify schematic strengths and weaknesses** that yardage totals obscure
- **Quantify the cost of bad plays**, not just celebrate the good ones
- **Understand situational efficiency** — 3rd down, red zone, two-minute drill — with precision

**Model baseline: +0.028 EPA per play.** Beating this consistently is what separates
productive offensive series from stalled drives.

---

### Questions This Report Answers

1. Are we generating positive value play-to-play?
2. Where situationally do we succeed — and where do we fail?
3. Are we running the right plays in the right situations?
4. Does our scheme (personnel, formation) produce value?
5. Are we generating explosive plays, and from where?
6. How do our drives actually unfold?
7. What separates our wins from our losses?

---
""")

# ══════════════════════════════════════════════════════════════════════════════
# SETUP
# ══════════════════════════════════════════════════════════════════════════════

md("## Setup — Libraries & Data Load")

code("""
import os
import re
import glob
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import to_html

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# ── Global style constants ─────────────────────────────────────────────────
TU_BLUE        = '#002868'
TU_GOLD        = '#C5960C'
WIN_COLOR      = '#1D9E75'
LOSS_COLOR     = '#C0392B'
RED            = '#C0392B'
LIGHT_GRAY     = '#F5F5F5'
MODEL_BASELINE = 0.028

# df and tu_df are injected from the calling session by build_report().
# No file I/O needed — the data is already in memory.
print(f"df:    {len(df):,} rows × {df.shape[1]} columns")
print(f"tu_df: {len(tu_df):,} rows")
""")

# ══════════════════════════════════════════════════════════════════════════════
# DATA RESHAPING
# ══════════════════════════════════════════════════════════════════════════════

md("""
## Data Reshaping

Normalize team and opponent naming so every row in the dataset carries a
consistent `team_name` and `opp_name`, derived from offensive play rows where
`odk == 'o'`.
""")

code("""
game_names = (
    df[df["odk"] == "o"]
    .groupby("game_id")[["team_name", "opp_name"]]
    .first()
    .reset_index()
)
df = df.drop(columns=["team_name", "opp_name"])
df = df.merge(game_names, on="game_id", how="left")

# Build Trinity-only dataframe
tu_df = (
    df[df['team_name'] == 'TU']
    .sort_values(['game_id', 'play'])
    .reset_index(drop=True)
)
print(f"TU rows: {len(tu_df):,}")

print("\\n--- EP by odk ---")
print(tu_df.groupby('odk')['ep'].describe().round(3))
print("\\n--- EPA by odk ---")
print(tu_df.groupby('odk')['epa'].describe().round(3))
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — EPA BY DOWN
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 1 · Is Our Offense Generating Positive Value?

### EPA by Down

**What it shows:** Average EPA per play on 1st through 4th down compared to the
model baseline (+0.028). This is the entry-level question — are we playing
*ahead of schedule* or constantly putting ourselves in difficult situations?

- **1st down** → are we winning the line of scrimmage early?
- **2nd down** → are we setting up manageable 3rd downs?
- **3rd down** → are drives extending or dying?

> **So what?** If 1st down EPA is below baseline, our opening play design needs
> attention — play-action, RPO, and early motion can prevent 2nd-and-long from
> becoming the default. Negative 3rd down EPA feeds directly into the next section.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()

down_df = (
    off_df[off_df['dn'].isin([1, 2, 3, 4])]
    .groupby('dn')['epa']
    .agg(['mean', 'sem', 'count'])
    .reset_index()
)
down_df['dn_label'] = down_df['dn'].map(
    {1: '1st Down', 2: '2nd Down', 3: '3rd Down', 4: '4th Down'}
)

fig, ax = plt.subplots(figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(
    down_df['dn_label'], down_df['mean'],
    color=[TU_BLUE if v >= 0 else RED for v in down_df['mean']],
    edgecolor='white', linewidth=1.2, width=0.5
)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=9, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, down_df.itertuples()):
    label_y = row.mean + 0.008 if row.mean >= 0 else row.mean - 0.008
    va      = 'bottom' if row.mean >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2, label_y,
            f'μ={row.mean:+.3f}', ha='center', va=va,
            fontsize=11, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x() + bar.get_width() / 2, ax.get_ylim()[0] - 0.025,
            f'n={row.count}', ha='center', va='bottom', fontsize=9, color='gray')

ax.set_title('Trinity Offensive EPA by Down', fontsize=15,
             fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Down', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#CCCCCC')
ax.set_ylim(bottom=-0.15, top=0.15)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — EPA BY OPPONENT
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 2 · EPA vs. Our Opponents

**What it shows:** Average offensive EPA per play for every game, filterable by
season and by individual opponent. Color encodes win (green) vs. loss (red).

> **So what?** Identify opponents against whom we consistently struggle — their
> scheme warrants targeted pre-game adjustments. Year-over-year clustering of
> negative-EPA games signals a systemic issue, not just a bad week.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()

game_meta = (
    off_df.groupby('game_id')
    .agg(opp_name=('opp_name','first'), game_date=('game_date','first'),
         win=('win','first'), mean_epa=('epa','mean'),
         sem_epa=('epa','sem'), n_plays=('epa','count'))
    .reset_index()
)
game_meta['game_date_parsed'] = pd.to_datetime(game_meta['game_date'], format='%m/%d/%Y')
game_meta['year'] = game_meta['game_date_parsed'].dt.year
game_meta['label'] = game_meta.apply(
    lambda r: f"{r['opp_name']} ({r['year']} {'W' if r['win']==1 else 'L'})", axis=1
)

all_years    = sorted(game_meta['year'].unique())
all_opps     = sorted(game_meta['opp_name'].unique())
overall_mean = off_df['epa'].mean()

fig       = go.Figure()
trace_map = {}

def make_trace(subset, sort_by, ascending, scope_key, visible):
    subset = subset.sort_values(sort_by, ascending=ascending).reset_index(drop=True)
    colors = [WIN_COLOR if w == 1 else LOSS_COLOR for w in subset['win']]
    hover  = [f"<b>{r['label']}</b><br>EPA/play: {r['mean_epa']:+.3f}<br>Plays: {int(r['n_plays'])}"
              for _, r in subset.iterrows()]
    idx = len(fig.data)
    fig.add_trace(go.Bar(
        x=subset['mean_epa'], y=subset['label'], orientation='h',
        marker_color=colors, marker_line_color='white', marker_line_width=1.2,
        text=hover, hovertemplate='%{text}<extra></extra>',
        visible=visible, name=scope_key, showlegend=False,
    ))
    trace_map[scope_key] = idx

season_scopes = ['All'] + [str(y) for y in all_years]
for scope in season_scopes:
    sub = game_meta if scope == 'All' else game_meta[game_meta['year'] == int(scope)]
    make_trace(sub.copy(), 'mean_epa', True, f'season_{scope}', visible=(scope == 'All'))

for opp in all_opps:
    make_trace(game_meta[game_meta['opp_name'] == opp].copy(),
               'game_date_parsed', True, f'opp_{opp}', visible=False)

total_traces = len(fig.data)
make_vis = lambda k: [i == trace_map[k] for i in range(total_traces)]

season_buttons = []
for scope in season_scopes:
    sub = game_meta if scope == 'All' else game_meta[game_meta['year'] == int(scope)]
    nw, nl = int(sub['win'].sum()), len(sub) - int(sub['win'].sum())
    season_buttons.append(dict(
        label=scope, method='update',
        args=[{'visible': make_vis(f'season_{scope}')},
              {'title.text': f'<b>TU Offensive EPA per Play by Opponent</b> — '
               f'{"All Seasons" if scope=="All" else scope}<br>'
               f'<sup>{len(sub)} games · {nw}W–{nl}L · Avg: {sub["mean_epa"].mean():+.3f}</sup>',
               'yaxis.autorange': True}]
    ))

opp_buttons = []
for opp in all_opps:
    sub = game_meta[game_meta['opp_name'] == opp].sort_values('game_date_parsed')
    nw, nl = int(sub['win'].sum()), len(sub) - int(sub['win'].sum())
    opp_buttons.append(dict(
        label=opp, method='update',
        args=[{'visible': make_vis(f'opp_{opp}')},
              {'title.text': f'<b>TU vs. {opp} — All Seasons</b><br>'
               f'<sup>{len(sub)} games · {nw}W–{nl}L · Avg: {sub["mean_epa"].mean():+.3f}</sup>',
               'yaxis.autorange': True}]
    ))

fig.add_vline(x=overall_mean, line_dash='dash', line_color=TU_BLUE, line_width=1.5,
              opacity=0.6, annotation_text=f'Season avg: {overall_mean:+.3f}',
              annotation_position='top', annotation_font=dict(color=TU_BLUE, size=10))
fig.add_vline(x=0, line_dash='dot', line_color='black', line_width=1, opacity=0.35)
fig.add_vline(x=MODEL_BASELINE, line_dash='dashdot', line_color=TU_GOLD, line_width=2,
              opacity=0.8, annotation_text='Model Baseline (+0.028)',
              annotation_position='bottom', annotation_font=dict(color=TU_GOLD, size=10))

n_all = len(game_meta)
nw_all, nl_all = int(game_meta['win'].sum()), n_all - int(game_meta['win'].sum())
fig.update_layout(
    title=dict(
        text=f'<b>TU Offensive EPA per Play by Opponent</b> — All Seasons<br>'
             f'<sup>{n_all} games · {nw_all}W–{nl_all}L · Avg: {game_meta["mean_epa"].mean():+.3f}</sup>',
        font=dict(size=16, color=TU_BLUE), x=0.5, xanchor='center'),
    xaxis=dict(title='Average EPA per Play', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD'),
    yaxis=dict(tickfont=dict(color=TU_BLUE, size=11), gridcolor='#DDDDDD', autorange=True),
    updatemenus=[
        dict(type='buttons', direction='right', showactive=True,
             x=0.5, xanchor='center', y=1.10, yanchor='top',
             buttons=season_buttons, bgcolor='white', bordercolor='#CCCCCC',
             font=dict(color=TU_BLUE, size=11)),
        dict(type='dropdown', direction='down', showactive=True,
             x=0.5, xanchor='center', y=1.03, yanchor='top',
             buttons=opp_buttons, bgcolor='white', bordercolor='#CCCCCC',
             font=dict(color=TU_BLUE, size=11)),
    ],
    annotations=[
        dict(text='Season:', showarrow=False, x=0.13, xanchor='right',
             y=1.115, yanchor='top', xref='paper', yref='paper',
             font=dict(size=10, color='#888888')),
        dict(text='Opponent:', showarrow=False, x=0.13, xanchor='right',
             y=1.045, yanchor='top', xref='paper', yref='paper',
             font=dict(size=10, color='#888888')),
    ],
    plot_bgcolor=LIGHT_GRAY, paper_bgcolor=LIGHT_GRAY, hovermode='closest',
    height=max(500, n_all * 28), margin=dict(t=160, b=60, l=200, r=80),
)
fig.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — 3RD DOWN PROBLEM
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 3 · The 3rd Down Problem

**What it shows:** 3rd down EPA and conversion rate sliced by distance-to-go bucket.
Left panel: EPA distribution (box plot). Right panel: conversion rate vs. average EPA.

| Bucket | Distance | Interpretation |
|---|---|---|
| Short | ≤3 yards | Must-convert — we should win these |
| Medium | 4–7 yards | Execution dependent |
| Standard | 8–10 yards | Difficult — requires a chunk gain |
| Long | 11+ yards | Damage control |

> **So what?** Short 3rd downs should convert at 60–70%+. If they don't, short-yardage
> personnel and play design need dedicated practice time. Standard and Long 3rd downs
> are drive-killers — minimizing their *frequency* (through better 1st/2nd down
> execution) is often more valuable than the play call once you're in them.
""")

code("""
third_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) &
    (tu_df['dist'].notna()) & (tu_df['dn'] == 3)
].copy()

bins   = [0, 3, 7, 10, float('inf')]
labels = ['Short\\n(≤3 yds)', 'Medium\\n(4–7 yds)',
          'Standard\\n(8–10 yds)', 'Long\\n(11+ yds)']
third_df['dist_bucket'] = pd.cut(third_df['dist'], bins=bins,
                                  labels=labels, include_lowest=True)

convert_results = ['complete', 'rush', 'scramble', 'complete, td', 'rush, td']
third_df['converted'] = (
    third_df['result'].str.strip().str.lower().isin(convert_results) &
    (third_df['gn_ls'] >= third_df['dist'])
)

bucket_df = (
    third_df.groupby('dist_bucket', observed=True)
    .agg(mean_epa=('epa','mean'), sem_epa=('epa','sem'),
         count=('epa','count'), conv_rate=('converted','mean'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(22, 10), gridspec_kw={'width_ratios': [1.2, 1]})
fig.patch.set_facecolor(LIGHT_GRAY)

ax1 = axes[0]
ax1.set_facecolor(LIGHT_GRAY)
bucket_colors = {
    b: (TU_BLUE if bucket_df.loc[bucket_df['dist_bucket']==b,'mean_epa'].values[0] >= 0
        else RED)
    for b in labels
}
sns.boxplot(data=third_df, x='dist_bucket', y='epa', order=labels,
            palette=bucket_colors, width=0.45, linewidth=1.3,
            flierprops=dict(marker='o', markersize=3, alpha=0.3,
                            linestyle='none', markeredgewidth=0), ax=ax1)

for i, bucket in enumerate(labels):
    subset = third_df[third_df['dist_bucket'] == bucket]['epa']
    if len(subset) == 0:
        continue
    mean_val    = subset.mean()
    Q3          = subset.quantile(0.75)
    whisker_top = subset[subset <= Q3 + 1.5*(Q3 - subset.quantile(0.25))].max()
    ax1.text(i, whisker_top + 0.12, f'μ={mean_val:+.3f}', ha='center', va='bottom',
             fontsize=10, fontweight='bold',
             color=TU_BLUE if mean_val >= 0 else RED)
    ax1.text(i, ax1.get_ylim()[0] + 0.1, f'n={len(subset)}', ha='center',
             va='bottom', fontsize=9, color='gray')

ax1.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax1.set_title('3rd Down EPA Distribution\\nby Distance Bucket',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=10)
ax1.set_xlabel('Distance to Go', fontsize=11, color=TU_BLUE)
ax1.set_ylabel('EPA per Play', fontsize=11, color=TU_BLUE)
ax1.tick_params(colors=TU_BLUE)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
x, width = range(len(bucket_df)), 0.35
bars1 = ax2.bar([i - width/2 for i in x], bucket_df['conv_rate']*100, width,
                color=TU_GOLD, edgecolor='white', linewidth=1.2, label='Conversion Rate (%)')
bars2 = ax2.bar([i + width/2 for i in x], bucket_df['mean_epa'], width,
                color=[TU_BLUE if v >= 0 else RED for v in bucket_df['mean_epa']],
                edgecolor='white', linewidth=1.2, label='Avg EPA')

for bar, row in zip(bars1, bucket_df.itertuples()):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{row.conv_rate*100:.1f}%', ha='center', va='bottom',
             fontsize=9, fontweight='bold', color=TU_GOLD)

for bar, row in zip(bars2, bucket_df.itertuples()):
    label_y = row.mean_epa + 0.3 if row.mean_epa >= 0 else row.mean_epa - 0.3
    ax2.text(bar.get_x()+bar.get_width()/2, label_y,
             f'μ={row.mean_epa:+.3f}', ha='center',
             va='bottom' if row.mean_epa >= 0 else 'top',
             fontsize=9, fontweight='bold',
             color=TU_BLUE if row.mean_epa >= 0 else RED)

ax2.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax2.set_xticks(list(x))
ax2.set_xticklabels(bucket_df['dist_bucket'], color=TU_BLUE)
ax2.set_title('3rd Down Conversion Rate &\\nAvg EPA by Distance',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=10)
ax2.set_xlabel('Distance to Go', fontsize=11, color=TU_BLUE)
ax2.tick_params(colors=TU_BLUE)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')
ax2.legend(fontsize=9, framealpha=0.6, edgecolor='#CCCCCC')

fig.suptitle(
    f'TU 3rd Down Conversion Efficiency\\n'
    f'Overall conversion rate: {third_df["converted"].mean()*100:.1f}%  |  '
    f'Overall avg EPA: {third_df["epa"].mean():+.3f}',
    fontsize=14, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — 3RD DOWN RUN VS PASS
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 4 · 3rd Down: Run vs. Pass by Distance

**What it shows:** Mean EPA and play volume for runs vs. passes on 3rd down,
broken out by distance bucket.

> **So what?** If pass EPA is significantly higher than run EPA across all
> distances, 3rd-down run calls are not converting and should be limited to
> short-yardage only. Watch the volume panel for predictability — if we pass
> 90%+ on 3rd-and-long, defenses already know what's coming.
""")

code("""
third_down = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['dn'] == 3) &
    (tu_df['play_type'].isin(['run', 'pass'])) &
    (tu_df['epa'].notna()) & (tu_df['dist'].notna())
].copy()

def dist_bucket(d):
    if d <= 3:   return '1–3\\n(Short)'
    elif d <= 6:  return '4–6\\n(Medium)'
    elif d <= 10: return '7–10\\n(Long)'
    else:         return '11+\\n(Very Long)'

bucket_order = ['1–3\\n(Short)', '4–6\\n(Medium)', '7–10\\n(Long)', '11+\\n(Very Long)']
third_down['dist_bucket'] = pd.Categorical(
    third_down['dist'].apply(dist_bucket), categories=bucket_order, ordered=True
)

summary = (
    third_down.groupby(['dist_bucket', 'play_type'], observed=True)
    .agg(mean_epa=('epa','mean'), count=('epa','count'), sem=('epa', lambda x: x.sem()))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
PLAY_COLOR = {'run': TU_BLUE, 'pass': TU_GOLD}
x, bar_width = np.arange(len(bucket_order)), 0.45

for ax, metric, ylabel, title_suffix in zip(
    axes, ['mean_epa', 'count'],
    ['Average EPA per Play', 'Number of Plays'],
    ['Mean EPA by Distance Bucket', 'Play Volume by Distance Bucket']
):
    ax.set_facecolor(LIGHT_GRAY)
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#CCCCCC')

    for play_type, offset in [('run', -bar_width/2), ('pass', bar_width/2)]:
        data = summary[summary['play_type'] == play_type].set_index('dist_bucket').reindex(bucket_order)
        vals = data[metric].values.astype(float)
        color = PLAY_COLOR[play_type]

        bars = ax.bar(x + offset, vals, bar_width, color=color,
                      edgecolor='white', linewidth=1.2, label=play_type.capitalize())

        for bar_obj, val in zip(bars, vals):
            if np.isnan(val):
                continue
            if metric == 'mean_epa':
                ax.text(bar_obj.get_x()+bar_obj.get_width()/2,
                        val + 0.01 if val >= 0 else val - 0.01,
                        f'μ={val:+.3f}', ha='center',
                        va='bottom' if val >= 0 else 'top',
                        fontsize=9, fontweight='bold',
                        color=RED if val < 0 else color)
            else:
                ax.text(bar_obj.get_x()+bar_obj.get_width()/2,
                        bar_obj.get_height()+0.5, f'{int(val)}',
                        ha='center', va='bottom', fontsize=9,
                        fontweight='bold', color=color)

    if metric == 'mean_epa':
        ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(bucket_order, fontsize=10, color=TU_BLUE)
    ax.tick_params(colors=TU_BLUE)
    ax.set_xlabel('Distance to Go', fontsize=12, color=TU_BLUE)
    ax.set_ylabel(ylabel, fontsize=12, color=TU_BLUE)
    ax.set_title(f'3rd Down {title_suffix}', fontsize=13,
                 fontweight='bold', color=TU_BLUE, pad=12)
    ax.legend(handles=[mpatches.Patch(color=TU_BLUE, label='Run'),
                        mpatches.Patch(color=TU_GOLD, label='Pass')],
              fontsize=10, framealpha=0.6, edgecolor='#CCCCCC')

total_runs   = int(summary[summary['play_type']=='run']['count'].sum())
total_passes = int(summary[summary['play_type']=='pass']['count'].sum())
fig.suptitle(f'Trinity 3rd Down — Run vs. Pass EPA by Distance\\nRuns: {total_runs}  |  Passes: {total_passes}',
             fontsize=14, fontweight='bold', color=TU_BLUE, y=1.01)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — RUN VS PASS EPA BY DOWN
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 5 · Run vs. Pass EPA by Down

**What it shows:** Mean EPA and play volume for runs vs. passes on 1st, 2nd,
and 3rd down across the full dataset.

> **So what?** 1st down run EPA is the single best indicator of our offensive
> foundation. If we cannot generate positive EPA running on 1st down, we play
> from behind all game. The volume panel reveals whether our play-calling
> philosophy matches what the EPA data supports.
""")

code("""
rp_down = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['dn'].isin([1, 2, 3])) &
    (tu_df['play_type'].isin(['run', 'pass'])) & (tu_df['epa'].notna())
].copy()

down_order  = [1, 2, 3]
down_labels = ['1st Down', '2nd Down', '3rd Down']

summary = (
    rp_down.groupby(['dn', 'play_type'])
    .agg(mean_epa=('epa','mean'), sem=('epa','sem'), count=('epa','count'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
PLAY_COLOR = {'run': TU_BLUE, 'pass': TU_GOLD}
x, bar_width = np.arange(len(down_order)), 0.38

ax1 = axes[0]
ax1.set_facecolor(LIGHT_GRAY)
for play_type, offset in [('run', -bar_width/2), ('pass', bar_width/2)]:
    data  = summary[summary['play_type']==play_type].set_index('dn').reindex(down_order)
    vals  = data['mean_epa'].values.astype(float)
    color = PLAY_COLOR[play_type]
    bars  = ax1.bar(x + offset, vals, bar_width, color=color,
                    edgecolor='white', linewidth=1.2)
    for bar_obj, val in zip(bars, vals):
        if np.isnan(val): continue
        ax1.text(bar_obj.get_x()+bar_obj.get_width()/2,
                 val + 0.012 if val >= 0 else val - 0.01,
                 f'μ={val:+.3f}', ha='center',
                 va='bottom' if val >= 0 else 'top',
                 fontsize=9, fontweight='bold',
                 color=RED if val < 0 else color)

ax1.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax1.axhline(MODEL_BASELINE, color='#888888', linewidth=1.3, linestyle='-.',
            alpha=0.75, label='Model Baseline (+0.028)')
ax1.set_xticks(x); ax1.set_xticklabels(down_labels, fontsize=11, color=TU_BLUE)
ax1.tick_params(colors=TU_BLUE)
ax1.set_xlabel('Down', fontsize=12, color=TU_BLUE)
ax1.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax1.set_title('Run vs. Pass Mean EPA by Down', fontsize=13,
              fontweight='bold', color=TU_BLUE, pad=12)
ax1.legend(handles=[mpatches.Patch(color=TU_BLUE, label='Run'),
                     mpatches.Patch(color=TU_GOLD, label='Pass'),
                     mpatches.Patch(color='#888888', label='Model Baseline (+0.028)')],
           fontsize=10, framealpha=0.6, edgecolor='#CCCCCC')
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
for play_type, offset in [('run', -bar_width/2), ('pass', bar_width/2)]:
    data  = summary[summary['play_type']==play_type].set_index('dn').reindex(down_order)
    vals  = data['count'].values.astype(float)
    color = PLAY_COLOR[play_type]
    bars  = ax2.bar(x + offset, vals, bar_width, color=color,
                    edgecolor='white', linewidth=1.2)
    for bar_obj, val in zip(bars, vals):
        if not np.isnan(val):
            ax2.text(bar_obj.get_x()+bar_obj.get_width()/2,
                     bar_obj.get_height()+0.8, f'{int(val)}',
                     ha='center', va='bottom', fontsize=9,
                     fontweight='bold', color=color)

ax2.set_xticks(x); ax2.set_xticklabels(down_labels, fontsize=11, color=TU_BLUE)
ax2.tick_params(colors=TU_BLUE)
ax2.set_xlabel('Down', fontsize=12, color=TU_BLUE)
ax2.set_ylabel('Number of Plays', fontsize=12, color=TU_BLUE)
ax2.set_title('Run vs. Pass Play Volume by Down', fontsize=13,
              fontweight='bold', color=TU_BLUE, pad=12)
ax2.legend(handles=[mpatches.Patch(color=TU_BLUE, label='Run'),
                     mpatches.Patch(color=TU_GOLD, label='Pass')],
           fontsize=10, framealpha=0.6, edgecolor='#CCCCCC')
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')

total_runs   = int(summary[summary['play_type']=='run']['count'].sum())
total_passes = int(summary[summary['play_type']=='pass']['count'].sum())
fig.suptitle(f'TU Offensive Run vs. Pass EPA by Down\\nTotal Runs: {total_runs}  |  Total Passes: {total_passes}',
             fontsize=14, fontweight='bold', color=TU_BLUE, y=1.01)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — FIELD ZONE
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 6 · Where on the Field Do We Succeed?

**What it shows:** Average EPA by field zone, from deep in our own territory
(76–99 yards to go) through the red zone (1–20 yards to go).

> **So what?** Midfield (41–60 yards) is where drives are built or broken.
> Red zone EPA is the most consequential single number — if it's below the
> baseline, we are leaving points on the field every week.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['yards_to_go'].notna())
].copy()

bins   = [0, 20, 40, 60, 75, 99]
labels = ['Redzone\\n1–20 yds', 'Opp Territory\\n21–40 yds',
          'Midfield\\n41–60 yds', 'Own Territory\\n61–75 yds', 'Own Territory\\n76–99 yds']
off_df['field_zone'] = pd.cut(off_df['yards_to_go'], bins=bins,
                               labels=labels, include_lowest=True)

zone_df = (
    off_df.groupby('field_zone', observed=True)['epa']
    .agg(['mean','sem','count']).reset_index()
).iloc[::-1].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(zone_df['field_zone'], zone_df['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in zone_df['mean']],
              edgecolor='white', linewidth=1.2, width=0.55)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=9, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, zone_df.itertuples()):
    label_y = row.mean + 0.008 if row.mean >= 0 else row.mean - 0.015
    ax.text(bar.get_x()+bar.get_width()/2, label_y, f'μ={row.mean:+.3f}',
            ha='center', va='bottom' if row.mean >= 0 else 'top',
            fontsize=11, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0] - 0.25,
            f'n={row.count}', ha='center', va='bottom', fontsize=9, color='gray')

ax.set_ylim(bottom=-0.40, top=0.32)
ax.annotate('← Own Territory', xy=(0.02, -0.15), xycoords='axes fraction',
            fontsize=9, color='gray', style='italic')
ax.annotate('Endzone →', xy=(0.80, -0.15), xycoords='axes fraction',
            fontsize=9, color='gray', style='italic')
ax.set_title('TU Offensive EPA by Field Zone', fontsize=15,
             fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Field Zone  (yards to go to endzone)', fontsize=12,
              color=TU_BLUE, labelpad=20)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — RED ZONE BY DOWN
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 7 · Red Zone Efficiency by Down

**What it shows:** Average EPA per play in the red zone (yards to go ≤ 20),
broken out by down. The compressed field gives defenses a structural advantage,
so positive EPA here is genuinely difficult and highly valuable.

> **So what?** Negative 3rd down red zone EPA means we are leaving the red zone
> with field goals or punts far too often. Every point left on the field in the
> red zone is a direct loss of win probability.
""")

code("""
rz_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) &
    (tu_df['yards_to_go'].notna()) & (tu_df['dn'].notna()) &
    (tu_df['yards_to_go'] <= 20)
].copy()

rz_df['down_group'] = rz_df['dn'].map(
    {1.0: '1st Down', 2.0: '2nd Down', 3.0: '3rd Down', 4.0: '4th Down'}
)
rz_df = rz_df[rz_df['down_group'].notna()]
down_order = ['1st Down', '2nd Down', '3rd Down', '4th Down']

down_agg = (
    rz_df.groupby('down_group', observed=True)['epa']
    .agg(['mean','sem','count']).reindex(down_order).reset_index()
)

fig, ax = plt.subplots(figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(down_agg['down_group'], down_agg['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in down_agg['mean']],
              edgecolor='white', linewidth=1.2, width=0.5)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=9, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, down_agg.itertuples()):
    if pd.isna(row.mean): continue
    ax.text(bar.get_x()+bar.get_width()/2,
            row.mean + 0.008 if row.mean >= 0 else row.mean - 0.015,
            f'μ={row.mean:+.3f}', ha='center',
            va='bottom' if row.mean >= 0 else 'top',
            fontsize=11, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0] - 0.25,
            f'n={int(row.count)}', ha='center', va='bottom', fontsize=9, color='gray')

ax.set_ylim(bottom=-1.2, top=0.35)
ax.set_title('TU Red Zone EPA by Down  (yards to go ≤ 20)',
             fontsize=15, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Down', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 8 — RED ZONE RUN VS PASS
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 8 · Red Zone: Run vs. Pass

**What it shows:** Mean EPA and play volume for runs vs. passes by down inside
the red zone (yards to go ≤ 20).

> **So what?** In the red zone, defenses sell out. Look at run/pass split
> percentage — if one type makes up 70%+ of our snaps, defenses will adjust.
> Maintaining balance, even when one type has a slight EPA edge, keeps
> coordinators honest.
""")

code("""
rz_rp = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['yards_to_go'].notna()) &
    (tu_df['yards_to_go'] <= 20) & (tu_df['play_type'].isin(['run', 'pass'])) &
    (tu_df['epa'].notna()) & (tu_df['dn'].isin([1, 2, 3, 4]))
].copy()

down_order  = [1, 2, 3, 4]
down_labels = ['1st Down', '2nd Down', '3rd Down', '4th Down']
PLAY_COLOR  = {'run': TU_BLUE, 'pass': TU_GOLD}

summary = (
    rz_rp.groupby(['dn', 'play_type'])
    .agg(mean_epa=('epa','mean'), sem=('epa','sem'), count=('epa','count'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
x, bar_width = np.arange(len(down_order)), 0.38

for ax, metric, ylabel, title in zip(
    axes,
    ['mean_epa', 'count'],
    ['Average EPA per Play', 'Number of Plays'],
    ['Red Zone EPA by Down\\n& Play Type', 'Red Zone Play Volume\\nby Down & Play Type']
):
    ax.set_facecolor(LIGHT_GRAY)
    for play_type, offset in [('run', -bar_width/2), ('pass', bar_width/2)]:
        data  = summary[summary['play_type']==play_type].set_index('dn').reindex(down_order)
        vals  = data[metric].values.astype(float)
        color = PLAY_COLOR[play_type]
        bars  = ax.bar(x + offset, vals, bar_width, color=color,
                       edgecolor='white', linewidth=1.2)
        for bar_obj, val in zip(bars, vals):
            if np.isnan(val): continue
            if metric == 'mean_epa':
                ax.text(bar_obj.get_x()+bar_obj.get_width()/2,
                        val + 0.015 if val >= 0 else val - 0.015,
                        f'μ={val:+.3f}', ha='center',
                        va='bottom' if val >= 0 else 'top',
                        fontsize=8.5, fontweight='bold',
                        color=RED if val < 0 else color)
            elif val > 0:
                ax.text(bar_obj.get_x()+bar_obj.get_width()/2,
                        bar_obj.get_height()+0.5, f'{int(val)}',
                        ha='center', va='bottom', fontsize=9,
                        fontweight='bold', color=color)

    if metric == 'mean_epa':
        ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)

    ax.set_xticks(x); ax.set_xticklabels(down_labels, fontsize=10, color=TU_BLUE)
    ax.tick_params(colors=TU_BLUE)
    ax.set_xlabel('Down', fontsize=11, color=TU_BLUE)
    ax.set_ylabel(ylabel, fontsize=11, color=TU_BLUE)
    ax.set_title(title, fontsize=13, fontweight='bold', color=TU_BLUE, pad=12)
    ax.legend(handles=[mpatches.Patch(color=TU_BLUE, label='Run'),
                        mpatches.Patch(color=TU_GOLD, label='Pass')],
              fontsize=10, framealpha=0.6, edgecolor='#CCCCCC')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#CCCCCC')

run_t = int(summary[summary['play_type']=='run']['count'].sum())
pass_t = int(summary[summary['play_type']=='pass']['count'].sum())
total  = run_t + pass_t
fig.suptitle(
    f'TU Red Zone Run vs. Pass EPA  (yards to go ≤ 20)\\n'
    f'Run: {run_t} ({run_t/total*100:.0f}%)  |  Pass: {pass_t} ({pass_t/total*100:.0f}%)  |  Total: {total}',
    fontsize=14, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 9 — EPA BY RESULT TYPE
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 9 · How Plays Finished — EPA by Result Type

**What it shows:** Distribution of EPA for each play result type
(Rush, Complete, Incomplete, Sack, Interception, etc.), ordered by median EPA.
Box plots show the full distribution, not just the average.

> **So what?** Interceptions and sacks carry extreme negative EPA — eliminating
> one per game may be worth several explosive plays. If scramble EPA is positive,
> quarterback mobility is creating real value and should be schemed for, not
> just tolerated.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['result'].notna())
].copy()
off_df['result'] = off_df['result'].str.strip().str.lower()

keep_results = ['rush','complete','incomplete','interception',
                'sack','scramble','complete, td','rush, td','fumble']
off_df = off_df[off_df['result'].isin(keep_results)].copy()

label_map = {
    'rush':'Rush', 'complete':'Complete', 'incomplete':'Incomplete',
    'interception':'Interception', 'sack':'Sack', 'scramble':'Scramble',
    'complete, td':'Complete TD', 'rush, td':'Rush TD', 'fumble':'Fumble'
}
off_df['result_label'] = off_df['result'].map(label_map)

result_order = (
    off_df.groupby('result_label')['epa'].median()
    .sort_values(ascending=False).index.tolist()
)
median_vals = off_df.groupby('result_label')['epa'].median()
box_colors  = {r: (TU_BLUE if median_vals[r] >= 0 else RED) for r in result_order}

fig, ax = plt.subplots(figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

sns.boxplot(data=off_df, x='result_label', y='epa', order=result_order,
            palette=box_colors, width=0.5, linewidth=1.3,
            flierprops=dict(marker='o', markersize=3, alpha=0.3,
                            linestyle='none', markeredgewidth=0), ax=ax)

for i, result in enumerate(result_order):
    subset      = off_df[off_df['result_label'] == result]['epa']
    Q3          = subset.quantile(0.75)
    whisker_top = subset[subset <= Q3 + 1.5*(Q3-subset.quantile(0.25))].max()
    ax.text(i, whisker_top + 0.15, f'μ={subset.mean():+.3f}',
            ha='center', va='bottom', fontsize=8.5, fontweight='bold',
            color=TU_BLUE if median_vals[result] >= 0 else RED)
    ax.text(i, ax.get_ylim()[0]+0.1, f'n={len(subset)}',
            ha='center', va='bottom', fontsize=8.5, color='gray')

ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.set_title('TU Offensive EPA Distribution by Result Type',
             fontsize=15, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Result Type', fontsize=12, color=TU_BLUE)
ax.set_ylabel('EPA per Play', fontsize=12, color=TU_BLUE)
ax.set_ylim(bottom=ax.get_ylim()[0]-0.5, top=ax.get_ylim()[1]+1.0)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 10 — RUN VS PASS DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 10 · Run vs. Pass EPA Distribution

**What it shows:** Full EPA distribution (box + strip plot) for all runs vs.
all passes across the dataset. Mean markers show where each type sits on average.

> **So what?** The shape of the distribution matters as much as the mean. A wide
> distribution means high upside but also higher risk. If run EPA mean is below
> zero, our rushing attack is hurting us more than helping in aggregate — not a
> reason to stop running, but a signal to be more selective about when and how.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()
rp_df  = off_df[off_df['play_type'].isin(['run','pass'])].copy()
rp_df['play_type'] = rp_df['play_type'].str.capitalize()
palette = {'Run': TU_BLUE, 'Pass': TU_GOLD}

fig, ax = plt.subplots(figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

sns.boxplot(data=rp_df, x='play_type', y='epa', palette=palette,
            order=['Run','Pass'], width=0.4, linewidth=1.3,
            flierprops=dict(marker='', markersize=0), ax=ax)
sns.stripplot(data=rp_df, x='play_type', y='epa', palette=palette,
              order=['Run','Pass'], size=2.5, alpha=0.35, jitter=True, ax=ax)

OFFSET = 0.3
for i, collection in enumerate(ax.collections):
    offsets = collection.get_offsets()
    offsets[:, 0] += OFFSET
    collection.set_offsets(offsets)

for i, pt in enumerate(['Run','Pass']):
    subset   = rp_df[rp_df['play_type'] == pt]
    mean_val = subset['epa'].mean()
    count    = len(subset)
    ax.plot(i + OFFSET, mean_val, marker='D', color='white', markersize=8, zorder=5)
    ax.plot(i + OFFSET, mean_val, marker='D', color='black', markersize=5, zorder=6)
    ax.text(i + OFFSET + 0.15, mean_val, f'μ={mean_val:+.3f}',
            va='center', fontsize=10, fontweight='bold', color='black')
    ax.text(i, ax.get_ylim()[0]+0.2, f'n={count}',
            ha='center', fontsize=9, color='gray')

ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.set_title('TU Offensive EPA Distribution\\nRun vs. Pass',
             fontsize=15, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Play Type', fontsize=12, color=TU_BLUE)
ax.set_ylabel('EPA per Play', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 11 — EPA BY DISTANCE
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 11 · EPA by Distance to Go

**What it shows:** Average EPA across all offensive plays grouped by
distance-to-gain bucket. Covers all downs — not just 3rd.

> **So what?** Long-distance situations (11+ yards) are a red flag regardless
> of the play result. Getting into 2nd-and-12 or 3rd-and-14 frequently means
> 1st and 2nd down execution is breaking down upstream. Preventing long-distance
> is often more important than the play call once you're in it.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['dist'].notna())
].copy()

bins   = [0, 3, 7, 10, float('inf')]
labels = ['Short\\n(≤3 yds)', 'Medium\\n(4–7 yds)',
          'Standard\\n(8–10 yds)', 'Long\\n(11+ yds)']
off_df['dist_bucket'] = pd.cut(off_df['dist'], bins=bins,
                                labels=labels, include_lowest=True)

dist_df = (
    off_df.groupby('dist_bucket', observed=True)['epa']
    .agg(['mean','sem','count']).reset_index()
)

fig, ax = plt.subplots(figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(dist_df['dist_bucket'], dist_df['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in dist_df['mean']],
              edgecolor='white', linewidth=1.2, width=0.55)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=9, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, dist_df.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2,
            row.mean + 0.008 if row.mean >= 0 else row.mean - 0.001,
            f'μ={row.mean:+.3f}', ha='center',
            va='bottom' if row.mean >= 0 else 'top',
            fontsize=11, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0]+0.001,
            f'n={row.count}', ha='center', va='bottom', fontsize=9, color='gray')

ax.set_title('TU Offensive EPA by Distance-to-Go',
             fontsize=15, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Distance to Go', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 12 — PERSONNEL & FORMATION
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 12 · Our Scheme — Personnel & Formation

### 12a · EPA by Personnel Grouping

**What it shows:** Average EPA for each personnel package (minimum 20 snaps),
sorted highest to lowest.

> **So what?** Rank your personnel groupings by EPA and ask whether your usage
> matches that ranking. Your most productive package should be getting the most
> snaps — unless there is a specific situational reason otherwise.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['personnel'].notna())
].copy()

pers_df = (
    off_df.groupby('personnel')['epa']
    .agg(['mean','sem','count']).reset_index()
)
pers_df = pers_df[pers_df['count'] >= 20].sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(pers_df['personnel'], pers_df['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in pers_df['mean']],
              edgecolor='white', linewidth=1.2, width=0.55)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=9, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, pers_df.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2,
            row.mean + 0.008 if row.mean >= 0 else row.mean - 0.008,
            f'μ={row.mean:+.3f}', ha='center',
            va='bottom' if row.mean >= 0 else 'top',
            fontsize=10, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0]-0.1,
            f'n={row.count}', ha='center', va='bottom', fontsize=9, color='gray')

ax.set_title('TU Offensive EPA by Personnel Grouping',
             fontsize=15, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Personnel', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.set_ylim(bottom=ax.get_ylim()[0]-0.3, top=ax.get_ylim()[1]+0.3)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

md("""
### 12b · EPA by Formation

**What it shows:** Average EPA by offensive formation (minimum 20 snaps),
with RIGHT/LEFT directional variants collapsed into their base formation name.

> **So what?** If one formation dramatically outperforms others within the same
> personnel package, ask why — is it a blocking angle, a route concept, or a
> coverage exploit? Replicate what's working. Low-EPA formations with high
> usage are a scheme inefficiency you can correct immediately.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['off_form'].notna())
].copy()

def normalize_formation(form):
    form = str(form).strip().upper()
    if form in ['RIGHT', 'LEFT']:
        return form
    form = re.sub(r'\\bRIGHT\\b', '', form)
    form = re.sub(r'\\bLEFT\\b', '', form)
    return ' '.join(form.split()) or 'OTHER'

off_df['form_normalized'] = off_df['off_form'].apply(normalize_formation)

form_df = (
    off_df.groupby('form_normalized')['epa']
    .agg(['mean','sem','count']).reset_index()
)
form_df = form_df[form_df['count'] >= 20].sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(form_df['form_normalized'], form_df['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in form_df['mean']],
              edgecolor='white', linewidth=1.2, width=0.3)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=9, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, form_df.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2,
            row.mean + 0.008 if row.mean >= 0 else row.mean - 0.008,
            f'μ={row.mean:+.3f}', ha='center',
            va='bottom' if row.mean >= 0 else 'top',
            fontsize=8, fontweight='bold', rotation=90,
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0]-0.2,
            f'n={row.count}', ha='center', va='bottom', fontsize=7.5, color='gray')

ax.set_title('TU Offensive EPA by Formation',
             fontsize=15, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Formation', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.set_ylim(bottom=ax.get_ylim()[0]-0.3, top=ax.get_ylim()[1]+0.3)
ax.tick_params(axis='x', colors=TU_BLUE, rotation=90)
ax.tick_params(axis='y', colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

md("""
### 12c · EPA by Personnel & Formation Combination (Interactive)

**What it shows:** Interactive grouped bar chart — click a personnel group in
the legend to display its formation-level EPA breakdown. Use Show All / Hide All
to compare across groups.

> **So what?** This is the most actionable schematic chart in the report. Before
> each game, identify which personnel/formation combinations have the highest EPA
> and build your game plan around them — especially against opponents whose
> defensive tendencies suggest vulnerability to those looks.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) &
    (tu_df['personnel'].notna()) & (tu_df['off_form'].notna())
].copy()

def normalize_formation(form):
    form = str(form).strip().upper()
    if form in ['RIGHT', 'LEFT']:
        return form
    form = re.sub(r'\\bRIGHT\\b', '', form)
    form = re.sub(r'\\bLEFT\\b', '', form)
    return ' '.join(form.split()) or 'OTHER'

off_df['form_normalized'] = off_df['off_form'].apply(normalize_formation)

combo_df = (
    off_df.groupby(['personnel', 'form_normalized'])['epa']
    .agg(['mean','sem','count']).reset_index()
    .rename(columns={'form_normalized': 'formation'})
)
combo_df = combo_df[combo_df['count'] >= 10].copy()

personnel_list = (
    combo_df.groupby('personnel')['mean'].mean()
    .sort_values(ascending=False).index.tolist()
)
pers_colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
               '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']
pers_color_map = {p: pers_colors[i % len(pers_colors)]
                  for i, p in enumerate(personnel_list)}

fig = go.Figure()
for pers in personnel_list:
    d = combo_df[combo_df['personnel'] == pers].sort_values('mean', ascending=False)
    hover = [f"<b>{pers} | {r['formation']}</b><br>μ EPA: {r['mean']:+.3f}<br>n={int(r['count'])}"
             for _, r in d.iterrows()]
    fig.add_trace(go.Bar(
        x=d['formation'], y=d['mean'], name=pers,
        marker_color=[TU_BLUE if v >= 0 else RED for v in d['mean']],
        marker_line_color='white', marker_line_width=1.2,
        text=hover, hovertemplate='%{text}<extra></extra>',
        visible='legendonly', legendgroup=pers, showlegend=True
    ))

fig.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4, line_width=1.5)
n_traces = len(fig.data)
fig.update_layout(
    title=dict(
        text='TU Offensive EPA by Personnel & Formation<br>'
             '<sup>Click a personnel group to show its formations</sup>',
        font=dict(size=17, color=TU_BLUE), x=0.5, xanchor='center'),
    xaxis=dict(title='Formation', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), tickangle=45, gridcolor='#DDDDDD'),
    yaxis=dict(title='Average EPA per Play', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD'),
    barmode='group',
    legend=dict(title=dict(text='Personnel', font=dict(color=TU_BLUE, size=11)),
                bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC',
                borderwidth=1, font=dict(size=10)),
    updatemenus=[dict(
        type='buttons', showactive=False,
        x=1.0, xanchor='right', y=1.08, yanchor='top',
        buttons=[
            dict(label='Show All', method='restyle',
                 args=[{'visible': True}, list(range(n_traces))]),
            dict(label='Hide All', method='restyle',
                 args=[{'visible': 'legendonly'}, list(range(n_traces))])
        ],
        bgcolor='white', bordercolor='#CCCCCC', font=dict(color=TU_BLUE, size=10)
    )],
    plot_bgcolor=LIGHT_GRAY, paper_bgcolor=LIGHT_GRAY,
    height=650, margin=dict(t=120, b=160, l=70, r=20)
)
fig.show()
""")

md("""
### 12d · Explosive Rate by Personnel

**What it shows:** Left panel — explosive play rate (%) per personnel group.
Right panel — mean EPA per personnel group in the same sort order.
A personnel package with both high explosive rate and high EPA is your best look.
""")

code("""
exp_pers = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['play_type'].isin(['run','pass'])) &
    (tu_df['gn_ls'].notna()) & (tu_df['personnel'].notna()) & (tu_df['epa'].notna())
].copy()

exp_pers['explosive'] = (
    ((exp_pers['play_type'] == 'run')  & (exp_pers['gn_ls'] >= 12)) |
    ((exp_pers['play_type'] == 'pass') & (exp_pers['gn_ls'] >= 21))
)

pers_df = (
    exp_pers.groupby('personnel')
    .agg(exp_rate=('explosive','mean'), exp_count=('explosive','sum'),
         total=('explosive','count'), mean_epa=('epa','mean'))
    .reset_index()
)
pers_df = pers_df[pers_df['total'] >= 20].copy()
pers_df['exp_rate_pct'] = pers_df['exp_rate'] * 100
pers_df = pers_df.sort_values('exp_rate_pct', ascending=False).reset_index(drop=True)
overall_exp_rate = exp_pers['explosive'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)

ax1 = axes[0]
ax1.set_facecolor(LIGHT_GRAY)
bar_colors = [TU_GOLD if v >= overall_exp_rate else TU_BLUE
              for v in pers_df['exp_rate_pct']]
bars = ax1.bar(pers_df['personnel'], pers_df['exp_rate_pct'],
               color=bar_colors, edgecolor='white', linewidth=1.2, width=0.55)
ax1.axhline(overall_exp_rate, color=RED, linewidth=1.4, linestyle='--', alpha=0.75)
ax1.text(len(pers_df)-0.5, overall_exp_rate+0.3, f'Avg: {overall_exp_rate:.1f}%',
         ha='right', va='bottom', fontsize=9, color=RED, fontweight='bold')
for bar, row in zip(bars, pers_df.itertuples()):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{row.exp_rate_pct:.1f}%', ha='center', va='bottom',
             fontsize=9, fontweight='bold',
             color=TU_GOLD if row.exp_rate_pct >= overall_exp_rate else TU_BLUE)
ax1.set_title('Explosive Play Rate\\nby Personnel Grouping',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=12)
ax1.set_xlabel('Personnel', fontsize=11, color=TU_BLUE)
ax1.set_ylabel('Explosive Play Rate (%)', fontsize=11, color=TU_BLUE)
ax1.tick_params(colors=TU_BLUE)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
bars2 = ax2.bar(pers_df['personnel'], pers_df['mean_epa'],
                color=[TU_BLUE if v >= 0 else RED for v in pers_df['mean_epa']],
                edgecolor='white', linewidth=1.2, width=0.55)
ax2.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax2.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
            alpha=0.85, label='Model Baseline (+0.028)')
ax2.legend(fontsize=9, framealpha=0.6, edgecolor='#CCCCCC')
for bar, row in zip(bars2, pers_df.itertuples()):
    ax2.text(bar.get_x()+bar.get_width()/2,
             row.mean_epa + 0.008 if row.mean_epa >= 0 else row.mean_epa - 0.008,
             f'μ={row.mean_epa:+.3f}', ha='center',
             va='bottom' if row.mean_epa >= 0 else 'top',
             fontsize=9, fontweight='bold',
             color=TU_BLUE if row.mean_epa >= 0 else RED)
ax2.set_title('Mean EPA per Play\\nby Personnel Grouping (same order)',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=12)
ax2.set_xlabel('Personnel', fontsize=11, color=TU_BLUE)
ax2.set_ylabel('Average EPA per Play', fontsize=11, color=TU_BLUE)
ax2.tick_params(colors=TU_BLUE)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')

total_exp   = int(exp_pers['explosive'].sum())
total_plays = len(exp_pers)
fig.suptitle(
    f'TU Explosive Play Generation by Personnel\\n'
    f'{total_exp} explosive plays from {total_plays} snaps '
    f'({overall_exp_rate:.1f}% overall)  |  Run ≥12 yds · Pass ≥21 yds  |  Gold = above avg rate',
    fontsize=13, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 13 — EXPLOSIVES
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 13 · Explosive Plays

### 13a · Explosive vs. Non-Explosive EPA

**What it shows:** Left — distribution comparison between explosive plays
(run ≥12 yds, pass ≥21 yds) and non-explosive plays. Right — total EPA
contribution split between the two categories.

> **So what?** If explosive plays account for the majority of our total EPA,
> our offense is variance-dependent. One or two explosives per game may be
> the difference between a 400-yard performance and a stalled drive night.
> We need to complement explosives with consistent drive-building efficiency.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) &
    (tu_df['gn_ls'].notna()) & (tu_df['play_type'].isin(['run','pass']))
].copy()

off_df['explosive'] = (
    ((off_df['play_type'] == 'run')  & (off_df['gn_ls'] >= 12)) |
    ((off_df['play_type'] == 'pass') & (off_df['gn_ls'] >= 21))
)
off_df['explosive_label'] = off_df['explosive'].map({True:'Explosive', False:'Non-Explosive'})

total_epa        = off_df['epa'].sum()
explosive_epa    = off_df[off_df['explosive']]['epa'].sum()
explosive_pct    = explosive_epa / total_epa * 100 if total_epa != 0 else 0
explosive_play_pct = off_df['explosive'].mean() * 100

order   = ['Explosive', 'Non-Explosive']
palette = {'Explosive': TU_GOLD, 'Non-Explosive': TU_BLUE}

fig, axes = plt.subplots(1, 2, figsize=(22, 10), gridspec_kw={'width_ratios': [2, 1]})
fig.patch.set_facecolor(LIGHT_GRAY)

ax1 = axes[0]
ax1.set_facecolor(LIGHT_GRAY)
sns.boxplot(data=off_df, x='explosive_label', y='epa', order=order,
            palette=palette, width=0.45, linewidth=1.3,
            flierprops=dict(marker='o', markersize=3, alpha=0.3,
                            linestyle='none', markeredgewidth=0), ax=ax1)

for i, label in enumerate(order):
    subset      = off_df[off_df['explosive_label'] == label]['epa']
    Q3          = subset.quantile(0.75)
    whisker_top = subset[subset <= Q3 + 1.5*(Q3-subset.quantile(0.25))].max()
    ax1.text(i, whisker_top + 0.15, f'μ={subset.mean():+.3f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold',
             color=TU_GOLD if label == 'Explosive' else TU_BLUE)
    ax1.text(i, ax1.get_ylim()[0]+0.1, f'n={len(subset)}',
             ha='center', va='bottom', fontsize=9, color='gray')

ax1.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax1.set_title('EPA Distribution\\nExplosive vs. Non-Explosive',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=10)
ax1.set_xlabel('Play Type', fontsize=11, color=TU_BLUE)
ax1.set_ylabel('EPA per Play', fontsize=11, color=TU_BLUE)
ax1.set_ylim(bottom=ax1.get_ylim()[0]-0.5, top=ax1.get_ylim()[1]+1.2)
ax1.tick_params(colors=TU_BLUE)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
contrib_vals   = [explosive_epa, total_epa - explosive_epa]
contrib_colors = [TU_GOLD, TU_BLUE]
bars = ax2.bar(['Explosive\\nPlays', 'Non-Explosive\\nPlays'],
               contrib_vals, color=contrib_colors,
               edgecolor='white', linewidth=1.2, width=0.45)
for bar, val in zip(bars, contrib_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
             f'{val:+.1f} EPA\\n({val/total_epa*100:.1f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold',
             color=TU_GOLD if val == explosive_epa else TU_BLUE)

ax2.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax2.set_title('Total EPA Contribution\\nby Play Type',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=10)
ax2.set_ylabel('Total EPA', fontsize=11, color=TU_BLUE)
ax2.set_ylim(bottom=0, top=max(contrib_vals)*1.3)
ax2.tick_params(colors=TU_BLUE)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')

fig.suptitle(
    f'TU Explosive vs. Non-Explosive Play EPA Analysis\\n'
    f'Explosive plays = {explosive_play_pct:.1f}% of snaps, '
    f'accounting for {explosive_pct:.1f}% of total offensive EPA',
    fontsize=14, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

md("""
### 13b · Explosive Plays by Field Position

**What it shows:** Left — explosive play count and rate by field zone, with a
rate line overlay. Right — mean EPA of explosive plays by zone.

> **So what?** Explosive play rate is a controllable schematic variable — it is
> not random. If our rate is highest at midfield but drops near zero in opponent
> territory, defenses are loading up once we cross the 40. Motions, shifts, and
> tempo changes can disrupt that loading tendency.
""")

code("""
exp_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['play_type'].isin(['run','pass'])) &
    (tu_df['gn_ls'].notna()) & (tu_df['yards_to_go'].notna()) & (tu_df['epa'].notna())
].copy()

exp_df['explosive'] = (
    ((exp_df['play_type'] == 'run')  & (exp_df['gn_ls'] >= 12)) |
    ((exp_df['play_type'] == 'pass') & (exp_df['gn_ls'] >= 21))
)

bins   = [0, 20, 40, 60, 75, 99]
labels = ['Red Zone\\n(1–20)', 'Opp Territory\\n(21–40)',
          'Midfield\\n(41–60)', 'Own Territory\\n(61–75)', 'Own Territory\\n(76–99)']
exp_df['field_zone'] = pd.cut(exp_df['yards_to_go'], bins=bins,
                               labels=labels, include_lowest=True)

zone_agg = (
    exp_df.groupby('field_zone', observed=True)
    .agg(
        total_plays=('explosive','count'), exp_count=('explosive','sum'),
        exp_rate=('explosive','mean'),
        exp_epa_mean=('epa', lambda x: x[exp_df.loc[x.index,'explosive']].mean()),
    )
    .reset_index()
).iloc[::-1].reset_index(drop=True)
zone_agg['exp_rate_pct'] = zone_agg['exp_rate'] * 100
overall_exp_rate = exp_df['explosive'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)

ax1      = axes[0]
ax1_twin = ax1.twinx()
ax1.set_facecolor(LIGHT_GRAY)

bar_colors = [TU_GOLD if r >= overall_exp_rate else TU_BLUE
              for r in zone_agg['exp_rate_pct']]
bars = ax1.bar(zone_agg['field_zone'], zone_agg['exp_count'],
               color=bar_colors, edgecolor='white', linewidth=1.2, width=0.55, zorder=2)

for bar, row in zip(bars, zone_agg.itertuples()):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{int(row.exp_count)}', ha='center', va='bottom', fontsize=10,
             fontweight='bold',
             color=TU_GOLD if row.exp_rate_pct >= overall_exp_rate else TU_BLUE)
    ax1.text(bar.get_x()+bar.get_width()/2, 0.15,
             f'({row.exp_rate_pct:.1f}%)', ha='center', va='bottom',
             fontsize=8.5, color='gray')

ax1_twin.plot(zone_agg['field_zone'], zone_agg['exp_rate_pct'],
              color=RED, linewidth=2, marker='o', markersize=7, zorder=3)
ax1_twin.axhline(overall_exp_rate, color=RED, linewidth=1.2, linestyle='--', alpha=0.5)
ax1_twin.set_ylabel('Explosive Rate (%)', fontsize=11, color=RED)
ax1_twin.tick_params(axis='y', colors=RED)
ax1_twin.spines[['top']].set_visible(False)
ax1.set_title('Explosive Play Count & Rate\\nby Field Zone',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=12)
ax1.set_xlabel('Field Zone', fontsize=11, color=TU_BLUE)
ax1.set_ylabel('Number of Explosive Plays', fontsize=11, color=TU_BLUE)
ax1.tick_params(axis='x', colors=TU_BLUE)
ax1.tick_params(axis='y', colors=TU_BLUE)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
bars2 = ax2.bar(zone_agg['field_zone'], zone_agg['exp_epa_mean'],
                color=[TU_BLUE if v >= 0 else RED for v in zone_agg['exp_epa_mean']],
                edgecolor='white', linewidth=1.2, width=0.55)
ax2.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
for bar, row in zip(bars2, zone_agg.itertuples()):
    if np.isnan(row.exp_epa_mean): continue
    ax2.text(bar.get_x()+bar.get_width()/2,
             row.exp_epa_mean + 0.05 if row.exp_epa_mean >= 0 else row.exp_epa_mean - 0.05,
             f'μ={row.exp_epa_mean:+.3f}', ha='center',
             va='bottom' if row.exp_epa_mean >= 0 else 'top',
             fontsize=9, fontweight='bold',
             color=TU_BLUE if row.exp_epa_mean >= 0 else RED)
ax2.set_title('Mean EPA of Explosive Plays\\nby Field Zone',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=12)
ax2.set_xlabel('Field Zone', fontsize=11, color=TU_BLUE)
ax2.set_ylabel('Average EPA per Explosive Play', fontsize=11, color=TU_BLUE)
ax2.tick_params(axis='x', colors=TU_BLUE)
ax2.tick_params(axis='y', colors=TU_BLUE)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')

total_exp   = int(exp_df['explosive'].sum())
total_plays = len(exp_df)
fig.suptitle(
    f'TU Explosive Play Field Position Analysis\\n'
    f'{total_exp} explosive plays from {total_plays} snaps ({overall_exp_rate:.1f}% overall)  |  '
    f'Gold = above-average explosive rate zone',
    fontsize=13, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 14 — DRIVE NARRATIVE
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 14 · Drive Narrative

### 14a · Cumulative Offensive EPA by Drive Series (Season Aggregate)

**What it shows:** Cumulative EPA accumulation across all offensive series in
the dataset. A rising line means consistent value addition; a flat or declining
line means drives are breaking even or costing us.

> **So what?** The shape of this curve is your season's offensive story in one
> line. Sharp upward turns mark run-game dominance or explosive play stretches.
> Flat sections reveal extended stretches of stalled drives.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()

series_df = (
    off_df.groupby('off_series')['epa']
    .agg(['sum','mean','count']).reset_index()
    .rename(columns={'sum':'total_epa','mean':'avg_epa','count':'n_plays'})
)
series_df = series_df[series_df['n_plays'] >= 5]
series_df['cumulative_epa'] = series_df['total_epa'].cumsum()

fig, ax = plt.subplots(figsize=(22, 10))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

ax.fill_between(series_df['off_series'], series_df['cumulative_epa'], 0,
                where=series_df['cumulative_epa'] >= 0, alpha=0.15, color=TU_BLUE)
ax.fill_between(series_df['off_series'], series_df['cumulative_epa'], 0,
                where=series_df['cumulative_epa'] < 0, alpha=0.15, color=RED)
ax.plot(series_df['off_series'], series_df['cumulative_epa'],
        color=TU_BLUE, linewidth=2.5, zorder=3)
ax.scatter(series_df['off_series'], series_df['cumulative_epa'],
           color=TU_GOLD, edgecolors=TU_BLUE, linewidth=1.2, s=45, zorder=4)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)

final_val = series_df['cumulative_epa'].iloc[-1]
final_ser = series_df['off_series'].iloc[-1]
ax.annotate(f'Final: μ={final_val:+.2f}',
            xy=(final_ser, final_val),
            xytext=(final_ser - 4, final_val + (8 if final_val >= 0 else -8)),
            fontsize=10, fontweight='bold',
            color=TU_BLUE if final_val >= 0 else RED,
            arrowprops=dict(arrowstyle='->', color=TU_GOLD, lw=1.5))

ax.set_title('TU Cumulative Offensive EPA by Drive Series\\n(Season Aggregate)',
             fontsize=15, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Offensive Series Number', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Cumulative EPA', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

md("""
### 14b · Cumulative EPA Per Game — Interactive

**What it shows:** Drive-by-drive cumulative EPA for each individual game,
togglable by season and color-coded by year or W/L result.

> **So what?** Compare drive trajectories across games to identify whether
> winning games follow a different pattern than losses. Games where EPA builds
> steadily indicate sustained efficiency; games with one sharp spike and then
> flat lines indicate explosion-dependent offense.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()

game_series_df = (
    off_df.groupby(['game_id','opp_name','game_date','team_pts','opp_pts','win','off_series'])['epa']
    .agg(['sum','count']).reset_index()
    .rename(columns={'sum':'total_epa','count':'n_plays'})
)
game_series_df = game_series_df[game_series_df['n_plays'] >= 1]
game_series_df['cumulative_epa'] = game_series_df.groupby('game_id')['total_epa'].cumsum()
game_series_df['game_date_parsed'] = pd.to_datetime(game_series_df['game_date'], format='%m/%d/%Y')
game_series_df['year'] = game_series_df['game_date_parsed'].dt.year

def build_label(row):
    result = 'W' if row['win'] == 1 else 'L'
    return f"{row['game_date']} vs. {row['opp_name']} ({result} {int(row['team_pts'])}-{int(row['opp_pts'])})"

game_series_df['game_label'] = game_series_df.apply(build_label, axis=1)

game_meta = (
    game_series_df[['game_id','game_label','game_date_parsed','year','win']]
    .drop_duplicates('game_id').sort_values('game_date_parsed', ascending=False)
)
all_years = sorted(game_meta['year'].unique())

YEAR_PALETTE = ['#002868','#C5960C','#1D9E75','#D85A30','#9467bd',
                '#378ADD','#BA7517','#e377c2','#17becf','#8c564b']
year_color_map = {yr: YEAR_PALETTE[i % len(YEAR_PALETTE)] for i, yr in enumerate(all_years)}

fig = go.Figure()
current_year = None
game_indices = []
year_traces  = []
wl_traces    = []
trace_index  = 0

# Set A: colored by year
for _, meta in game_meta.iterrows():
    game_id = meta['game_id']
    year    = meta['year']
    label   = meta['game_label']
    group   = game_series_df[game_series_df['game_id'] == game_id]
    color   = year_color_map[year]

    if year != current_year:
        current_year = year
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=0, color='rgba(0,0,0,0)'),
            name=f'<b>── {year} ──</b>', showlegend=True,
            hoverinfo='skip', visible=True, legendgroup=f'year_header_{year}',
        ))
        trace_index += 1

    hover = [
        f"<b>{label}</b><br>Series: {int(r.off_series)}<br>"
        f"Series EPA: {r.total_epa:+.3f}<br>Cumulative EPA: {r.cumulative_epa:+.3f}"
        for r in group.itertuples()
    ]
    fig.add_trace(go.Scatter(
        x=group['off_series'], y=group['cumulative_epa'],
        mode='lines+markers', name=label, legendgroup=game_id,
        line=dict(color=color, width=2.5),
        marker=dict(size=7, color=color, line=dict(color='white', width=1)),
        hovertemplate='%{text}<extra></extra>', text=hover, visible='legendonly',
    ))
    year_traces.append(trace_index)
    game_indices.append(trace_index)
    trace_index += 1

# Set B: colored by W/L
current_year = None
for _, meta in game_meta.iterrows():
    game_id = meta['game_id']
    year    = meta['year']
    label   = meta['game_label']
    group   = game_series_df[game_series_df['game_id'] == game_id]
    color   = WIN_COLOR if meta['win'] == 1 else LOSS_COLOR

    if year != current_year:
        current_year = year
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=0, color='rgba(0,0,0,0)'),
            name=f'<b>── {year} ──</b>', showlegend=True,
            hoverinfo='skip', visible=False, legendgroup=f'wl_header_{year}',
        ))
        trace_index += 1

    hover = [
        f"<b>{label}</b><br>Series: {int(r.off_series)}<br>"
        f"Series EPA: {r.total_epa:+.3f}<br>Cumulative EPA: {r.cumulative_epa:+.3f}"
        for r in group.itertuples()
    ]
    fig.add_trace(go.Scatter(
        x=group['off_series'], y=group['cumulative_epa'],
        mode='lines+markers', name=label, legendgroup=f'wl_{game_id}',
        showlegend=True, line=dict(color=color, width=2.5),
        marker=dict(size=7, color=color, line=dict(color='white', width=1)),
        hovertemplate='%{text}<extra></extra>', text=hover, visible=False,
    ))
    wl_traces.append(trace_index)
    game_indices.append(trace_index)
    trace_index += 1

total_traces = trace_index

def make_visibility(mode):
    vis = []
    for i in range(total_traces):
        if mode == 'year':
            vis.append(False if i in wl_traces else
                       ('legendonly' if i in year_traces else True))
        else:
            vis.append(False if i in year_traces else
                       ('legendonly' if i in wl_traces else True))
    return vis

fig.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4, line_width=1.5)
fig.update_layout(
    title=dict(
        text='TU Cumulative Offensive EPA by Drive Series<br>'
             '<sup>Click a game to add / remove  ·  Toggle color mode below</sup>',
        font=dict(size=18, color=TU_BLUE), x=0.5, xanchor='center'),
    xaxis=dict(title='Offensive Series Number', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD'),
    yaxis=dict(title='Cumulative EPA', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD'),
    legend=dict(title=dict(text='Game', font=dict(color=TU_BLUE, size=11)),
                bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC',
                borderwidth=1, font=dict(size=10), tracegroupgap=2),
    updatemenus=[
        dict(type='buttons', direction='right', showactive=True,
             x=0.5, xanchor='center', y=1.08, yanchor='top',
             bgcolor='white', bordercolor='#CCCCCC', font=dict(color=TU_BLUE, size=10),
             buttons=[
                 dict(label='Color by Year', method='restyle',
                      args=[{'visible': make_visibility('year')}]),
                 dict(label='Color by W/L', method='restyle',
                      args=[{'visible': make_visibility('wl')}]),
             ]),
        dict(type='buttons', showactive=False, x=1.0, xanchor='right',
             y=1.08, yanchor='top', bgcolor='white', bordercolor='#CCCCCC',
             font=dict(color=TU_BLUE, size=10),
             buttons=[
                 dict(label='Show All', method='restyle',
                      args=[{'visible': True}, game_indices]),
                 dict(label='Hide All', method='restyle',
                      args=[{'visible': 'legendonly'}, game_indices]),
             ]),
    ],
    plot_bgcolor=LIGHT_GRAY, paper_bgcolor=LIGHT_GRAY,
    height=650, margin=dict(t=140, b=60, l=70, r=20),
)
fig.show()
""")

md("""
### 14c · Drive Process — Outcomes & Trajectories (Interactive)

**What it shows:** Top panel — every drive plotted as net EPA vs. drive points,
color-coded by outcome (Touchdown, Field Goal, Punt, Turnover, etc.). Bottom
panel — normalized drive EPA trajectories (thin = individual drives, bold =
outcome average), showing the *shape* of how drives unfold from first to last play.

> **So what?** Drives that end in turnovers often show a characteristic EPA
> decline before the play occurs. Touchdown drives typically build differently
> than punt drives. The drive process panel makes those patterns visible.
""")

code("""
all_plays = tu_df.copy()
all_plays['play'] = pd.to_numeric(all_plays['play'], errors='coerce')
all_plays = all_plays.sort_values(['game_id','play']).reset_index(drop=True)

off_plays = all_plays[(all_plays['odk'] == 'o') & (all_plays['epa'].notna())].copy()
off_plays['year'] = pd.to_datetime(off_plays['game_date'], format='%m/%d/%Y').dt.year
off_plays['drive_key'] = (off_plays['game_id'].astype(str) + '_s' +
                          off_plays['off_series'].astype(str))

def classify_outcome(group):
    last  = group.iloc[-1]
    nxt   = last['next_odk'] if pd.notna(last['next_odk']) else ''
    tod   = last['turnover_on_downs'] if 'turnover_on_downs' in last.index else 0
    if last['score_event'] == 7:
        return 'Touchdown'
    last_idx = all_plays[
        (all_plays['game_id'] == last['game_id']) &
        (all_plays['play'] == last['play'])
    ].index
    if len(last_idx) > 0:
        next_idx = last_idx[0] + 1
        if next_idx < len(all_plays):
            nr = all_plays.iloc[next_idx]
            if nr['game_id'] == last['game_id'] and nr['score_event'] == 3:
                return 'Field Goal'
    if   tod == 1:   return 'Turnover on Downs'
    elif nxt == 'd': return 'Turnover'
    elif nxt == 'k': return 'Punt'
    else:            return 'Other'

records = []
for key, grp in off_plays.groupby('drive_key', sort=False):
    grp = grp.sort_values('play')
    outcome = classify_outcome(grp)
    records.append({
        'drive_key': key, 'outcome': outcome,
        'net_epa': round(grp['epa'].sum(), 3),
        'drive_pts': 7 if outcome=='Touchdown' else 3 if outcome=='Field Goal' else 0,
        'n_plays': len(grp), 'std_epa': round(grp['epa'].std() if len(grp)>1 else 0.0, 3),
        'epa_series': grp['epa'].tolist(),
        'game_id': grp['game_id'].iloc[0],
        'opp_name': grp['opp_name'].iloc[0] if 'opp_name' in grp.columns else '',
        'year': int(grp['year'].iloc[0]),
        'win': int(grp['win'].iloc[0]),
        'team_pts': int(grp['team_pts'].iloc[0]),
        'opp_pts': int(grp['opp_pts'].iloc[0]),
    })

drives_df = pd.DataFrame(records)
all_years = sorted(drives_df['year'].unique())

OUTCOME_COLORS  = {
    'Touchdown':'#1D9E75', 'Field Goal':'#378ADD', 'Punt':'#888780',
    'Turnover':'#D85A30', 'Turnover on Downs':'#BA7517', 'Other':'#B4B2A9',
}
OUTCOME_SYMBOLS = {
    'Touchdown':'circle', 'Field Goal':'diamond', 'Punt':'square',
    'Turnover':'x', 'Turnover on Downs':'triangle-up', 'Other':'cross',
}
N_NORM = 24

def make_hover(row):
    result = 'W' if row['win'] else 'L'
    return (f"vs {row['opp_name']}  {row['team_pts']}-{row['opp_pts']} ({result})<br>"
            f"Drive {row['drive_key'].split('_s')[-1]}  |  "
            f"Net EPA: {row['net_epa']:+.2f}  |  Pts: {row['drive_pts']}  |  "
            f"Plays: {row['n_plays']}  |  StdDev: {row['std_epa']:.2f}")

scopes = [('All seasons', 'All games', drives_df)]
for yr in all_years:
    yr_df = drives_df[drives_df['year'] == yr]
    scopes.append((str(yr), 'All games', yr_df))
    for _, grow in yr_df.drop_duplicates('game_id').sort_values('game_id').iterrows():
        result = 'W' if grow['win'] else 'L'
        glabel = f"vs {grow['opp_name']}  ({grow['team_pts']}-{grow['opp_pts']} {result})"
        scopes.append((str(yr), glabel, yr_df[yr_df['game_id'] == grow['game_id']]))

fig = make_subplots(rows=2, cols=1, subplot_titles=(' ',' '),
                    vertical_spacing=0.13, row_heights=[0.46, 0.54])

rng    = np.random.default_rng(42)
x_norm = np.linspace(0, 1, N_NORM)
all_traces      = []
scope_trace_map = []
global_idx      = 0

for yr_lbl, gm_lbl, sub in scopes:
    scope_indices = []
    if sub.empty:
        scope_trace_map.append(scope_indices)
        continue

    scatter_added = set()
    for outcome in OUTCOME_COLORS:
        mask = sub['outcome'] == outcome
        if not mask.any(): continue
        grp    = sub[mask]
        jitter = rng.uniform(-0.18, 0.18, size=len(grp))
        labels = [make_hover(r) for _, r in grp.iterrows()]
        t = go.Scatter(
            x=grp['net_epa'], y=grp['drive_pts'] + jitter,
            mode='markers', name=outcome,
            marker=dict(color=OUTCOME_COLORS[outcome], symbol=OUTCOME_SYMBOLS[outcome],
                        size=9, opacity=0.82, line=dict(width=0.6, color='white')),
            text=labels, hovertemplate='%{text}<extra></extra>',
            legendgroup=outcome, showlegend=(outcome not in scatter_added), visible=False,
        )
        all_traces.append((t, 1, 1))
        scope_indices.append(global_idx)
        global_idx += 1
        scatter_added.add(outcome)

    outcome_traj = {o: [] for o in OUTCOME_COLORS}
    for _, row in sub.iterrows():
        epas = row['epa_series']
        if len(epas) < 2: continue
        cum   = np.cumsum(epas)
        x_raw = np.linspace(0, 1, len(cum))
        y_i   = np.interp(x_norm, x_raw, cum)
        outcome_traj[row['outcome']].append(y_i)
        t = go.Scatter(
            x=x_norm, y=np.round(y_i, 3), mode='lines',
            line=dict(color=OUTCOME_COLORS[row['outcome']], width=1),
            opacity=0.25, showlegend=False, hoverinfo='skip',
            legendgroup=row['outcome'], visible=False,
        )
        all_traces.append((t, 2, 1))
        scope_indices.append(global_idx)
        global_idx += 1

    for outcome, tlist in outcome_traj.items():
        if not tlist: continue
        avg = np.mean(tlist, axis=0)
        t = go.Scatter(
            x=x_norm, y=np.round(avg, 3), mode='lines', name=outcome,
            line=dict(color=OUTCOME_COLORS[outcome], width=3),
            showlegend=False, legendgroup=outcome,
            hovertemplate=f'{outcome} avg: %{{y:.2f}}<extra></extra>', visible=False,
        )
        all_traces.append((t, 2, 1))
        scope_indices.append(global_idx)
        global_idx += 1

    scope_trace_map.append(scope_indices)

for t, row, col in all_traces:
    fig.add_trace(t, row=row, col=col)

total_traces = global_idx
for idx in scope_trace_map[0]:
    fig.data[idx].visible = True

def scope_visibility(scope_i):
    vis = [False] * total_traces
    for idx in scope_trace_map[scope_i]:
        vis[idx] = True
    return vis

def subtitle(sub):
    n     = len(sub)
    avg   = sub['net_epa'].mean() if n else 0
    td_r  = (sub['outcome']=='Touchdown').mean()*100 if n else 0
    to_r  = sub['outcome'].isin(['Turnover','Turnover on Downs']).mean()*100 if n else 0
    return f"{n} drives  ·  Avg EPA {avg:+.2f}  ·  TD rate {td_r:.0f}%  ·  TO rate {to_r:.0f}%"

combined_buttons = [dict(
    label='— All Seasons —', method='update',
    args=[{'visible': scope_visibility(0)},
          {'title.text': f'<b>TU Offensive Drive EPA</b>   — All seasons<br>'
           f'<sup style="color:#888780">{subtitle(drives_df)}</sup>'}]
)]
for yr in all_years:
    yr_scope_i = next(i for i,(yl,gl,_) in enumerate(scopes) if yl==str(yr) and gl=='All games')
    yr_sub = scopes[yr_scope_i][2]
    combined_buttons.append(dict(
        label=f'── {yr} ──', method='update',
        args=[{'visible': scope_visibility(yr_scope_i)},
              {'title.text': f'<b>TU Offensive Drive EPA</b>   — {yr}<br>'
               f'<sup style="color:#888780">{subtitle(yr_sub)}</sup>'}]
    ))
    for i,(yr_lbl,gm_lbl,sub) in enumerate(scopes):
        if yr_lbl != str(yr) or gm_lbl == 'All games': continue
        combined_buttons.append(dict(
            label=f'  › {gm_lbl}', method='update',
            args=[{'visible': scope_visibility(i)},
                  {'title.text': f'<b>TU Offensive Drive EPA</b>   — {yr}  ›  {gm_lbl}<br>'
                   f'<sup style="color:#888780">{subtitle(sub)}</sup>'}]
        ))

init_sub = scopes[0][2]
fig.update_layout(
    title=dict(
        text=f'<b>TU Offensive Drive EPA</b>   — All seasons<br>'
             f'<sup style="color:#888780">{subtitle(init_sub)}</sup>',
        font=dict(size=15, color='#2C2C2A'), x=0.0, xanchor='left',
        y=0.97, yanchor='top', pad=dict(l=10)),
    height=900, plot_bgcolor='#FAFAF8', paper_bgcolor='#FAFAF8',
    font=dict(family='Arial', color='#5F5E5A', size=11),
    legend=dict(title=dict(text='Outcome'), x=1.02, y=0.98,
                bgcolor='rgba(250,250,248,0.9)', bordercolor='#D3D1C7', borderwidth=0.5),
    margin=dict(l=60, r=160, t=160, b=55), hovermode='closest',
    updatemenus=[dict(
        buttons=combined_buttons, direction='down', showactive=True,
        x=0.0, xanchor='left', y=1.09, yanchor='top',
        bgcolor='#FFFFFF', bordercolor='#D3D1C7', borderwidth=0.5,
        font=dict(size=11, color='#2C2C2A'), pad=dict(r=10),
    )],
    annotations=[
        dict(text='Season / Game', showarrow=False, x=0.0, xanchor='left',
             y=1.135, yanchor='bottom', xref='paper', yref='paper',
             font=dict(size=10, color='#888780')),
        dict(text='<b>Drive Outcomes</b> — Net EPA vs Drive Points',
             showarrow=False, x=0.0, xanchor='left', y=0.97, yanchor='top',
             xref='paper', yref='paper', font=dict(size=12, color='#2C2C2A')),
        dict(text='<b>Drive Process</b> — Cumulative EPA trajectories  '
                  '<i>(thin = individual · bold = outcome average)</i>',
             showarrow=False, x=0.0, xanchor='left', y=0.46, yanchor='top',
             xref='paper', yref='paper', font=dict(size=12, color='#2C2C2A')),
    ],
)
fig.add_vline(x=0, line_width=1, line_dash='dot', line_color='#C4C2BA', row=1, col=1)
fig.add_hline(y=0, line_width=1, line_dash='dot', line_color='#C4C2BA', row=2, col=1)
fig.update_xaxes(title_text='Net Drive EPA', gridcolor='#EEEDFE', title_font_size=11, row=1, col=1)
fig.update_yaxes(title_text='Drive Points', gridcolor='#EEEDFE', title_font_size=11, row=1, col=1)
fig.update_xaxes(title_text='Drive progression  (0 = first play → 1 = last play, normalized)',
                 gridcolor='#EEEDFE', tickvals=[0,0.25,0.5,0.75,1.0],
                 ticktext=['Start','25%','50%','75%','End'], title_font_size=11, row=2, col=1)
fig.update_yaxes(title_text='Cumulative EPA', gridcolor='#EEEDFE', title_font_size=11, row=2, col=1)
fig.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 15 — WIN/LOSS FINGERPRINT
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## 15 · The Win/Loss Fingerprint

**What it shows:** Radar chart comparing eight offensive metrics between wins
and losses. The larger the gap between the two polygons on any axis, the more
predictive that metric is of game outcome. Filterable by season year.

| Metric | What It Measures |
|---|---|
| 1st Down EPA | Early down efficiency |
| 2nd Down EPA | Schedule maintenance |
| 3rd Down EPA | Conversion efficiency |
| Run EPA | Rushing effectiveness |
| Pass EPA | Passing effectiveness |
| Red Zone EPA | Scoring zone execution |
| Explosive EPA | Big play generation |
| 3rd Down Conv Rate | Sustained drive rate |

> **So what?** The metric with the largest gap between wins and losses is your
> program's pressure point. Use this chart year-over-year as a report card —
> if last year's gap in 3rd down EPA has closed, that is measurable program
> improvement.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()
off_df['game_date_parsed'] = pd.to_datetime(off_df['game_date'], format='%m/%d/%Y')
off_df['year'] = off_df['game_date_parsed'].dt.year
years = sorted(off_df['year'].unique(), reverse=True)

def safe_mean(series):
    return series.mean() if len(series) > 0 else 0

def build_metrics(df):
    run_df  = df[df['play_type'] == 'run']
    pass_df = df[df['play_type'] == 'pass']
    dn1_df  = df[df['dn'] == 1]
    dn2_df  = df[df['dn'] == 2]
    dn3_df  = df[(df['dn'] == 3) & (df['dist'].notna())].copy()
    rz_df   = df[df['yard_ln'] >= 20]

    if len(dn3_df) > 0:
        dn3_df['converted'] = (
            dn3_df['result'].str.strip().str.lower().isin(
                ['complete','rush','scramble','complete, td','rush, td']
            ) & (dn3_df['gn_ls'] >= dn3_df['dist'])
        )
        conv_rate = dn3_df['converted'].mean()
    else:
        conv_rate = 0

    exp_df  = df[((df['play_type']=='run') & (df['gn_ls'] >= 12)) |
                 ((df['play_type']=='pass') & (df['gn_ls'] >= 21))]
    exp_epa = safe_mean(exp_df['epa']) if df['gn_ls'].notna().any() else 0

    return {
        '1st Down EPA': safe_mean(dn1_df['epa']),
        '2nd Down EPA': safe_mean(dn2_df['epa']),
        '3rd Down EPA': safe_mean(dn3_df['epa']),
        'Run EPA':      safe_mean(run_df['epa']),
        'Pass EPA':     safe_mean(pass_df['epa']),
        'Red Zone EPA': safe_mean(rz_df['epa']),
        'Explosive EPA':safe_mean(exp_df['epa']),
        '3rd Down Conv Rate': conv_rate,
    }

def normalize_metrics(wm, lm):
    cats = list(wm.keys())
    nw, nl = [], []
    for cat in cats:
        vals = [wm[cat], lm[cat]]
        vmin = min(vals) - abs(min(vals))*0.2
        vmax = max(vals) + abs(max(vals))*0.2
        rng  = vmax - vmin if vmax != vmin else 1
        nw.append((wm[cat] - vmin) / rng)
        nl.append((lm[cat] - vmin) / rng)
    return nw, nl

def build_traces(df):
    cats      = ['1st Down EPA','2nd Down EPA','3rd Down EPA',
                 'Run EPA','Pass EPA','Red Zone EPA','Explosive EPA','3rd Down Conv Rate']
    wm        = build_metrics(df[df['win']==1])
    lm        = build_metrics(df[df['win']==0])
    nw, nl    = normalize_metrics(wm, lm)
    rw        = [wm[c] for c in cats]
    rl        = [lm[c] for c in cats]
    cats_c    = cats + [cats[0]]
    nw_c, nl_c = nw+[nw[0]], nl+[nl[0]]
    rw_c, rl_c = rw+[rw[0]], rl+[rl[0]]
    gr        = df.groupby('game_id')['win'].first()
    nwins, nlosses = (gr==1).sum(), (gr==0).sum()
    div       = [abs(w-l) for w,l in zip(rw,rl)]
    max_div   = cats[div.index(max(div))]
    return nw_c, nl_c, rw_c, rl_c, cats_c, nwins, nlosses, max_div

fig         = go.Figure()
year_options= ['All Years'] + [str(y) for y in years]
all_traces  = []
all_titles  = []

WIN_COLOR_R  = '#2ca02c'
LOSS_COLOR_R = '#C0392B'

for option in year_options:
    subset = off_df.copy() if option == 'All Years' else off_df[off_df['year']==int(option)].copy()
    nw, nl, rw, rl, cats, nwins, nlosses, max_div = build_traces(subset)
    all_traces.append((nw, nl, rw, rl, cats))
    all_titles.append(
        f'TU Offensive EPA Win/Loss Fingerprint — {option}<br>'
        f'<sup>Record: {nwins}W – {nlosses}L  |  '
        f'Biggest divergence: <b>{max_div}</b>  |  Hover for raw values</sup>'
    )

nw, nl, rw, rl, cats = all_traces[0]
fig.add_trace(go.Scatterpolar(
    r=nw, theta=cats, fill='toself', fillcolor='rgba(44,160,44,0.15)',
    line=dict(color=WIN_COLOR_R, width=2.5), name='Wins',
    customdata=list(zip(rw, cats)),
    hovertemplate='<b>%{customdata[1]}</b><br>Value: %{customdata[0]:.3f}<extra></extra>'
))
fig.add_trace(go.Scatterpolar(
    r=nl, theta=cats, fill='toself', fillcolor='rgba(192,57,43,0.15)',
    line=dict(color=LOSS_COLOR_R, width=2.5), name='Losses',
    customdata=list(zip(rl, cats)),
    hovertemplate='<b>%{customdata[1]}</b><br>Value: %{customdata[0]:.3f}<extra></extra>'
))

buttons = []
for i, option in enumerate(year_options):
    nw, nl, rw, rl, cats = all_traces[i]
    buttons.append(dict(
        label=option, method='update',
        args=[{'r':[nw,nl], 'theta':[cats,cats],
               'customdata':[list(zip(rw,cats)), list(zip(rl,cats))]},
              {'title.text': all_titles[i]}]
    ))

fig.update_layout(
    polar=dict(
        bgcolor=LIGHT_GRAY,
        radialaxis=dict(visible=True, showticklabels=False, showline=False,
                        gridcolor='#CCCCCC', gridwidth=1, range=[0,1]),
        angularaxis=dict(tickfont=dict(size=12, color=TU_BLUE, family='Arial'),
                         linecolor='#CCCCCC', gridcolor='#CCCCCC')
    ),
    updatemenus=[dict(
        type='dropdown', direction='down',
        x=0.0, xanchor='left', y=1.12, yanchor='top',
        buttons=buttons, bgcolor='white', bordercolor='#CCCCCC',
        font=dict(color=TU_BLUE, size=11), showactive=True, active=0
    )],
    title=dict(text=all_titles[0], font=dict(size=18, color=TU_BLUE), x=0.5, xanchor='center'),
    legend=dict(font=dict(size=12, color=TU_BLUE), bgcolor='rgba(255,255,255,0.85)',
                bordercolor='#CCCCCC', borderwidth=1, x=1.1, y=1.0),
    paper_bgcolor=LIGHT_GRAY, height=680, margin=dict(t=140, b=60, l=80, r=150)
)
fig.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# CLOSING
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## Closing Note

This analysis represents the first full cycle of EPA-based performance evaluation
for Trinity University Football. The goal was never to replace the coach's eye —
it is to give it context. The numbers confirm what great coaches already feel, and
they surface what is easy to miss in the flow of a season.

The next phase of this project will extend this framework to **defensive EPA
analysis** and, ultimately, **individual player-level EPA attribution** — giving
us the ability to evaluate not just *what* we are calling, but *who* is executing
it and at what efficiency.

---
*Analysis conducted using a custom EPA model built on multi-season TUFB
play-by-play data. Model baseline: +0.028 EPA/play. All visualizations
generated in Python (matplotlib, seaborn, plotly).*
""")

# ══════════════════════════════════════════════════════════════════════════════
# REPORT BUILDER — assembles and renders the notebook
# ══════════════════════════════════════════════════════════════════════════════

def build_report(
    excel_file: str  = str(PROCESSED_FILE),
    output_dir: str  = str(DOCS_DIR),
    report_name: str = 'TU_EPA_Offensive_Report',
):
    """
    Load the source data, assemble REPORT_CELLS into a Jupyter notebook,
    execute every code cell directly in the current Python process, then
    convert to a single self-contained HTML file.

    Parameters
    ----------
    excel_file  : Full path to TUFB_EPA_Analysis_FULL_DATA.xlsx on Drive.
    output_dir  : Directory on Google Drive where the HTML will be saved.
    report_name : Base filename (no extension) for both the .ipynb and .html.
    """
    import subprocess, sys, io, base64, traceback, os
    from IPython import get_ipython
    from IPython.core.interactiveshell import InteractiveShell

    # ── 1. Install dependencies if needed ─────────────────────────────────────
    for pkg in ['nbformat', 'nbconvert']:
        try:
            __import__(pkg.replace('-', '_'))
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                                   pkg, '-q', '--quiet'])

    import nbformat
    from nbconvert import HTMLExporter

    # ── 2. Build the notebook object ──────────────────────────────────────────
    nb = nbformat.v4.new_notebook()
    nb.metadata['celltoolbar'] = 'Tags'
    nb.metadata['kernelspec'] = {
        'display_name': 'Python 3', 'language': 'python', 'name': 'python3'
    }
    nb.metadata['language_info'] = {'name': 'python', 'version': '3'}

    for cell in REPORT_CELLS:
        if cell['type'] == 'markdown':
            nb.cells.append(nbformat.v4.new_markdown_cell(cell['source']))
        elif cell['type'] == 'code':
            c = nbformat.v4.new_code_cell(cell['source'])
            c.metadata['tags'] = ['hide-input']
            nb.cells.append(c)

    # ── 3. Execute each code cell in the live Colab kernel ────────────────────
    # Rather than spawning a new kernel (which has no Drive mount, no session
    # state), we run each cell's source directly via IPython's run_cell().
    # Outputs are captured by temporarily redirecting the IPython display
    # machinery and matplotlib's figure renderer.
    print("⚙️  Executing notebook cells in current session...")

    ip = get_ipython()
    if ip is None:
        # Fallback for non-interactive environments
        ip = InteractiveShell.instance()

    # ── Load source data and inject into IPython user namespace ──────────────
    # build_report() owns the data loading so it is fully self-contained.
    # If df / tu_df are already in the caller's namespace we reuse them to
    # avoid re-reading the file; otherwise we load from excel_file.
    import inspect
    caller_frame   = inspect.stack()[1].frame
    caller_ns      = {**caller_frame.f_globals, **caller_frame.f_locals}

    if 'df' in caller_ns and 'tu_df' in caller_ns:
        _df    = caller_ns['df']
        _tu_df = caller_ns['tu_df']
        print(f"   ✓ Reusing df and tu_df already in session.")
    else:
        print(f"   ↳ Loading data from {excel_file} ...")

        # Mount Google Drive if reading from Drive and not already mounted
        if str(excel_file).startswith('/content/drive') and not os.path.isdir('/content/drive/MyDrive'):
            print("   ↳ Mounting Google Drive...")
            from google.colab import drive as _drive
            _drive.mount('/content/drive')
            print("   ✓ Drive mounted.")
        else:
            print("   ✓ Drive already mounted.")

        import pandas as _pd
        _df = _pd.read_excel(excel_file)

        # Reshape: broadcast stable team_name / opp_name from offensive rows
        _game_names = (
            _df[_df["odk"] == "o"]
            .groupby("game_id")[["team_name", "opp_name"]]
            .first()
            .reset_index()
        )
        _df = _df.drop(columns=["team_name", "opp_name"])
        _df = _df.merge(_game_names, on="game_id", how="left")

        # Trinity-only subset
        _tu_df = (
            _df[_df['team_name'] == 'TU']
            .sort_values(['game_id', 'play'])
            .reset_index(drop=True)
        )
        print(f"   ✓ Loaded df ({len(_df):,} rows) and tu_df ({len(_tu_df):,} rows).")

    # Push into IPython namespace so all executed cells see them as globals
    ip.user_ns['df']    = _df
    ip.user_ns['tu_df'] = _tu_df
    print(f"   ✓ df and tu_df available in execution namespace.")

    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.use('Agg')   # non-interactive backend so figures are captured

    import plotly.graph_objects as _go

    # Intercept fig.show() so Plotly figures are captured as HTML
    # instead of being rendered to the Colab output widget (which
    # nbconvert can't see).
    _captured_plotly = []
    _original_show   = _go.Figure.show

    def _capture_show(self, *args, **kwargs):
        _captured_plotly.append(
            self.to_html(full_html=False, include_plotlyjs='cdn')
        )

    _go.Figure.show = _capture_show   # patch

    total_code_cells = sum(1 for c in nb.cells if c['cell_type'] == 'code')
    exec_count = 1

    try:
        for i, cell in enumerate(nb.cells):
            if cell['cell_type'] != 'code':
                continue

            outputs = []
            _captured_plotly.clear()
            plt.close('all')

            # ── Run cell in the live kernel ────────────────────────────────
            result = ip.run_cell(cell.source, store_history=False, silent=False)

            if result.error_in_exec is not None:
                tb = ''.join(traceback.format_exception(
                    type(result.error_in_exec),
                    result.error_in_exec,
                    result.error_in_exec.__traceback__
                ))
                print(f"\n❌  Error in cell {exec_count}:\n{tb}")
                raise result.error_in_exec

            # ── Capture matplotlib figures ─────────────────────────────────
            for fig_num in plt.get_fignums():
                fig = plt.figure(fig_num)
                buf = io.BytesIO()
                fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
                buf.seek(0)
                img_b64 = base64.b64encode(buf.read()).decode('utf-8')
                outputs.append(nbformat.v4.new_output(
                    output_type='display_data',
                    data={'image/png': img_b64, 'text/plain': '<Figure>'},
                    metadata={}
                ))
                plt.close(fig)

            # ── Capture Plotly figures (intercepted from fig.show()) ───────
            for fig_html in _captured_plotly:
                outputs.append(nbformat.v4.new_output(
                    output_type='display_data',
                    data={'text/html': fig_html, 'text/plain': '<Plotly Figure>'},
                    metadata={}
                ))

            # ── Capture plain text / DataFrame output ──────────────────────
            if result.result is not None and not isinstance(result.result, _go.Figure):
                try:
                    outputs.append(nbformat.v4.new_output(
                        output_type='execute_result',
                        execution_count=exec_count,
                        data={'text/plain': repr(result.result)},
                        metadata={}
                    ))
                except Exception:
                    pass

            cell.outputs         = outputs
            cell.execution_count = exec_count
            exec_count += 1
            print(f"   ✓ Cell {exec_count - 1}/{total_code_cells}")

    finally:
        # Always restore the original fig.show() even if we error out
        _go.Figure.show = _original_show

    print(f"✅  All {total_code_cells} cells executed.")

    # ── 4. Save the executed .ipynb (useful for debugging) ────────────────────
    os.makedirs(output_dir, exist_ok=True)
    ipynb_path = os.path.join(output_dir, f'{report_name}.ipynb')
    with open(ipynb_path, 'w') as f:
        nbformat.write(nb, f)
    print(f"📓  Notebook saved → {ipynb_path}")

    # ── 5. Convert to HTML ────────────────────────────────────────────────────
    html_exporter = HTMLExporter()

    # Hide code input cells — output only, like R Markdown's echo=FALSE default
    html_exporter.exclude_input = True

    # Embed all resources (images, JS, CSS) inline → truly self-contained file
    html_exporter.embed_images = True

    # Custom CSS injected into the HTML header for TU branding
    html_exporter.extra_template_basedirs = []
    custom_css = """
    <style>
      body       { font-family: 'Georgia', serif; max-width: 1100px;
                   margin: auto; padding: 2rem; background: #FAFAFA; color: #1a1a1a; }
      h1         { color: #002868; border-bottom: 3px solid #C5960C;
                   padding-bottom: 0.4rem; }
      h2         { color: #002868; border-left: 5px solid #C5960C;
                   padding-left: 0.6rem; margin-top: 2.5rem; }
      h3         { color: #002868; margin-top: 1.8rem; }
      blockquote { border-left: 4px solid #C5960C; margin-left: 0;
                   padding-left: 1rem; color: #444; font-style: italic; }
      table      { border-collapse: collapse; width: 100%; margin: 1rem 0; }
      th         { background: #002868; color: white; padding: 0.5rem 1rem; }
      td         { border: 1px solid #ddd; padding: 0.4rem 1rem; }
      tr:nth-child(even) { background: #f2f2f2; }
      .jp-OutputArea-output { margin: 1rem 0; }
    </style>
    """

    (html_body, resources) = html_exporter.from_notebook_node(nb)

    # Inject custom CSS right after <head>
    html_body = html_body.replace('<head>', f'<head>{custom_css}', 1)

    html_path = os.path.join(output_dir, f'{report_name}.html')
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_body)

    print(f"📄  Report saved → {html_path}")
    print(f"\n🏈  Open the HTML file from your Google Drive to view the report.")


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == '__main__':
    build_report()

⚙️  Executing notebook cells in current session...
   ↳ Loading data from /root/work/real/TUFB_EPA_Analysis_FULL_DATA.xlsx ...
   ✓ Drive already mounted.


   ✓ Loaded df (20,344 rows) and tu_df (12,186 rows).
   ✓ df and tu_df available in execution namespace.


df:    20,344 rows × 51 columns
tu_df: 12,186 rows
   ✓ Cell 1/23
TU rows: 12,186

--- EP by odk ---
      count   mean    std    min    25%    50%    75%    max
odk                                                         
d    4803.0  2.069  1.141 -1.394  1.299  1.836  2.850  5.991
k       0.0    NaN    NaN    NaN    NaN    NaN    NaN    NaN
o    5140.0  2.556  1.332 -1.360  1.555  2.367  3.395  6.790

--- EPA by odk ---
      count   mean    std     min    25%    50%    75%    max
odk                                                          
d    4803.0 -0.173  1.267 -10.409 -0.648 -0.294  0.351  6.771
k     130.0  2.846  2.835  -7.000  3.000  3.000  3.000  7.000
o    5140.0  0.057  1.391 -10.073 -0.557 -0.060  0.653  6.315
   ✓ Cell 2/23


   ✓ Cell 3/23
   ✓ Cell 4/23


   ✓ Cell 5/23


...

   ✓ Cell 19/23


   ✓ Cell 20/23


   ✓ Cell 21/23


   ✓ Cell 22/23
   ✓ Cell 23/23
✅  All 23 cells executed.
📓  Notebook saved → /root/work/repo/docs/TU_EPA_Offensive_Report.ipynb


📄  Report saved → /root/work/repo/docs/TU_EPA_Offensive_Report.html

🏈  Open the HTML file from your Google Drive to view the report.


## Slideshow Builder
This builds the slideshow as an html and ipynb file

In [3]:
# -*- coding: utf-8 -*-
"""
epa_tufb_report_builder.py
==========================
Run this script in Google Colab to execute the full TU offensive EPA
analysis and render a single self-contained HTML report — the Python
equivalent of R Markdown.

How it works:
  1. REPORT_CELLS defines the report as an ordered list of markdown and
     code cells (identical pattern to a .Rmd file with narrative + chunks).
  2. build_report() assembles those cells into a real .ipynb using nbformat,
     executes it via nbconvert ExecutePreprocessor, then converts the
     executed notebook to a styled, self-contained HTML file.
  3. The HTML is saved to your Google Drive alongside the source data.

Usage:
  Simply run this file in Colab. The last line calls build_report().
  Output: docs/TU_EPA_Offensive_Report.html
"""

# ── Report cell registry ───────────────────────────────────────────────────────
# Each entry is a dict with keys:
#   'type' : 'markdown' | 'code'
#   'source': the cell content as a string
# Markdown cells become formatted narrative in the HTML output.
# Code cells are executed and their outputs (plots, tables, Plotly charts)
# are embedded inline.
# ──────────────────────────────────────────────────────────────────────────────

REPORT_CELLS = []

def md(source):
    """Register a markdown narrative cell."""
    REPORT_CELLS.append({'type': 'markdown', 'source': source})

def code(source):
    """Register an executable code cell."""
    REPORT_CELLS.append({'type': 'code', 'source': source})


# ══════════════════════════════════════════════════════════════════════════════
# SLIDE 1 — TITLE
# ══════════════════════════════════════════════════════════════════════════════

md("""
# TITLE_SLIDE
# Trinity University Football
## Offensive EPA Analysis

### Offensive Performance · Multi-Season Review

---

**Presented by:** Sebastian Trevino & Dr. Eduardo Cabral Balreira
""")

# ══════════════════════════════════════════════════════════════════════════════
# SLIDE 2 — WHAT WE SET OUT TO DO
# ══════════════════════════════════════════════════════════════════════════════

md("""
## What We Set Out to Do

The goal of this project is simple:

> **Move beyond the box score and understand the true value of every offensive play.**

Traditional stats — yards, completions, yards per carry — tell us *what happened.*
They don't tell us if it was *good.*

- **A 4-yard gain on 3rd & 3** → Success. Drive continues.
- **A 4-yard gain on 3rd & 10** → Failure. Drive dies.

Averages treat both plays the same. **EPA does not.**
""")

# ══════════════════════════════════════════════════════════════════════════════
# SLIDE 3 — WHY EPA
# ══════════════════════════════════════════════════════════════════════════════

md("""
## Why Expected Points Added (EPA)?

At any moment — given **down, distance, and field position** — we can estimate how many points a team is expected to score on that drive.

**EPA measures how much a single play moves that number.**

| Result | What it means |
|---|---|
| EPA > 0 | Play helped us — moved chains, created leverage |
| EPA < 0 | Play hurt us — behind schedule, stalled drive |
| EPA = 0 | Neutral — no gain, no loss in scoring probability |

**Model baseline: +0.028 EPA per play.**
**Our average: +0.057 EPA per play.**
On average, we beat the baseline and produce positive  offensive value.
Beating this consistently separates productive series from stalled drives.
""")

# ══════════════════════════════════════════════════════════════════════════════
# SLIDE 4 — WHAT THIS REPORT ANSWERS
# ══════════════════════════════════════════════════════════════════════════════

md("""
## What This Report Answers

Seven questions every offensive staff should be asking:

- **Are we generating positive value** play-to-play?
- **Where situationally do we succeed** — and where do we fail?
- **Are we calling the right plays** in the right situations?
- **Does our scheme** — personnel and formation — produce value?
- **Are we generating explosive plays**, and from where on the field?
- **How do our drives actually unfold** from first play to last?
- **What separates our wins from our losses?**
""")

# ══════════════════════════════════════════════════════════════════════════════
# SETUP# ══════════════════════════════════════════════════════════════════════════════
# SETUP
# ══════════════════════════════════════════════════════════════════════════════

md("## HIDDEN_SLIDE Setup — Libraries & Data Load")

code("""
import os
import re
import glob
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.io import to_html

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# ── Global style constants ─────────────────────────────────────────────────
TU_BLUE        = '#002868'
TU_GOLD        = '#C5960C'
WIN_COLOR      = '#1D9E75'
LOSS_COLOR     = '#C0392B'
RED            = '#C0392B'
LIGHT_GRAY     = '#F5F5F5'
MODEL_BASELINE = 0.028

# df and tu_df are injected from the calling session by build_report().
# No file I/O needed — the data is already in memory.
print(f"df:    {len(df):,} rows × {df.shape[1]} columns")
print(f"tu_df: {len(tu_df):,} rows")
""")

# ══════════════════════════════════════════════════════════════════════════════
# DATA RESHAPING
# ══════════════════════════════════════════════════════════════════════════════

md("## HIDDEN_SLIDE Data Reshaping")

code("""
game_names = (
    df[df["odk"] == "o"]
    .groupby("game_id")[["team_name", "opp_name"]]
    .first()
    .reset_index()
)
df = df.drop(columns=["team_name", "opp_name"])
df = df.merge(game_names, on="game_id", how="left")

# Build Trinity-only dataframe
tu_df = (
    df[df['team_name'] == 'TU']
    .sort_values(['game_id', 'play'])
    .reset_index(drop=True)
)
print(f"TU rows: {len(tu_df):,}")

print("\\n--- EP by odk ---")
print(tu_df.groupby('odk')['ep'].describe().round(3))
print("\\n--- EPA by odk ---")
print(tu_df.groupby('odk')['epa'].describe().round(3))
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — EPA BY DOWN
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 1 · EPA by Down
> **So what?** Negative 1st down EPA = playing from behind all game. Negative 3rd down EPA = drives dying.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()

down_df = (
    off_df[off_df['dn'].isin([1, 2, 3, 4])]
    .groupby('dn')['epa']
    .agg(['mean', 'sem', 'count'])
    .reset_index()
)
down_df['dn_label'] = down_df['dn'].map(
    {1: '1st Down', 2: '2nd Down', 3: '3rd Down', 4: '4th Down'}
)

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(
    down_df['dn_label'], down_df['mean'],
    color=[TU_BLUE if v >= 0 else RED for v in down_df['mean']],
    edgecolor='white', linewidth=1.2, width=0.5
)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=9, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, down_df.itertuples()):
    label_y = row.mean + 0.008 if row.mean >= 0 else row.mean - 0.008
    va      = 'bottom' if row.mean >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2, label_y,
            f'μ={row.mean:+.3f}', ha='center', va=va,
            fontsize=15, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x() + bar.get_width() / 2, ax.get_ylim()[0] - 0.025,
            f'n={row.count}', ha='center', va='bottom', fontsize=12, color='gray')

ax.set_title('Trinity Offensive EPA by Down', fontsize=24,
             fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Down', fontsize=15, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=15, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#CCCCCC')
ax.set_ylim(bottom=-0.15, top=0.15)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — EPA BY OPPONENT
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 2 · EPA vs. Our Opponents
> **So what?** Green = win, Red = loss. Consistent negative EPA vs. a specific opponent signals a schematic problem worth targeting.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()

game_meta = (
    off_df.groupby('game_id')
    .agg(opp_name=('opp_name','first'), game_date=('game_date','first'),
         win=('win','first'), mean_epa=('epa','mean'),
         sem_epa=('epa','sem'), n_plays=('epa','count'))
    .reset_index()
)
game_meta['game_date_parsed'] = pd.to_datetime(game_meta['game_date'], format='%m/%d/%Y')
game_meta['year'] = game_meta['game_date_parsed'].dt.year
game_meta['label'] = game_meta.apply(
    lambda r: f"{r['opp_name']} ({r['year']} {'W' if r['win']==1 else 'L'})", axis=1
)

all_years    = sorted(game_meta['year'].unique())
all_opps     = sorted(game_meta['opp_name'].unique())
overall_mean = off_df['epa'].mean()

fig       = go.Figure()
trace_map = {}

def make_trace(subset, sort_by, ascending, scope_key, visible):
    subset = subset.sort_values(sort_by, ascending=ascending).reset_index(drop=True)
    colors = [WIN_COLOR if w == 1 else LOSS_COLOR for w in subset['win']]
    hover  = [f"<b>{r['label']}</b><br>EPA/play: {r['mean_epa']:+.3f}<br>Plays: {int(r['n_plays'])}"
              for _, r in subset.iterrows()]
    idx = len(fig.data)
    fig.add_trace(go.Bar(
        x=subset['mean_epa'], y=subset['label'], orientation='h',
        marker_color=colors, marker_line_color='white', marker_line_width=1.2,
        text=hover, hovertemplate='%{text}<extra></extra>',
        visible=visible, name=scope_key, showlegend=False,
    ))
    trace_map[scope_key] = idx

season_scopes = ['All'] + [str(y) for y in all_years]
for scope in season_scopes:
    sub = game_meta if scope == 'All' else game_meta[game_meta['year'] == int(scope)]
    make_trace(sub.copy(), 'mean_epa', True, f'season_{scope}', visible=(scope == 'All'))

for opp in all_opps:
    make_trace(game_meta[game_meta['opp_name'] == opp].copy(),
               'game_date_parsed', True, f'opp_{opp}', visible=False)

total_traces = len(fig.data)
make_vis = lambda k: [i == trace_map[k] for i in range(total_traces)]

season_buttons = []
for scope in season_scopes:
    sub = game_meta if scope == 'All' else game_meta[game_meta['year'] == int(scope)]
    nw, nl = int(sub['win'].sum()), len(sub) - int(sub['win'].sum())
    season_buttons.append(dict(
        label=scope, method='update',
        args=[{'visible': make_vis(f'season_{scope}')},
              {'title.text': f'<b>TU Offensive EPA per Play by Opponent</b> — '
               f'{"All Seasons" if scope=="All" else scope}<br>'
               f'<sup>{len(sub)} games · {nw}W–{nl}L · Avg: {sub["mean_epa"].mean():+.3f}</sup>',
               'yaxis.autorange': True}]
    ))

opp_buttons = []
for opp in all_opps:
    sub = game_meta[game_meta['opp_name'] == opp].sort_values('game_date_parsed')
    nw, nl = int(sub['win'].sum()), len(sub) - int(sub['win'].sum())
    opp_buttons.append(dict(
        label=opp, method='update',
        args=[{'visible': make_vis(f'opp_{opp}')},
              {'title.text': f'<b>TU vs. {opp} — All Seasons</b><br>'
               f'<sup>{len(sub)} games · {nw}W–{nl}L · Avg: {sub["mean_epa"].mean():+.3f}</sup>',
               'yaxis.autorange': True}]
    ))

fig.add_vline(x=overall_mean, line_dash='dash', line_color=TU_BLUE, line_width=1.5,
              opacity=0.6, annotation_text=f'Season avg: {overall_mean:+.3f}',
              annotation_position='top', annotation_font=dict(color=TU_BLUE, size=10))
fig.add_vline(x=0, line_dash='dot', line_color='black', line_width=1, opacity=0.35)
fig.add_vline(x=MODEL_BASELINE, line_dash='dashdot', line_color=TU_GOLD, line_width=2,
              opacity=0.8, annotation_text='Model Baseline (+0.028)',
              annotation_position='bottom', annotation_font=dict(color=TU_GOLD, size=10))

n_all = len(game_meta)
nw_all, nl_all = int(game_meta['win'].sum()), n_all - int(game_meta['win'].sum())
fig.update_layout(
    title=dict(
        text=f'<b>TU Offensive EPA per Play by Opponent</b> — All Seasons<br>'
             f'<sup>{n_all} games · {nw_all}W–{nl_all}L · Avg: {game_meta["mean_epa"].mean():+.3f}</sup>',
        font=dict(size=16, color=TU_BLUE), x=0.5, xanchor='center'),
    xaxis=dict(title='Average EPA per Play', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD'),
    yaxis=dict(tickfont=dict(color=TU_BLUE, size=11), gridcolor='#DDDDDD', autorange=True),
    updatemenus=[
        dict(type='buttons', direction='right', showactive=True,
             x=0.0, xanchor='left', y=-0.08, yanchor='top',
             buttons=season_buttons, bgcolor='white', bordercolor='#CCCCCC',
             font=dict(color=TU_BLUE, size=10), pad=dict(t=3,b=3)),
        dict(type='dropdown', direction='up', showactive=True,
             x=0.55, xanchor='left', y=-0.08, yanchor='top',
             buttons=opp_buttons, bgcolor='white', bordercolor='#CCCCCC',
             font=dict(color=TU_BLUE, size=10), pad=dict(t=3,b=3)),
    ],
    annotations=[
        dict(text='Season:', showarrow=False, x=0.0, xanchor='left',
             y=-0.03, yanchor='bottom', xref='paper', yref='paper',
             font=dict(size=9, color='#555555')),
        dict(text='Opponent:', showarrow=False, x=0.55, xanchor='left',
             y=-0.03, yanchor='bottom', xref='paper', yref='paper',
             font=dict(size=9, color='#555555')),
    ],
    plot_bgcolor=LIGHT_GRAY, paper_bgcolor=LIGHT_GRAY, hovermode='closest',
    height=750, margin=dict(t=60, b=90, l=160, r=40),
)
fig.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — 3RD DOWN PROBLEM
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 3 · The 3rd Down Problem
> **So what?** Short (≤3 yds) should convert 60–70%+. Long (11+) = drive over. The best fix is avoiding long 3rd downs — not the play call once you're in them.
""")

code("""
third_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) &
    (tu_df['dist'].notna()) & (tu_df['dn'] == 3)
].copy()

bins   = [0, 3, 7, 10, float('inf')]
labels = ['Short\\n(≤3 yds)', 'Medium\\n(4–7 yds)',
          'Standard\\n(8–10 yds)', 'Long\\n(11+ yds)']
third_df['dist_bucket'] = pd.cut(third_df['dist'], bins=bins,
                                  labels=labels, include_lowest=True)

convert_results = ['complete', 'rush', 'scramble', 'complete, td', 'rush, td']
third_df['converted'] = (
    third_df['result'].str.strip().str.lower().isin(convert_results) &
    (third_df['gn_ls'] >= third_df['dist'])
)

bucket_df = (
    third_df.groupby('dist_bucket', observed=True)
    .agg(mean_epa=('epa','mean'), sem_epa=('epa','sem'),
         count=('epa','count'), conv_rate=('converted','mean'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(22, 14), gridspec_kw={'width_ratios': [1.2, 1]})
fig.patch.set_facecolor(LIGHT_GRAY)

ax1 = axes[0]
ax1.set_facecolor(LIGHT_GRAY)
bucket_colors = {
    b: (TU_BLUE if bucket_df.loc[bucket_df['dist_bucket']==b,'mean_epa'].values[0] >= 0
        else RED)
    for b in labels
}
sns.boxplot(data=third_df, x='dist_bucket', y='epa', order=labels,
            palette=bucket_colors, width=0.45, linewidth=1.3,
            flierprops=dict(marker='o', markersize=3, alpha=0.3,
                            linestyle='none', markeredgewidth=0), ax=ax1)

for i, bucket in enumerate(labels):
    subset = third_df[third_df['dist_bucket'] == bucket]['epa']
    if len(subset) == 0:
        continue
    mean_val    = subset.mean()
    Q3          = subset.quantile(0.75)
    whisker_top = subset[subset <= Q3 + 1.5*(Q3 - subset.quantile(0.25))].max()
    ax1.text(i, whisker_top + 0.12, f'μ={mean_val:+.3f}', ha='center', va='bottom',
             fontsize=15, fontweight='bold',
             color=TU_BLUE if mean_val >= 0 else RED)
    ax1.text(i, ax1.get_ylim()[0] + 0.1, f'n={len(subset)}', ha='center',
             va='bottom', fontsize=12, color='gray')

ax1.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax1.set_title('3rd Down EPA Distribution\\nby Distance Bucket',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=10)
ax1.set_xlabel('Distance to Go', fontsize=11, color=TU_BLUE)
ax1.set_ylabel('EPA per Play', fontsize=11, color=TU_BLUE)
ax1.tick_params(colors=TU_BLUE)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
x, width = range(len(bucket_df)), 0.35
bars1 = ax2.bar([i - width/2 for i in x], bucket_df['conv_rate']*100, width,
                color=TU_GOLD, edgecolor='white', linewidth=1.2, label='Conversion Rate (%)')
bars2 = ax2.bar([i + width/2 for i in x], bucket_df['mean_epa'], width,
                color=[TU_BLUE if v >= 0 else RED for v in bucket_df['mean_epa']],
                edgecolor='white', linewidth=1.2, label='Avg EPA')

for bar, row in zip(bars1, bucket_df.itertuples()):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{row.conv_rate*100:.1f}%', ha='center', va='bottom',
             fontsize=15, fontweight='bold', color=TU_GOLD)

for bar, row in zip(bars2, bucket_df.itertuples()):
    label_y = row.mean_epa + 0.3 if row.mean_epa >= 0 else row.mean_epa - 0.3
    ax2.text(bar.get_x()+bar.get_width()/2, label_y,
             f'μ={row.mean_epa:+.3f}', ha='center',
             va='bottom' if row.mean_epa >= 0 else 'top',
             fontsize=15, fontweight='bold',
             color=TU_BLUE if row.mean_epa >= 0 else RED)

ax2.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax2.set_xticks(list(x))
ax2.set_xticklabels(bucket_df['dist_bucket'], color=TU_BLUE)
ax2.set_title('3rd Down Conversion Rate &\\nAvg EPA by Distance',
              fontsize=13, fontweight='bold', color=TU_BLUE, pad=10)
ax2.set_xlabel('Distance to Go', fontsize=11, color=TU_BLUE)
ax2.tick_params(colors=TU_BLUE)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')
ax2.legend(fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')

fig.suptitle(
    f'TU 3rd Down Conversion Efficiency\\n'
    f'Overall conversion rate: {third_df["converted"].mean()*100:.1f}%  |  '
    f'Overall avg EPA: {third_df["epa"].mean():+.3f}',
    fontsize=18, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — RUN VS PASS EPA BY DOWN
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 4 · Run vs. Pass EPA by Down
> **So what?** 1st down run EPA is the foundation metric. If it's negative, we're playing from behind before the drive has a chance.
""")

code("""
rp_down = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['dn'].isin([1, 2, 3])) &
    (tu_df['play_type'].isin(['run', 'pass'])) & (tu_df['epa'].notna())
].copy()

down_order  = [1, 2, 3]
down_labels = ['1st Down', '2nd Down', '3rd Down']

summary = (
    rp_down.groupby(['dn', 'play_type'])
    .agg(mean_epa=('epa','mean'), sem=('epa','sem'), count=('epa','count'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
PLAY_COLOR = {'run': TU_BLUE, 'pass': TU_GOLD}
x, bar_width = np.arange(len(down_order)), 0.38

ax1 = axes[0]
ax1.set_facecolor(LIGHT_GRAY)
for play_type, offset in [('run', -bar_width/2), ('pass', bar_width/2)]:
    data  = summary[summary['play_type']==play_type].set_index('dn').reindex(down_order)
    vals  = data['mean_epa'].values.astype(float)
    color = PLAY_COLOR[play_type]
    bars  = ax1.bar(x + offset, vals, bar_width, color=color,
                    edgecolor='white', linewidth=1.2)
    for bar_obj, val in zip(bars, vals):
        if np.isnan(val): continue
        ax1.text(bar_obj.get_x()+bar_obj.get_width()/2,
                 val + 0.012 if val >= 0 else val - 0.01,
                 f'μ={val:+.3f}', ha='center',
                 va='bottom' if val >= 0 else 'top',
                 fontsize=12, fontweight='bold',
                 color=RED if val < 0 else color)

ax1.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax1.axhline(MODEL_BASELINE, color='#888888', linewidth=1.3, linestyle='-.',
            alpha=0.75, label='Model Baseline (+0.028)')
ax1.set_xticks(x); ax1.set_xticklabels(down_labels, fontsize=11, color=TU_BLUE)
ax1.tick_params(colors=TU_BLUE)
ax1.set_xlabel('Down', fontsize=12, color=TU_BLUE)
ax1.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax1.set_title('Run vs. Pass Mean EPA by Down', fontsize=13,
              fontweight='bold', color=TU_BLUE, pad=12)
ax1.legend(handles=[mpatches.Patch(color=TU_BLUE, label='Run'),
                     mpatches.Patch(color=TU_GOLD, label='Pass'),
                     mpatches.Patch(color='#888888', label='Model Baseline (+0.028)')],
           fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
for play_type, offset in [('run', -bar_width/2), ('pass', bar_width/2)]:
    data  = summary[summary['play_type']==play_type].set_index('dn').reindex(down_order)
    vals  = data['count'].values.astype(float)
    color = PLAY_COLOR[play_type]
    bars  = ax2.bar(x + offset, vals, bar_width, color=color,
                    edgecolor='white', linewidth=1.2)
    for bar_obj, val in zip(bars, vals):
        if not np.isnan(val):
            ax2.text(bar_obj.get_x()+bar_obj.get_width()/2,
                     bar_obj.get_height()+0.8, f'{int(val)}',
                     ha='center', va='bottom', fontsize=12,
                     fontweight='bold', color=color)

ax2.set_xticks(x); ax2.set_xticklabels(down_labels, fontsize=11, color=TU_BLUE)
ax2.tick_params(colors=TU_BLUE)
ax2.set_xlabel('Down', fontsize=12, color=TU_BLUE)
ax2.set_ylabel('Number of Plays', fontsize=12, color=TU_BLUE)
ax2.set_title('Run vs. Pass Play Volume by Down', fontsize=13,
              fontweight='bold', color=TU_BLUE, pad=12)
ax2.legend(handles=[mpatches.Patch(color=TU_BLUE, label='Run'),
                     mpatches.Patch(color=TU_GOLD, label='Pass')],
           fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')

total_runs   = int(summary[summary['play_type']=='run']['count'].sum())
total_passes = int(summary[summary['play_type']=='pass']['count'].sum())
fig.suptitle(f'TU Offensive Run vs. Pass EPA by Down\\nTotal Runs: {total_runs}  |  Total Passes: {total_passes}',
             fontsize=18, fontweight='bold', color=TU_BLUE, y=1.01)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — 3RD DOWN RUN VS PASS
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 5 · 3rd Down: Run vs. Pass
> **So what?** If we pass 90%+ on 3rd-and-long, defenses know it's coming. Volume + EPA together tell the full story.
""")

code("""
third_down = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['dn'] == 3) &
    (tu_df['play_type'].isin(['run', 'pass'])) &
    (tu_df['epa'].notna()) & (tu_df['dist'].notna())
].copy()

def dist_bucket(d):
    if d <= 3:   return '1–3\\n(Short)'
    elif d <= 6:  return '4–6\\n(Medium)'
    elif d <= 10: return '7–10\\n(Long)'
    else:         return '11+\\n(Very Long)'

bucket_order = ['1–3\\n(Short)', '4–6\\n(Medium)', '7–10\\n(Long)', '11+\\n(Very Long)']
third_down['dist_bucket'] = pd.Categorical(
    third_down['dist'].apply(dist_bucket), categories=bucket_order, ordered=True
)

summary = (
    third_down.groupby(['dist_bucket', 'play_type'], observed=True)
    .agg(mean_epa=('epa','mean'), count=('epa','count'), sem=('epa', lambda x: x.sem()))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(24, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
PLAY_COLOR = {'run': TU_BLUE, 'pass': TU_GOLD}
x, bar_width = np.arange(len(bucket_order)), 0.45

for ax, metric, ylabel, title_suffix in zip(
    axes, ['mean_epa', 'count'],
    ['Average EPA per Play', 'Number of Plays'],
    ['Mean EPA by Distance Bucket', 'Play Volume by Distance Bucket']
):
    ax.set_facecolor(LIGHT_GRAY)
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#CCCCCC')

    for play_type, offset in [('run', -bar_width/2), ('pass', bar_width/2)]:
        data = summary[summary['play_type'] == play_type].set_index('dist_bucket').reindex(bucket_order)
        vals = data[metric].values.astype(float)
        color = PLAY_COLOR[play_type]

        bars = ax.bar(x + offset, vals, bar_width, color=color,
                      edgecolor='white', linewidth=1.2, label=play_type.capitalize())

        for bar_obj, val in zip(bars, vals):
            if np.isnan(val):
                continue
            if metric == 'mean_epa':
                ax.text(bar_obj.get_x()+bar_obj.get_width()/2,
                        val + 0.01 if val >= 0 else val - 0.01,
                        f'μ={val:+.3f}', ha='center',
                        va='bottom' if val >= 0 else 'top',
                        fontsize=12, fontweight='bold',
                        color=RED if val < 0 else color)
            else:
                ax.text(bar_obj.get_x()+bar_obj.get_width()/2,
                        bar_obj.get_height()+0.5, f'{int(val)}',
                        ha='center', va='bottom', fontsize=12,
                        fontweight='bold', color=color)

    if metric == 'mean_epa':
        ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(bucket_order, fontsize=10, color=TU_BLUE)
    ax.tick_params(colors=TU_BLUE)
    ax.set_xlabel('Distance to Go', fontsize=12, color=TU_BLUE)
    ax.set_ylabel(ylabel, fontsize=12, color=TU_BLUE)
    ax.set_title(f'3rd Down {title_suffix}', fontsize=13,
                 fontweight='bold', color=TU_BLUE, pad=12)
    ax.legend(handles=[mpatches.Patch(color=TU_BLUE, label='Run'),
                        mpatches.Patch(color=TU_GOLD, label='Pass')],
              fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')

total_runs   = int(summary[summary['play_type']=='run']['count'].sum())
total_passes = int(summary[summary['play_type']=='pass']['count'].sum())
fig.suptitle(f'Trinity 3rd Down — Run vs. Pass EPA by Distance\\nRuns: {total_runs}  |  Passes: {total_passes}',
             fontsize=18, fontweight='bold', color=TU_BLUE, y=1.01)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — FIELD ZONE
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 6 · EPA by Field Zone
> **So what?** Red zone EPA is the most consequential number here. Below baseline = points left on the field every week.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['yards_to_go'].notna())
].copy()

bins   = [0, 20, 40, 60, 75, 99]
labels = ['Redzone\\n1–20 yds', 'Opp Territory\\n21–40 yds',
          'Midfield\\n41–60 yds', 'Own Territory\\n61–75 yds', 'Own Territory\\n76–99 yds']
off_df['field_zone'] = pd.cut(off_df['yards_to_go'], bins=bins,
                               labels=labels, include_lowest=True)

zone_df = (
    off_df.groupby('field_zone', observed=True)['epa']
    .agg(['mean','sem','count']).reset_index()
).iloc[::-1].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(zone_df['field_zone'], zone_df['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in zone_df['mean']],
              edgecolor='white', linewidth=1.2, width=0.55)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, zone_df.itertuples()):
    label_y = row.mean + 0.01 if row.mean >= 0 else row.mean - 0.02
    ax.text(bar.get_x()+bar.get_width()/2, label_y, f'μ={row.mean:+.3f}',
            ha='center', va='bottom' if row.mean >= 0 else 'top',
            fontsize=15, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0] - 0.25,
            f'n={row.count}', ha='center', va='bottom', fontsize=12, color='gray')

ax.set_ylim(bottom=-0.40, top=0.32)
ax.annotate('← Own Territory', xy=(0.02, -0.15), xycoords='axes fraction',
            fontsize=9, color='gray', style='italic')
ax.annotate('Endzone →', xy=(0.80, -0.15), xycoords='axes fraction',
            fontsize=9, color='gray', style='italic')
ax.set_title('TU Offensive EPA by Field Zone', fontsize=24,
             fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Field Zone  (yards to go to endzone)', fontsize=18,
              color=TU_BLUE, labelpad=20)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.tick_params(axis='x', labelsize=15, colors=TU_BLUE)
ax.tick_params(axis='y', colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — RED ZONE BY DOWN
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 7 · Red Zone EPA by Down
> **So what?** Negative 3rd down red zone EPA = field goals instead of touchdowns. Every missed TD in the red zone directly costs win probability.
""")

code("""
rz_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) &
    (tu_df['yards_to_go'].notna()) & (tu_df['dn'].notna()) &
    (tu_df['yards_to_go'] <= 20)
].copy()

rz_df['down_group'] = rz_df['dn'].map(
    {1.0: '1st Down', 2.0: '2nd Down', 3.0: '3rd Down', 4.0: '4th Down'}
)
rz_df = rz_df[rz_df['down_group'].notna()]
down_order = ['1st Down', '2nd Down', '3rd Down', '4th Down']

down_agg = (
    rz_df.groupby('down_group', observed=True)['epa']
    .agg(['mean','sem','count']).reindex(down_order).reset_index()
)

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(down_agg['down_group'], down_agg['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in down_agg['mean']],
              edgecolor='white', linewidth=1.2, width=0.5)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, down_agg.itertuples()):
    if pd.isna(row.mean): continue
    ax.text(bar.get_x()+bar.get_width()/2,
            row.mean + 0.008 if row.mean >= 0 else row.mean - 0.015,
            f'μ={row.mean:+.3f}', ha='center',
            va='bottom' if row.mean >= 0 else 'top',
            fontsize=14, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0] - 0.25,
            f'n={int(row.count)}', ha='center', va='bottom', fontsize=12, color='gray')

ax.set_ylim(bottom=-1.2, top=0.35)
ax.set_title('TU Red Zone EPA by Down  (yards to go ≤ 20)',
             fontsize=24, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Down', fontsize=15, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 8 — RED ZONE RUN VS PASS
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 8 · Red Zone: Run vs. Pass
> **So what?** 70%+ usage of one play type in the red zone is predictable. Balance keeps the defense honest even when one has a slight EPA edge.
""")

code("""
rz_rp = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['yards_to_go'].notna()) &
    (tu_df['yards_to_go'] <= 20) & (tu_df['play_type'].isin(['run', 'pass'])) &
    (tu_df['epa'].notna()) & (tu_df['dn'].isin([1, 2, 3, 4]))
].copy()

down_order  = [1, 2, 3, 4]
down_labels = ['1st Down', '2nd Down', '3rd Down', '4th Down']
PLAY_COLOR  = {'run': TU_BLUE, 'pass': TU_GOLD}

summary = (
    rz_rp.groupby(['dn', 'play_type'])
    .agg(mean_epa=('epa','mean'), sem=('epa','sem'), count=('epa','count'))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(26, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
x, bar_width = np.arange(len(down_order)), 0.38

for ax, metric, ylabel, title in zip(
    axes,
    ['mean_epa', 'count'],
    ['Average EPA per Play', 'Number of Plays'],
    ['Red Zone EPA by Down\\n& Play Type', 'Red Zone Play Volume\\nby Down & Play Type']
):
    ax.set_facecolor(LIGHT_GRAY)
    for play_type, offset in [('run', -bar_width/2), ('pass', bar_width/2)]:
        data  = summary[summary['play_type']==play_type].set_index('dn').reindex(down_order)
        vals  = data[metric].values.astype(float)
        color = PLAY_COLOR[play_type]
        bars  = ax.bar(x + offset, vals, bar_width, color=color,
                       edgecolor='white', linewidth=1.2)
        for bar_obj, val in zip(bars, vals):
            if np.isnan(val): continue
            if metric == 'mean_epa':
                ax.text(bar_obj.get_x()+bar_obj.get_width()/2,
                        val + 0.015 if val >= 0 else val - 0.015,
                        f'μ={val:+.3f}', ha='center',
                        va='bottom' if val >= 0 else 'top',
                        fontsize=12, fontweight='bold',
                        color=RED if val < 0 else color)
            elif val > 0:
                ax.text(bar_obj.get_x()+bar_obj.get_width()/2,
                        bar_obj.get_height()+0.5, f'{int(val)}',
                        ha='center', va='bottom', fontsize=12,
                        fontweight='bold', color=color)

    if metric == 'mean_epa':
        ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)

    ax.set_xticks(x); ax.set_xticklabels(down_labels, fontsize=10, color=TU_BLUE)
    ax.tick_params(colors=TU_BLUE)
    ax.set_xlabel('Down', fontsize=11, color=TU_BLUE)
    ax.set_ylabel(ylabel, fontsize=11, color=TU_BLUE)
    ax.set_title(title, fontsize=18, fontweight='bold', color=TU_BLUE, pad=12)
    ax.legend(handles=[mpatches.Patch(color=TU_BLUE, label='Run'),
                        mpatches.Patch(color=TU_GOLD, label='Pass')],
              fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#CCCCCC')

run_t = int(summary[summary['play_type']=='run']['count'].sum())
pass_t = int(summary[summary['play_type']=='pass']['count'].sum())
total  = run_t + pass_t
fig.suptitle(
    f'TU Red Zone Run vs. Pass EPA  (yards to go ≤ 20)\\n'
    f'Run: {run_t} ({run_t/total*100:.0f}%)  |  Pass: {pass_t} ({pass_t/total*100:.0f}%)  |  Total: {total}',
    fontsize=22, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 9 — EPA BY RESULT TYPE
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 9 · EPA by Result Type
> **So what?** Eliminating one sack or interception per game can outweigh several explosive plays. Scramble EPA tells you if QB mobility is a real asset.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['result'].notna())
].copy()
off_df['result'] = off_df['result'].str.strip().str.lower()

keep_results = ['rush','complete','incomplete','interception',
                'sack','scramble','complete, td','rush, td','fumble']
off_df = off_df[off_df['result'].isin(keep_results)].copy()

label_map = {
    'rush':'Rush', 'complete':'Complete', 'incomplete':'Incomplete',
    'interception':'Interception', 'sack':'Sack', 'scramble':'Scramble',
    'complete, td':'Complete TD', 'rush, td':'Rush TD', 'fumble':'Fumble'
}
off_df['result_label'] = off_df['result'].map(label_map)

result_order = (
    off_df.groupby('result_label')['epa'].median()
    .sort_values(ascending=False).index.tolist()
)
median_vals = off_df.groupby('result_label')['epa'].median()
box_colors  = {r: (TU_BLUE if median_vals[r] >= 0 else RED) for r in result_order}

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

sns.boxplot(data=off_df, x='result_label', y='epa', order=result_order,
            palette=box_colors, width=0.5, linewidth=1.3,
            flierprops=dict(marker='o', markersize=3, alpha=0.3,
                            linestyle='none', markeredgewidth=0), ax=ax)

for i, result in enumerate(result_order):
    subset      = off_df[off_df['result_label'] == result]['epa']
    Q3          = subset.quantile(0.75)
    whisker_top = subset[subset <= Q3 + 1.5*(Q3-subset.quantile(0.25))].max()
    ax.text(i, whisker_top + 0.15, f'μ={subset.mean():+.3f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold',
            color=TU_BLUE if median_vals[result] >= 0 else RED)
    ax.text(i, ax.get_ylim()[0]+0.1, f'n={len(subset)}',
            ha='center', va='bottom', fontsize=12, color='gray')

ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.set_title('TU Offensive EPA Distribution by Result Type',
             fontsize=24, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Result Type', fontsize=12, color=TU_BLUE)
ax.set_ylabel('EPA per Play', fontsize=12, color=TU_BLUE)
ax.set_ylim(bottom=ax.get_ylim()[0]-0.5, top=ax.get_ylim()[1]+1.0)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 10 — RUN VS PASS DISTRIBUTION
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 10 · Run vs. Pass Distribution
> **So what?** A wide distribution = high upside, high risk. Run EPA below zero doesn't mean stop running — it means be more selective about when.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()
rp_df  = off_df[off_df['play_type'].isin(['run','pass'])].copy()
rp_df['play_type'] = rp_df['play_type'].str.capitalize()
palette = {'Run': TU_BLUE, 'Pass': TU_GOLD}

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

sns.boxplot(data=rp_df, x='play_type', y='epa', palette=palette,
            order=['Run','Pass'], width=0.4, linewidth=1.3,
            flierprops=dict(marker='', markersize=0), ax=ax)
sns.stripplot(data=rp_df, x='play_type', y='epa', palette=palette,
              order=['Run','Pass'], size=2.5, alpha=0.35, jitter=True, ax=ax)

OFFSET = 0.3
for i, collection in enumerate(ax.collections):
    offsets = collection.get_offsets()
    offsets[:, 0] += OFFSET
    collection.set_offsets(offsets)

for i, pt in enumerate(['Run','Pass']):
    subset   = rp_df[rp_df['play_type'] == pt]
    mean_val = subset['epa'].mean()
    count    = len(subset)
    ax.plot(i + OFFSET, mean_val, marker='D', color='white', markersize=8, zorder=5)
    ax.plot(i + OFFSET, mean_val, marker='D', color='black', markersize=5, zorder=6)
    ax.text(i + OFFSET + 0.15, mean_val, f'μ={mean_val:+.3f}',
            va='center', fontsize=15, fontweight='bold', color='black')
    ax.text(i, ax.get_ylim()[0]+0.2, f'n={count}',
            ha='center', fontsize=12, color='gray')

ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.set_title('TU Offensive EPA Distribution\\nRun vs. Pass',
             fontsize=24, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Play Type', fontsize=12, color=TU_BLUE)
ax.set_ylabel('EPA per Play', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 11 — EPA BY DISTANCE
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 11 · EPA by Distance to Go
> **So what?** Frequent long-distance situations (11+) means 1st/2nd down broke down. Preventing long-distance matters more than the play call once you're in it.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['dist'].notna())
].copy()

bins   = [0, 3, 7, 10, float('inf')]
labels = ['Short\\n(≤3 yds)', 'Medium\\n(4–7 yds)',
          'Standard\\n(8–10 yds)', 'Long\\n(11+ yds)']
off_df['dist_bucket'] = pd.cut(off_df['dist'], bins=bins,
                                labels=labels, include_lowest=True)

dist_df = (
    off_df.groupby('dist_bucket', observed=True)['epa']
    .agg(['mean','sem','count']).reset_index()
)

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(dist_df['dist_bucket'], dist_df['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in dist_df['mean']],
              edgecolor='white', linewidth=1.2, width=0.55)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, dist_df.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2,
            row.mean + 0.008 if row.mean >= 0 else row.mean - 0.001,
            f'μ={row.mean:+.3f}', ha='center',
            va='bottom' if row.mean >= 0 else 'top',
            fontsize=15, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0]+0.001,
            f'n={row.count}', ha='center', va='bottom', fontsize=12, color='gray')

ax.set_title('TU Offensive EPA by Distance-to-Go',
             fontsize=22, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Distance to Go', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 12 — PERSONNEL & FORMATION
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 12a · EPA by Personnel Grouping
> **So what?** Your highest-EPA personnel package should be getting the most snaps. If it isn't, that's an immediate correction.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['personnel'].notna())
].copy()

pers_df = (
    off_df.groupby('personnel')['epa']
    .agg(['mean','sem','count']).reset_index()
)
pers_df = pers_df[pers_df['count'] >= 20].sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(pers_df['personnel'], pers_df['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in pers_df['mean']],
              edgecolor='white', linewidth=1.2, width=0.55)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, pers_df.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2,
            row.mean + 0.008 if row.mean >= 0 else row.mean - 0.008,
            f'μ={row.mean:+.3f}', ha='center',
            va='bottom' if row.mean >= 0 else 'top',
            fontsize=12, fontweight='bold',
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0]-0.1,
            f'n={row.count}', ha='center', va='bottom', fontsize=12, color='gray')

ax.set_title('TU Offensive EPA by Personnel Grouping',
             fontsize=24, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Personnel', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.set_ylim(bottom=ax.get_ylim()[0]-0.3, top=ax.get_ylim()[1]+0.3)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

md("""
## 12b · EPA by Formation
> **So what?** High usage + low EPA = fixable inefficiency. Find the formations that outperform and replicate them.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) & (tu_df['off_form'].notna())
].copy()

def normalize_formation(form):
    form = str(form).strip().upper()
    if form in ['RIGHT', 'LEFT']:
        return form
    form = re.sub(r'\\bRIGHT\\b', '', form)
    form = re.sub(r'\\bLEFT\\b', '', form)
    return ' '.join(form.split()) or 'OTHER'

off_df['form_normalized'] = off_df['off_form'].apply(normalize_formation)

form_df = (
    off_df.groupby('form_normalized')['epa']
    .agg(['mean','sem','count']).reset_index()
)
form_df = form_df[form_df['count'] >= 20].sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

bars = ax.bar(form_df['form_normalized'], form_df['mean'],
              color=[TU_BLUE if v >= 0 else RED for v in form_df['mean']],
              edgecolor='white', linewidth=1.2, width=0.3)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
           alpha=0.85, label='Model Baseline (+0.028)')
ax.legend(fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')

for bar, row in zip(bars, form_df.itertuples()):
    ax.text(bar.get_x()+bar.get_width()/2,
            row.mean + 0.008 if row.mean >= 0 else row.mean - 0.008,
            f'μ={row.mean:+.3f}', ha='center',
            va='bottom' if row.mean >= 0 else 'top',
            fontsize=12, fontweight='bold', rotation=90,
            color=TU_BLUE if row.mean >= 0 else RED)
    ax.text(bar.get_x()+bar.get_width()/2, ax.get_ylim()[0]-0.2,
            f'n={row.count}', ha='center', va='bottom', fontsize=12, color='gray')

ax.set_title('TU Offensive EPA by Formation',
             fontsize=24, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Formation', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Average EPA per Play', fontsize=12, color=TU_BLUE)
ax.set_ylim(bottom=ax.get_ylim()[0]-0.3, top=ax.get_ylim()[1]+0.3)
ax.tick_params(axis='x', colors=TU_BLUE, rotation=90)
ax.tick_params(axis='y', colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

md("""
## 12c · Personnel × Formation (Interactive)
> **So what?** Click a personnel group to see formation EPA. Use this before game-planning — find the combinations that work and build around them.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) &
    (tu_df['personnel'].notna()) & (tu_df['off_form'].notna())
].copy()

def normalize_formation(form):
    form = str(form).strip().upper()
    if form in ['RIGHT', 'LEFT']:
        return form
    form = re.sub(r'\\bRIGHT\\b', '', form)
    form = re.sub(r'\\bLEFT\\b', '', form)
    return ' '.join(form.split()) or 'OTHER'

off_df['form_normalized'] = off_df['off_form'].apply(normalize_formation)

combo_df = (
    off_df.groupby(['personnel', 'form_normalized'])['epa']
    .agg(['mean','sem','count']).reset_index()
    .rename(columns={'form_normalized': 'formation'})
)
combo_df = combo_df[combo_df['count'] >= 10].copy()

personnel_list = (
    combo_df.groupby('personnel')['mean'].mean()
    .sort_values(ascending=False).index.tolist()
)
pers_colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
               '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf']
pers_color_map = {p: pers_colors[i % len(pers_colors)]
                  for i, p in enumerate(personnel_list)}

fig = go.Figure()
for pers in personnel_list:
    d = combo_df[combo_df['personnel'] == pers].sort_values('mean', ascending=False)
    hover = [f"<b>{pers} | {r['formation']}</b><br>μ EPA: {r['mean']:+.3f}<br>n={int(r['count'])}"
             for _, r in d.iterrows()]
    fig.add_trace(go.Bar(
        x=d['formation'], y=d['mean'], name=pers,
        marker_color=[TU_BLUE if v >= 0 else RED for v in d['mean']],
        marker_line_color='white', marker_line_width=1.2,
        text=hover, hovertemplate='%{text}<extra></extra>',
        visible='legendonly', legendgroup=pers, showlegend=True
    ))

fig.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4, line_width=1.5)
n_traces = len(fig.data)
fig.update_layout(
    title=dict(
        text='TU Offensive EPA by Personnel & Formation<br>'
             '<sup>Click a personnel group to show its formations</sup>',
        font=dict(size=17, color=TU_BLUE), x=0.5, xanchor='center'),
    xaxis=dict(title='Formation', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), tickangle=45, gridcolor='#DDDDDD'),
    yaxis=dict(title='Average EPA per Play', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD'),
    barmode='group',
    legend=dict(title=dict(text='Personnel', font=dict(color=TU_BLUE, size=11)),
                bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC',
                borderwidth=1, font=dict(size=10)),
    updatemenus=[dict(
        type='buttons', showactive=False,
        x=1.0, xanchor='right', y=-0.08, yanchor='top',
        buttons=[
            dict(label='Show All', method='restyle',
                 args=[{'visible': True}, list(range(n_traces))]),
            dict(label='Hide All', method='restyle',
                 args=[{'visible': 'legendonly'}, list(range(n_traces))])
        ],
        bgcolor='white', bordercolor='#CCCCCC', font=dict(color=TU_BLUE, size=10)
    )],
    plot_bgcolor=LIGHT_GRAY, paper_bgcolor=LIGHT_GRAY,
    height=700, margin=dict(t=100, b=120, l=70, r=20)
)
fig.show()
""")

md("""
## 12d · Explosive Rate by Personnel
> **So what?** High EPA + high explosive rate = your best personnel package. Use it more.
""")

code("""
exp_pers = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['play_type'].isin(['run','pass'])) &
    (tu_df['gn_ls'].notna()) & (tu_df['personnel'].notna()) & (tu_df['epa'].notna())
].copy()

exp_pers['explosive'] = (
    ((exp_pers['play_type'] == 'run')  & (exp_pers['gn_ls'] >= 12)) |
    ((exp_pers['play_type'] == 'pass') & (exp_pers['gn_ls'] >= 21))
)

pers_df = (
    exp_pers.groupby('personnel')
    .agg(exp_rate=('explosive','mean'), exp_count=('explosive','sum'),
         total=('explosive','count'), mean_epa=('epa','mean'))
    .reset_index()
)
pers_df = pers_df[pers_df['total'] >= 20].copy()
pers_df['exp_rate_pct'] = pers_df['exp_rate'] * 100
pers_df = pers_df.sort_values('exp_rate_pct', ascending=False).reset_index(drop=True)
overall_exp_rate = exp_pers['explosive'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(26, 14))
fig.patch.set_facecolor(LIGHT_GRAY)

ax1 = axes[0]
ax1.set_facecolor(LIGHT_GRAY)
bar_colors = [TU_GOLD if v >= overall_exp_rate else TU_BLUE
              for v in pers_df['exp_rate_pct']]
bars = ax1.bar(pers_df['personnel'], pers_df['exp_rate_pct'],
               color=bar_colors, edgecolor='white', linewidth=1.2, width=0.55)
ax1.axhline(overall_exp_rate, color=RED, linewidth=1.4, linestyle='--', alpha=0.75)
ax1.text(len(pers_df)-0.5, overall_exp_rate+0.3, f'Avg: {overall_exp_rate:.1f}%',
         ha='right', va='bottom', fontsize=12, color=RED, fontweight='bold')
for bar, row in zip(bars, pers_df.itertuples()):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{row.exp_rate_pct:.1f}%', ha='center', va='bottom',
             fontsize=12, fontweight='bold',
             color=TU_GOLD if row.exp_rate_pct >= overall_exp_rate else TU_BLUE)
ax1.set_title('Explosive Play Rate\\nby Personnel Grouping',
              fontsize=18, fontweight='bold', color=TU_BLUE, pad=12)
ax1.set_xlabel('Personnel', fontsize=11, color=TU_BLUE)
ax1.set_ylabel('Explosive Play Rate (%)', fontsize=11, color=TU_BLUE)
ax1.tick_params(colors=TU_BLUE)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
bars2 = ax2.bar(pers_df['personnel'], pers_df['mean_epa'],
                color=[TU_BLUE if v >= 0 else RED for v in pers_df['mean_epa']],
                edgecolor='white', linewidth=1.2, width=0.55)
ax2.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax2.axhline(MODEL_BASELINE, color=TU_GOLD, linewidth=1.5, linestyle='-.',
            alpha=0.85, label='Model Baseline (+0.028)')
ax2.legend(fontsize=12, framealpha=0.6, edgecolor='#CCCCCC')
ax2.legend(loc='upper right', bbox_to_anchor=(1.0, 0.95))
for bar, row in zip(bars2, pers_df.itertuples()):
    ax2.text(bar.get_x()+bar.get_width()/2,
             row.mean_epa + 0.008 if row.mean_epa >= 0 else row.mean_epa - 0.008,
             f'μ={row.mean_epa:+.3f}', ha='center',
             va='bottom' if row.mean_epa >= 0 else 'top',
             fontsize=10, fontweight='bold',
             color=TU_BLUE if row.mean_epa >= 0 else RED)
ax2.set_title('Mean EPA per Play\\nby Personnel Grouping (same order)',
              fontsize=18, fontweight='bold', color=TU_BLUE, pad=12)
ax2.set_xlabel('Personnel', fontsize=11, color=TU_BLUE)
ax2.set_ylabel('Average EPA per Play', fontsize=11, color=TU_BLUE)
ax2.tick_params(colors=TU_BLUE)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')

total_exp   = int(exp_pers['explosive'].sum())
total_plays = len(exp_pers)
fig.suptitle(
    f'TU Explosive Play Generation by Personnel\\n'
    f'{total_exp} explosive plays from {total_plays} snaps '
    f'({overall_exp_rate:.1f}% overall)  |  Run ≥12 yds · Pass ≥21 yds  |  Gold = above avg rate',
    fontsize=20, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 13 — EXPLOSIVES
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 13a · Explosive vs. Non-Explosive EPA
> **So what?** If explosives account for the majority of total EPA, the offense lives and dies on big plays. That's fragile — we need consistent drive-building too.
""")

code("""
off_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['epa'].notna()) &
    (tu_df['gn_ls'].notna()) & (tu_df['play_type'].isin(['run','pass']))
].copy()

off_df['explosive'] = (
    ((off_df['play_type'] == 'run')  & (off_df['gn_ls'] >= 12)) |
    ((off_df['play_type'] == 'pass') & (off_df['gn_ls'] >= 21))
)
off_df['explosive_label'] = off_df['explosive'].map({True:'Explosive', False:'Non-Explosive'})

total_epa        = off_df['epa'].sum()
explosive_epa    = off_df[off_df['explosive']]['epa'].sum()
explosive_pct    = explosive_epa / total_epa * 100 if total_epa != 0 else 0
explosive_play_pct = off_df['explosive'].mean() * 100

order   = ['Explosive', 'Non-Explosive']
palette = {'Explosive': TU_GOLD, 'Non-Explosive': TU_BLUE}

fig, axes = plt.subplots(1, 2, figsize=(22, 14), gridspec_kw={'width_ratios': [2, 1]})
fig.patch.set_facecolor(LIGHT_GRAY)

ax1 = axes[0]
ax1.set_facecolor(LIGHT_GRAY)
sns.boxplot(data=off_df, x='explosive_label', y='epa', order=order,
            palette=palette, width=0.45, linewidth=1.3,
            flierprops=dict(marker='o', markersize=3, alpha=0.3,
                            linestyle='none', markeredgewidth=0), ax=ax1)

for i, label in enumerate(order):
    subset      = off_df[off_df['explosive_label'] == label]['epa']
    Q3          = subset.quantile(0.75)
    whisker_top = subset[subset <= Q3 + 1.5*(Q3-subset.quantile(0.25))].max()
    ax1.text(i, whisker_top + 0.15, f'μ={subset.mean():+.3f}',
             ha='center', va='bottom', fontsize=12, fontweight='bold',
             color=TU_GOLD if label == 'Explosive' else TU_BLUE)
    ax1.text(i, ax1.get_ylim()[0]+0.1, f'n={len(subset)}',
             ha='center', va='bottom', fontsize=12, color='gray')

ax1.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax1.set_title('EPA Distribution\\nExplosive vs. Non-Explosive',
              fontsize=18, fontweight='bold', color=TU_BLUE, pad=10)
ax1.set_xlabel('Play Type', fontsize=11, color=TU_BLUE)
ax1.set_ylabel('EPA per Play', fontsize=11, color=TU_BLUE)
ax1.set_ylim(bottom=ax1.get_ylim()[0]-0.5, top=ax1.get_ylim()[1]+1.2)
ax1.tick_params(colors=TU_BLUE)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
contrib_vals   = [explosive_epa, total_epa - explosive_epa]
contrib_colors = [TU_GOLD, TU_BLUE]
bars = ax2.bar(['Explosive\\nPlays', 'Non-Explosive\\nPlays'],
               contrib_vals, color=contrib_colors,
               edgecolor='white', linewidth=1.2, width=0.45)
for bar, val in zip(bars, contrib_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
             f'{val:+.1f} EPA\\n({val/total_epa*100:.1f}%)',
             ha='center', va='bottom', fontsize=12, fontweight='bold',
             color=TU_GOLD if val == explosive_epa else TU_BLUE)

ax2.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax2.set_title('Total EPA Contribution\\nby Play Type',
              fontsize=18, fontweight='bold', color=TU_BLUE, pad=10)
ax2.set_ylabel('Total EPA', fontsize=11, color=TU_BLUE)
ax2.set_ylim(bottom=0, top=max(contrib_vals)*1.3)
ax2.tick_params(colors=TU_BLUE)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')

fig.suptitle(
    f'TU Explosive vs. Non-Explosive Play EPA Analysis\\n'
    f'Explosive plays = {explosive_play_pct:.1f}% of snaps, '
    f'accounting for {explosive_pct:.1f}% of total offensive EPA',
    fontsize=22, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

md("""
## 13b · Explosive Plays by Field Position
> **So what?** If our explosive rate drops near zero past the 40-yard line, defenses are loading up. Motions and tempo changes can disrupt that.
""")

code("""
exp_df = tu_df[
    (tu_df['odk'] == 'o') & (tu_df['play_type'].isin(['run','pass'])) &
    (tu_df['gn_ls'].notna()) & (tu_df['yards_to_go'].notna()) & (tu_df['epa'].notna())
].copy()

exp_df['explosive'] = (
    ((exp_df['play_type'] == 'run')  & (exp_df['gn_ls'] >= 12)) |
    ((exp_df['play_type'] == 'pass') & (exp_df['gn_ls'] >= 21))
)

bins   = [0, 20, 40, 60, 75, 99]
labels = ['Red Zone\\n(1–20)', 'Opp Territory\\n(21–40)',
          'Midfield\\n(41–60)', 'Own Territory\\n(61–75)', 'Own Territory\\n(76–99)']
exp_df['field_zone'] = pd.cut(exp_df['yards_to_go'], bins=bins,
                               labels=labels, include_lowest=True)

zone_agg = (
    exp_df.groupby('field_zone', observed=True)
    .agg(
        total_plays=('explosive','count'), exp_count=('explosive','sum'),
        exp_rate=('explosive','mean'),
        exp_epa_mean=('epa', lambda x: x[exp_df.loc[x.index,'explosive']].mean()),
    )
    .reset_index()
).iloc[::-1].reset_index(drop=True)
zone_agg['exp_rate_pct'] = zone_agg['exp_rate'] * 100
overall_exp_rate = exp_df['explosive'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(26, 14))
fig.patch.set_facecolor(LIGHT_GRAY)

ax1      = axes[0]
ax1_twin = ax1.twinx()
ax1.set_facecolor(LIGHT_GRAY)

bar_colors = [TU_GOLD if r >= overall_exp_rate else TU_BLUE
              for r in zone_agg['exp_rate_pct']]
bars = ax1.bar(zone_agg['field_zone'], zone_agg['exp_count'],
               color=bar_colors, edgecolor='white', linewidth=1.2, width=0.55, zorder=2)

for bar, row in zip(bars, zone_agg.itertuples()):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{int(row.exp_count)}', ha='center', va='bottom', fontsize=14,
             fontweight='bold',
             color=TU_GOLD if row.exp_rate_pct >= overall_exp_rate else TU_BLUE)
    ax1.text(bar.get_x()+bar.get_width()/2, 0.15,
             f'({row.exp_rate_pct:.1f}%)', ha='center', va='bottom',
             fontsize=12, color='gray')

ax1_twin.plot(zone_agg['field_zone'], zone_agg['exp_rate_pct'],
              color=RED, linewidth=2, marker='o', markersize=7, zorder=3)
ax1_twin.axhline(overall_exp_rate, color=RED, linewidth=1.2, linestyle='--', alpha=0.5)
ax1_twin.set_ylabel('Explosive Rate (%)', fontsize=11, color=RED)
ax1_twin.tick_params(axis='y', colors=RED)
ax1_twin.spines[['top']].set_visible(False)
ax1.set_title('Explosive Play Count & Rate\\nby Field Zone',
              fontsize=15, fontweight='bold', color=TU_BLUE, pad=12)
ax1.set_xlabel('Field Zone', fontsize=11, color=TU_BLUE)
ax1.set_ylabel('Number of Explosive Plays', fontsize=11, color=TU_BLUE)
ax1.tick_params(axis='x', colors=TU_BLUE)
ax1.tick_params(axis='y', colors=TU_BLUE)
ax1.spines[['top','right']].set_visible(False)
ax1.spines[['left','bottom']].set_color('#CCCCCC')

ax2 = axes[1]
ax2.set_facecolor(LIGHT_GRAY)
bars2 = ax2.bar(zone_agg['field_zone'], zone_agg['exp_epa_mean'],
                color=[TU_BLUE if v >= 0 else RED for v in zone_agg['exp_epa_mean']],
                edgecolor='white', linewidth=1.2, width=0.55)
ax2.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
for bar, row in zip(bars2, zone_agg.itertuples()):
    if np.isnan(row.exp_epa_mean): continue
    ax2.text(bar.get_x()+bar.get_width()/2,
             row.exp_epa_mean + 0.05 if row.exp_epa_mean >= 0 else row.exp_epa_mean - 0.05,
             f'μ={row.exp_epa_mean:+.3f}', ha='center',
             va='bottom' if row.exp_epa_mean >= 0 else 'top',
             fontsize=12, fontweight='bold',
             color=TU_BLUE if row.exp_epa_mean >= 0 else RED)
ax2.set_title('Mean EPA of Explosive Plays\\nby Field Zone',
              fontsize=15, fontweight='bold', color=TU_BLUE, pad=12)
ax2.set_xlabel('Field Zone', fontsize=11, color=TU_BLUE)
ax2.set_ylabel('Average EPA per Explosive Play', fontsize=11, color=TU_BLUE)
ax2.tick_params(axis='x', colors=TU_BLUE)
ax2.tick_params(axis='y', colors=TU_BLUE)
ax2.spines[['top','right']].set_visible(False)
ax2.spines[['left','bottom']].set_color('#CCCCCC')

total_exp   = int(exp_df['explosive'].sum())
total_plays = len(exp_df)
fig.suptitle(
    f'TU Explosive Play Field Position Analysis\\n'
    f'{total_exp} explosive plays from {total_plays} snaps ({overall_exp_rate:.1f}% overall)  |  '
    f'Gold = above-average explosive rate zone',
    fontsize=20, fontweight='bold', color=TU_BLUE, y=1.02
)
plt.tight_layout()
plt.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 14 — DRIVE NARRATIVE
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 14a · Cumulative EPA by Drive Series
> **So what?** Sharp upward turns = explosive stretches. Flat sections = stalled drives. This is the season's offensive story in one line.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()

series_df = (
    off_df.groupby('off_series')['epa']
    .agg(['sum','mean','count']).reset_index()
    .rename(columns={'sum':'total_epa','mean':'avg_epa','count':'n_plays'})
)
series_df = series_df[series_df['n_plays'] >= 5]
series_df['cumulative_epa'] = series_df['total_epa'].cumsum()

fig, ax = plt.subplots(figsize=(22, 14))
fig.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

ax.fill_between(series_df['off_series'], series_df['cumulative_epa'], 0,
                where=series_df['cumulative_epa'] >= 0, alpha=0.15, color=TU_BLUE)
ax.fill_between(series_df['off_series'], series_df['cumulative_epa'], 0,
                where=series_df['cumulative_epa'] < 0, alpha=0.15, color=RED)
ax.plot(series_df['off_series'], series_df['cumulative_epa'],
        color=TU_BLUE, linewidth=2.5, zorder=3)
ax.scatter(series_df['off_series'], series_df['cumulative_epa'],
           color=TU_GOLD, edgecolors=TU_BLUE, linewidth=1.2, s=45, zorder=4)
ax.axhline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)

final_val = series_df['cumulative_epa'].iloc[-1]
final_ser = series_df['off_series'].iloc[-1]
ax.annotate(f'Final: μ={final_val:+.2f}',
            xy=(final_ser, final_val),
            xytext=(final_ser - 4, final_val + (8 if final_val >= 0 else -8)),
            fontsize=14, fontweight='bold',
            color=TU_BLUE if final_val >= 0 else RED,
            arrowprops=dict(arrowstyle='->', color=TU_GOLD, lw=1.5))

ax.set_title('TU Cumulative Offensive EPA by Drive Series\\n(Season Aggregate)',
             fontsize=24, fontweight='bold', color=TU_BLUE, pad=12)
ax.set_xlabel('Offensive Series Number', fontsize=12, color=TU_BLUE)
ax.set_ylabel('Cumulative EPA', fontsize=12, color=TU_BLUE)
ax.tick_params(colors=TU_BLUE)
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#CCCCCC')
plt.tight_layout()
plt.show()
""")

md("""
## 14b · Cumulative EPA Per Game — Simple (Interactive)
> **So what?** Click any game in the legend to show its drive-by-drive EPA trajectory. Steady climb = sustained efficiency. One spike then flat = explosion-dependent.
""")

code("""
# ── Explosive flags ───────────────────────────────────────────────────────────
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()

off_df['run_explosive']  = (
    (off_df['play_type'] == 'run') &
    (off_df['gn_ls'].notna()) &
    (off_df['gn_ls'] >= 12)
).astype(int)

off_df['pass_explosive'] = (
    (off_df['play_type'] == 'pass') &
    (off_df['gn_ls'].notna()) &
    (off_df['gn_ls'] >= 21)
).astype(int)

# ── Drive outcome classification ──────────────────────────────────────────────
all_plays_s = tu_df.copy()
all_plays_s['play'] = pd.to_numeric(all_plays_s['play'], errors='coerce')
all_plays_s = all_plays_s.sort_values(['game_id', 'play']).reset_index(drop=True)

def classify_outcome_s(group):
    last = group.iloc[-1]
    nxt  = last['next_odk'] if pd.notna(last['next_odk']) else ''
    tod  = last['turnover_on_downs'] if 'turnover_on_downs' in last.index else 0

    if last['score_event'] == 7:
        return 'Touchdown'

    last_idx = all_plays_s[
        (all_plays_s['game_id'] == last['game_id']) &
        (all_plays_s['play'] == last['play'])
    ].index
    if len(last_idx) > 0:
        next_idx = last_idx[0] + 1
        if next_idx < len(all_plays_s):
            next_row = all_plays_s.iloc[next_idx]
            if next_row['game_id'] == last['game_id'] and next_row['score_event'] == 3:
                return 'Field Goal'

    if   tod == 1:   return 'Turnover on Downs'
    elif nxt == 'd': return 'Turnover'
    elif nxt == 'k': return 'Punt'
    else:            return 'Other'

off_plays_s = off_df.copy()
off_plays_s['play'] = pd.to_numeric(off_plays_s['play'], errors='coerce')
off_plays_s = off_plays_s.sort_values(['game_id', 'play'])

series_outcomes_s = (
    off_plays_s
    .groupby(['game_id', 'off_series'], sort=False)
    .apply(classify_outcome_s)
    .reset_index()
    .rename(columns={0: 'outcome'})
)

# ── Aggregate EPA + explosives by game and series ─────────────────────────────
game_series_simple = (
    off_df.groupby(['game_id','opp_name','game_date','team_pts','opp_pts','win','off_series'])
    .agg(
        total_epa      = ('epa',            'sum'),
        n_plays        = ('epa',            'count'),
        run_explosive  = ('run_explosive',   'sum'),
        pass_explosive = ('pass_explosive',  'sum'),
    )
    .reset_index()
)

game_series_simple = game_series_simple[game_series_simple['n_plays'] >= 1]

game_series_simple = game_series_simple.merge(series_outcomes_s, on=['game_id', 'off_series'], how='left')
game_series_simple['outcome'] = game_series_simple['outcome'].fillna('Other')

game_series_simple['cumulative_epa'] = game_series_simple.groupby('game_id')['total_epa'].cumsum()
game_series_simple['game_date_parsed'] = pd.to_datetime(game_series_simple['game_date'], format='%m/%d/%Y')
game_series_simple['year'] = game_series_simple['game_date_parsed'].dt.year

def build_label_simple(row):
    result = 'W' if row['win'] == 1 else 'L'
    return f"{row['game_date']} vs. {row['opp_name']} ({result} {int(row['team_pts'])}-{int(row['opp_pts'])})"
game_series_simple['game_label'] = game_series_simple.apply(build_label_simple, axis=1)

game_meta_simple = (
    game_series_simple[['game_id','game_label','game_date_parsed','year']]
    .drop_duplicates('game_id')
    .sort_values('game_date_parsed', ascending=False)
)

colors_s = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
            '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf',
            '#aec7e8','#ffbb78','#98df8a','#ff9896','#c5b0d5']

fig_s = go.Figure()
current_year_s = None
game_count_s   = 0
game_indices_s = []
trace_idx_s    = 0

for _, meta in game_meta_simple.iterrows():
    game_id = meta['game_id']
    year    = meta['year']
    label   = meta['game_label']
    group   = game_series_simple[game_series_simple['game_id'] == game_id]
    color   = colors_s[game_count_s % len(colors_s)]
    game_count_s += 1

    if year != current_year_s:
        current_year_s = year
        fig_s.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=0, color='rgba(0,0,0,0)'),
            name=f'<b>── {year} ──</b>', showlegend=True,
            hoverinfo='skip', visible=True, legendgroup=f'yr_{year}',
        ))
        trace_idx_s += 1

    hover_text = [
        f"<b>{label}</b><br>"
        f"Series: {int(r.off_series)}<br>"
        f"Series EPA: {r.total_epa:+.3f}<br>"
        f"Cumulative EPA: {r.cumulative_epa:+.3f}<br>"
        f"Plays in series: {int(r.n_plays)}<br>"
        f"Run explosives: {int(r.run_explosive)}  |  Pass explosives: {int(r.pass_explosive)}<br>"
        f"Outcome: {r.outcome}"
        for r in group.itertuples()
    ]

    fig_s.add_trace(go.Scatter(
        x=group['off_series'], y=group['cumulative_epa'],
        mode='lines+markers', name=label,
        legendgroup=game_id,
        line=dict(color=color, width=2.5),
        marker=dict(size=7, color=color, line=dict(color='white', width=1)),
        hovertemplate='%{text}<extra></extra>', text=hover_text,
        visible='legendonly'
    ))
    game_indices_s.append(trace_idx_s)
    trace_idx_s += 1

fig_s.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4, line_width=1.5)
fig_s.update_layout(
    title=dict(
        text='TU Cumulative Offensive EPA by Drive Series<br>'
             '<sup>Click a game in the legend to show it — click again to remove</sup>',
        font=dict(size=15, color=TU_BLUE), x=0.5, xanchor='center'),
    xaxis=dict(title='Offensive Series Number', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD', zeroline=False),
    yaxis=dict(title='Cumulative EPA', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD', zeroline=False),
    legend=dict(title=dict(text='Game', font=dict(color=TU_BLUE, size=10)),
                bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC',
                borderwidth=1, font=dict(size=9), tracegroupgap=2),
    updatemenus=[dict(
        type='buttons', showactive=False,
        x=1.0, xanchor='right', y=1.05, yanchor='top',
        buttons=[
            dict(label='Show All', method='restyle',
                 args=[{'visible': True}, game_indices_s]),
            dict(label='Hide All', method='restyle',
                 args=[{'visible': 'legendonly'}, game_indices_s])
        ],
        bgcolor='white', bordercolor='#CCCCCC', font=dict(color=TU_BLUE, size=9),
        pad=dict(t=2,b=2)
    )],
    plot_bgcolor=LIGHT_GRAY, paper_bgcolor=LIGHT_GRAY, hovermode='closest',
    height=700, margin=dict(t=60, b=30, l=60, r=20),
)
fig_s.show()
""")

md("""
## 14c · Cumulative EPA Per Game — Color by Year / W/L (Interactive)
> **So what?** Toggle color mode to compare seasons or see win/loss patterns. Steady climb = sustained efficiency. One spike then flat = explosion-dependent.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()

game_series_df = (
    off_df.groupby(['game_id','opp_name','game_date','team_pts','opp_pts','win','off_series'])['epa']
    .agg(['sum','count']).reset_index()
    .rename(columns={'sum':'total_epa','count':'n_plays'})
)
game_series_df = game_series_df[game_series_df['n_plays'] >= 1]
game_series_df['cumulative_epa'] = game_series_df.groupby('game_id')['total_epa'].cumsum()
game_series_df['game_date_parsed'] = pd.to_datetime(game_series_df['game_date'], format='%m/%d/%Y')
game_series_df['year'] = game_series_df['game_date_parsed'].dt.year

def build_label(row):
    result = 'W' if row['win'] == 1 else 'L'
    return f"{row['game_date']} vs. {row['opp_name']} ({result} {int(row['team_pts'])}-{int(row['opp_pts'])})"

game_series_df['game_label'] = game_series_df.apply(build_label, axis=1)

game_meta = (
    game_series_df[['game_id','game_label','game_date_parsed','year','win']]
    .drop_duplicates('game_id').sort_values('game_date_parsed', ascending=False)
)
all_years = sorted(game_meta['year'].unique())

YEAR_PALETTE = ['#002868','#C5960C','#1D9E75','#D85A30','#9467bd',
                '#378ADD','#BA7517','#e377c2','#17becf','#8c564b']
year_color_map = {yr: YEAR_PALETTE[i % len(YEAR_PALETTE)] for i, yr in enumerate(all_years)}

fig = go.Figure()
current_year = None
game_indices = []
year_traces  = []
wl_traces    = []
trace_index  = 0

# Set A: colored by year
for _, meta in game_meta.iterrows():
    game_id = meta['game_id']
    year    = meta['year']
    label   = meta['game_label']
    group   = game_series_df[game_series_df['game_id'] == game_id]
    color   = year_color_map[year]

    if year != current_year:
        current_year = year
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=0, color='rgba(0,0,0,0)'),
            name=f'<b>── {year} ──</b>', showlegend=True,
            hoverinfo='skip', visible=True, legendgroup=f'year_header_{year}',
        ))
        trace_index += 1

    hover = [
        f"<b>{label}</b><br>Series: {int(r.off_series)}<br>"
        f"Series EPA: {r.total_epa:+.3f}<br>Cumulative EPA: {r.cumulative_epa:+.3f}"
        for r in group.itertuples()
    ]
    fig.add_trace(go.Scatter(
        x=group['off_series'], y=group['cumulative_epa'],
        mode='lines+markers', name=label, legendgroup=game_id,
        line=dict(color=color, width=2.5),
        marker=dict(size=7, color=color, line=dict(color='white', width=1)),
        hovertemplate='%{text}<extra></extra>', text=hover, visible='legendonly',
    ))
    year_traces.append(trace_index)
    game_indices.append(trace_index)
    trace_index += 1

# Set B: colored by W/L
current_year = None
for _, meta in game_meta.iterrows():
    game_id = meta['game_id']
    year    = meta['year']
    label   = meta['game_label']
    group   = game_series_df[game_series_df['game_id'] == game_id]
    color   = WIN_COLOR if meta['win'] == 1 else LOSS_COLOR

    if year != current_year:
        current_year = year
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers',
            marker=dict(size=0, color='rgba(0,0,0,0)'),
            name=f'<b>── {year} ──</b>', showlegend=True,
            hoverinfo='skip', visible=False, legendgroup=f'wl_header_{year}',
        ))
        trace_index += 1

    hover = [
        f"<b>{label}</b><br>Series: {int(r.off_series)}<br>"
        f"Series EPA: {r.total_epa:+.3f}<br>Cumulative EPA: {r.cumulative_epa:+.3f}"
        for r in group.itertuples()
    ]
    fig.add_trace(go.Scatter(
        x=group['off_series'], y=group['cumulative_epa'],
        mode='lines+markers', name=label, legendgroup=f'wl_{game_id}',
        showlegend=True, line=dict(color=color, width=2.5),
        marker=dict(size=7, color=color, line=dict(color='white', width=1)),
        hovertemplate='%{text}<extra></extra>', text=hover, visible=False,
    ))
    wl_traces.append(trace_index)
    game_indices.append(trace_index)
    trace_index += 1

total_traces = trace_index

def make_visibility(mode):
    vis = []
    for i in range(total_traces):
        if mode == 'year':
            vis.append(False if i in wl_traces else
                       ('legendonly' if i in year_traces else True))
        else:
            vis.append(False if i in year_traces else
                       ('legendonly' if i in wl_traces else True))
    return vis

fig.add_hline(y=0, line_dash='dash', line_color='black', opacity=0.4, line_width=1.5)
fig.update_layout(
    title=dict(
        text='TU Cumulative Offensive EPA by Drive Series<br>'
             '<sup>Click a game to add / remove  ·  Toggle color mode below</sup>',
        font=dict(size=18, color=TU_BLUE), x=0.5, xanchor='center'),
    xaxis=dict(title='Offensive Series Number', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD'),
    yaxis=dict(title='Cumulative EPA', titlefont=dict(color=TU_BLUE),
               tickfont=dict(color=TU_BLUE), gridcolor='#DDDDDD'),
    legend=dict(title=dict(text='Game', font=dict(color=TU_BLUE, size=11)),
                bgcolor='rgba(255,255,255,0.85)', bordercolor='#CCCCCC',
                borderwidth=1, font=dict(size=10), tracegroupgap=2),
    updatemenus=[
        dict(type='buttons', direction='right', showactive=True,
             x=0.5, xanchor='center', y=-0.08, yanchor='top',
             bgcolor='white', bordercolor='#CCCCCC', font=dict(color=TU_BLUE, size=10),
             buttons=[
                 dict(label='Color by Year', method='restyle',
                      args=[{'visible': make_visibility('year')}]),
                 dict(label='Color by W/L', method='restyle',
                      args=[{'visible': make_visibility('wl')}]),
             ]),
        dict(type='buttons', showactive=False, x=1.0, xanchor='right',
             y=-0.08, yanchor='top', bgcolor='white', bordercolor='#CCCCCC',
             font=dict(color=TU_BLUE, size=10),
             buttons=[
                 dict(label='Show All', method='restyle',
                      args=[{'visible': True}, game_indices]),
                 dict(label='Hide All', method='restyle',
                      args=[{'visible': 'legendonly'}, game_indices]),
             ]),
    ],
    plot_bgcolor=LIGHT_GRAY, paper_bgcolor=LIGHT_GRAY,
    height=780, margin=dict(t=120, b=40, l=70, r=20),
)
fig.show()
""")

md("""
## 14c · Drive Process (Interactive)
> **So what?** Each dot is a drive. Touchdowns cluster at high EPA. Turnovers and punts cluster at negative EPA. Use the dropdown to filter by season or game.
""")

code("""
all_plays = tu_df.copy()
all_plays['play'] = pd.to_numeric(all_plays['play'], errors='coerce')
all_plays = all_plays.sort_values(['game_id','play']).reset_index(drop=True)

off_plays = all_plays[(all_plays['odk'] == 'o') & (all_plays['epa'].notna())].copy()
off_plays['year'] = pd.to_datetime(off_plays['game_date'], format='%m/%d/%Y').dt.year
off_plays['drive_key'] = (off_plays['game_id'].astype(str) + '_s' +
                          off_plays['off_series'].astype(str))

def classify_outcome(group):
    last  = group.iloc[-1]
    nxt   = last['next_odk'] if pd.notna(last['next_odk']) else ''
    tod   = last['turnover_on_downs'] if 'turnover_on_downs' in last.index else 0
    if last['score_event'] == 7:
        return 'Touchdown'
    last_idx = all_plays[
        (all_plays['game_id'] == last['game_id']) &
        (all_plays['play'] == last['play'])
    ].index
    if len(last_idx) > 0:
        next_idx = last_idx[0] + 1
        if next_idx < len(all_plays):
            nr = all_plays.iloc[next_idx]
            if nr['game_id'] == last['game_id'] and nr['score_event'] == 3:
                return 'Field Goal'
    if   tod == 1:   return 'Turnover on Downs'
    elif nxt == 'd': return 'Turnover'
    elif nxt == 'k': return 'Punt'
    else:            return 'Other'

records = []
for key, grp in off_plays.groupby('drive_key', sort=False):
    grp = grp.sort_values('play')
    outcome = classify_outcome(grp)
    records.append({
        'drive_key': key, 'outcome': outcome,
        'net_epa': round(grp['epa'].sum(), 3),
        'drive_pts': 7 if outcome=='Touchdown' else 3 if outcome=='Field Goal' else 0,
        'n_plays': len(grp), 'std_epa': round(grp['epa'].std() if len(grp)>1 else 0.0, 3),
        'epa_series': grp['epa'].tolist(),
        'game_id': grp['game_id'].iloc[0],
        'opp_name': grp['opp_name'].iloc[0] if 'opp_name' in grp.columns else '',
        'year': int(grp['year'].iloc[0]),
        'win': int(grp['win'].iloc[0]),
        'team_pts': int(grp['team_pts'].iloc[0]),
        'opp_pts': int(grp['opp_pts'].iloc[0]),
    })

drives_df = pd.DataFrame(records)
all_years = sorted(drives_df['year'].unique(), reverse=True)

OUTCOME_COLORS  = {
    'Touchdown':'#1D9E75', 'Field Goal':'#378ADD', 'Punt':'#888780',
    'Turnover':'#D85A30', 'Turnover on Downs':'#BA7517', 'Other':'#B4B2A9',
}
OUTCOME_SYMBOLS = {
    'Touchdown':'circle', 'Field Goal':'diamond', 'Punt':'square',
    'Turnover':'x', 'Turnover on Downs':'triangle-up', 'Other':'cross',
}
N_NORM = 24

def make_hover(row):
    result = 'W' if row['win'] else 'L'
    return (f"vs {row['opp_name']}  {row['team_pts']}-{row['opp_pts']} ({result})<br>"
            f"Drive {row['drive_key'].split('_s')[-1]}  |  "
            f"Net EPA: {row['net_epa']:+.2f}  |  Pts: {row['drive_pts']}  |  "
            f"Plays: {row['n_plays']}  |  StdDev: {row['std_epa']:.2f}")

scopes = [('All seasons', 'All games', drives_df)]
for yr in all_years:
    yr_df = drives_df[drives_df['year'] == yr]
    scopes.append((str(yr), 'All games', yr_df))
    for _, grow in yr_df.drop_duplicates('game_id').sort_values('game_id').iterrows():
        result = 'W' if grow['win'] else 'L'
        glabel = f"vs {grow['opp_name']}  ({grow['team_pts']}-{grow['opp_pts']} {result})"
        scopes.append((str(yr), glabel, yr_df[yr_df['game_id'] == grow['game_id']]))

fig = make_subplots(rows=2, cols=1, subplot_titles=(' ',' '),
                    vertical_spacing=0.13, row_heights=[0.46, 0.54])

rng    = np.random.default_rng(42)
x_norm = np.linspace(0, 1, N_NORM)
all_traces      = []
scope_trace_map = []
global_idx      = 0

for yr_lbl, gm_lbl, sub in scopes:
    scope_indices = []
    if sub.empty:
        scope_trace_map.append(scope_indices)
        continue

    scatter_added = set()
    for outcome in OUTCOME_COLORS:
        mask = sub['outcome'] == outcome
        if not mask.any(): continue
        grp    = sub[mask]
        jitter = rng.uniform(-0.18, 0.18, size=len(grp))
        labels = [make_hover(r) for _, r in grp.iterrows()]
        t = go.Scatter(
            x=grp['net_epa'], y=grp['drive_pts'] + jitter,
            mode='markers', name=outcome,
            marker=dict(color=OUTCOME_COLORS[outcome], symbol=OUTCOME_SYMBOLS[outcome],
                        size=9, opacity=0.82, line=dict(width=0.6, color='white')),
            text=labels, hovertemplate='%{text}<extra></extra>',
            legendgroup=outcome, showlegend=(outcome not in scatter_added), visible=False,
        )
        all_traces.append((t, 1, 1))
        scope_indices.append(global_idx)
        global_idx += 1
        scatter_added.add(outcome)

    outcome_traj = {o: [] for o in OUTCOME_COLORS}
    for _, row in sub.iterrows():
        epas = row['epa_series']
        if len(epas) < 2: continue
        cum   = np.cumsum(epas)
        x_raw = np.linspace(0, 1, len(cum))
        y_i   = np.interp(x_norm, x_raw, cum)
        outcome_traj[row['outcome']].append(y_i)
        t = go.Scatter(
            x=x_norm, y=np.round(y_i, 3), mode='lines',
            line=dict(color=OUTCOME_COLORS[row['outcome']], width=1),
            opacity=0.25, showlegend=False, hoverinfo='skip',
            legendgroup=row['outcome'], visible=False,
        )
        all_traces.append((t, 2, 1))
        scope_indices.append(global_idx)
        global_idx += 1

    for outcome, tlist in outcome_traj.items():
        if not tlist: continue
        avg = np.mean(tlist, axis=0)
        t = go.Scatter(
            x=x_norm, y=np.round(avg, 3), mode='lines', name=outcome,
            line=dict(color=OUTCOME_COLORS[outcome], width=3),
            showlegend=False, legendgroup=outcome,
            hovertemplate=f'{outcome} avg: %{{y:.2f}}<extra></extra>', visible=False,
        )
        all_traces.append((t, 2, 1))
        scope_indices.append(global_idx)
        global_idx += 1

    scope_trace_map.append(scope_indices)

for t, row, col in all_traces:
    fig.add_trace(t, row=row, col=col)

total_traces = global_idx
for idx in scope_trace_map[0]:
    fig.data[idx].visible = True

def scope_visibility(scope_i):
    vis = [False] * total_traces
    for idx in scope_trace_map[scope_i]:
        vis[idx] = True
    return vis

def subtitle(sub):
    n     = len(sub)
    avg   = sub['net_epa'].mean() if n else 0
    td_r  = (sub['outcome']=='Touchdown').mean()*100 if n else 0
    to_r  = sub['outcome'].isin(['Turnover','Turnover on Downs']).mean()*100 if n else 0
    return f"{n} drives  ·  Avg EPA {avg:+.2f}  ·  TD rate {td_r:.0f}%  ·  TO rate {to_r:.0f}%"

combined_buttons = [dict(
    label='— All Seasons —', method='update',
    args=[{'visible': scope_visibility(0)},
          {'title.text': f'<b>TU Offensive Drive EPA</b>   — All seasons<br>'
           f'<sup style="color:#888780">{subtitle(drives_df)}</sup>'}]
)]
for yr in all_years:
    yr_scope_i = next(i for i,(yl,gl,_) in enumerate(scopes) if yl==str(yr) and gl=='All games')
    yr_sub = scopes[yr_scope_i][2]
    combined_buttons.append(dict(
        label=f'── {yr} ──', method='update',
        args=[{'visible': scope_visibility(yr_scope_i)},
              {'title.text': f'<b>TU Offensive Drive EPA</b>   — {yr}<br>'
               f'<sup style="color:#888780">{subtitle(yr_sub)}</sup>'}]
    ))
    for i,(yr_lbl,gm_lbl,sub) in enumerate(scopes):
        if yr_lbl != str(yr) or gm_lbl == 'All games': continue
        combined_buttons.append(dict(
            label=f'  › {gm_lbl}', method='update',
            args=[{'visible': scope_visibility(i)},
                  {'title.text': f'<b>TU Offensive Drive EPA</b>   — {yr}  ›  {gm_lbl}<br>'
                   f'<sup style="color:#888780">{subtitle(sub)}</sup>'}]
        ))

init_sub = scopes[0][2]
fig.update_layout(
    title=dict(
        text=f'<b>TU Offensive Drive EPA</b>   — All seasons<br>'
             f'<sup style="color:#888780">{subtitle(init_sub)}</sup>',
        font=dict(size=15, color='#2C2C2A'), x=0.0, xanchor='left',
        y=0.97, yanchor='top', pad=dict(l=10)),
    height=1400, plot_bgcolor='#FAFAF8', paper_bgcolor='#FAFAF8',
    font=dict(family='Arial', color='#5F5E5A', size=11),
    legend=dict(title=dict(text='Outcome'), x=1.02, y=0.98,
                bgcolor='rgba(250,250,248,0.9)', bordercolor='#D3D1C7', borderwidth=0.5),
    margin=dict(l=60, r=140, t=120, b=40), hovermode='closest',
    updatemenus=[dict(
        buttons=combined_buttons, direction='up', showactive=True,
        x=0.0, xanchor='left', y=-0.05, yanchor='top',
        bgcolor='#FFFFFF', bordercolor='#D3D1C7', borderwidth=0.5,
        font=dict(size=11, color='#2C2C2A'), pad=dict(r=10),
    )],
    annotations=[
        dict(text='Season / Game', showarrow=False, x=0.0, xanchor='left',
             y=1.135, yanchor='bottom', xref='paper', yref='paper',
             font=dict(size=10, color='#888780')),
        dict(text='<b>Drive Outcomes</b> — Net EPA vs Drive Points',
             showarrow=False, x=0.0, xanchor='left', y=0.97, yanchor='top',
             xref='paper', yref='paper', font=dict(size=12, color='#2C2C2A')),
        dict(text='<b>Drive Process</b> — Cumulative EPA trajectories  '
                  '<i>(thin = individual · bold = outcome average)</i>',
             showarrow=False, x=0.0, xanchor='left', y=0.46, yanchor='top',
             xref='paper', yref='paper', font=dict(size=12, color='#2C2C2A')),
    ],
)
fig.add_vline(x=0, line_width=1, line_dash='dot', line_color='#C4C2BA', row=1, col=1)
fig.add_hline(y=0, line_width=1, line_dash='dot', line_color='#C4C2BA', row=2, col=1)
fig.update_xaxes(title_text='Net Drive EPA', gridcolor='#EEEDFE', title_font_size=11, row=1, col=1)
fig.update_yaxes(title_text='Drive Points', gridcolor='#EEEDFE', title_font_size=11, row=1, col=1)
fig.update_xaxes(title_text='Drive progression  (0 = first play → 1 = last play, normalized)',
                 gridcolor='#EEEDFE', tickvals=[0,0.25,0.5,0.75,1.0],
                 ticktext=['Start','25%','50%','75%','End'], title_font_size=11, row=2, col=1)
fig.update_yaxes(title_text='Cumulative EPA', gridcolor='#EEEDFE', title_font_size=11, row=2, col=1)
fig.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 15 — WIN/LOSS FINGERPRINT
# ══════════════════════════════════════════════════════════════════════════════

md("""
## 15 · The Win/Loss Fingerprint
> **So what?** The biggest gap between the two polygons is your program's pressure point. Track it year-over-year — a closing gap is measurable improvement.
""")

code("""
off_df = tu_df[(tu_df['odk'] == 'o') & (tu_df['epa'].notna())].copy()
off_df['game_date_parsed'] = pd.to_datetime(off_df['game_date'], format='%m/%d/%Y')
off_df['year'] = off_df['game_date_parsed'].dt.year
years = sorted(off_df['year'].unique(), reverse=True)

def safe_mean(series):
    return series.mean() if len(series) > 0 else 0

def build_metrics(df):
    run_df  = df[df['play_type'] == 'run']
    pass_df = df[df['play_type'] == 'pass']
    dn1_df  = df[df['dn'] == 1]
    dn2_df  = df[df['dn'] == 2]
    dn3_df  = df[(df['dn'] == 3) & (df['dist'].notna())].copy()
    rz_df   = df[df['yard_ln'] >= 20]

    if len(dn3_df) > 0:
        dn3_df['converted'] = (
            dn3_df['result'].str.strip().str.lower().isin(
                ['complete','rush','scramble','complete, td','rush, td']
            ) & (dn3_df['gn_ls'] >= dn3_df['dist'])
        )
        conv_rate = dn3_df['converted'].mean()
    else:
        conv_rate = 0

    exp_df  = df[((df['play_type']=='run') & (df['gn_ls'] >= 12)) |
                 ((df['play_type']=='pass') & (df['gn_ls'] >= 21))]
    exp_epa = safe_mean(exp_df['epa']) if df['gn_ls'].notna().any() else 0

    return {
        '1st Down EPA': safe_mean(dn1_df['epa']),
        '2nd Down EPA': safe_mean(dn2_df['epa']),
        '3rd Down EPA': safe_mean(dn3_df['epa']),
        'Run EPA':      safe_mean(run_df['epa']),
        'Pass EPA':     safe_mean(pass_df['epa']),
        'Red Zone EPA': safe_mean(rz_df['epa']),
        'Explosive EPA':safe_mean(exp_df['epa']),
        '3rd Down Conv Rate': conv_rate,
    }

def normalize_metrics(wm, lm):
    cats = list(wm.keys())
    nw, nl = [], []
    for cat in cats:
        vals = [wm[cat], lm[cat]]
        vmin = min(vals) - abs(min(vals))*0.2
        vmax = max(vals) + abs(max(vals))*0.2
        rng  = vmax - vmin if vmax != vmin else 1
        nw.append((wm[cat] - vmin) / rng)
        nl.append((lm[cat] - vmin) / rng)
    return nw, nl

def build_traces(df):
    cats      = ['1st Down EPA','2nd Down EPA','3rd Down EPA',
                 'Run EPA','Pass EPA','Red Zone EPA','Explosive EPA','3rd Down Conv Rate']
    wm        = build_metrics(df[df['win']==1])
    lm        = build_metrics(df[df['win']==0])
    nw, nl    = normalize_metrics(wm, lm)
    rw        = [wm[c] for c in cats]
    rl        = [lm[c] for c in cats]
    cats_c    = cats + [cats[0]]
    nw_c, nl_c = nw+[nw[0]], nl+[nl[0]]
    rw_c, rl_c = rw+[rw[0]], rl+[rl[0]]
    gr        = df.groupby('game_id')['win'].first()
    nwins, nlosses = (gr==1).sum(), (gr==0).sum()
    div       = [abs(w-l) for w,l in zip(rw,rl)]
    max_div   = cats[div.index(max(div))]
    return nw_c, nl_c, rw_c, rl_c, cats_c, nwins, nlosses, max_div

fig         = go.Figure()
year_options= ['All Years'] + [str(y) for y in years]
all_traces  = []
all_titles  = []

WIN_COLOR_R  = '#2ca02c'
LOSS_COLOR_R = '#C0392B'

for option in year_options:
    subset = off_df.copy() if option == 'All Years' else off_df[off_df['year']==int(option)].copy()
    nw, nl, rw, rl, cats, nwins, nlosses, max_div = build_traces(subset)
    all_traces.append((nw, nl, rw, rl, cats))
    all_titles.append(
        f'TU Offensive EPA Win/Loss Fingerprint — {option}<br>'
        f'<sup>Record: {nwins}W – {nlosses}L  |  '
        f'Biggest divergence: <b>{max_div}</b>  |  Hover for raw values</sup>'
    )

nw, nl, rw, rl, cats = all_traces[0]
fig.add_trace(go.Scatterpolar(
    r=nw, theta=cats, fill='toself', fillcolor='rgba(44,160,44,0.15)',
    line=dict(color=WIN_COLOR_R, width=2.5), name='Wins',
    customdata=list(zip(rw, cats)),
    hovertemplate='<b>%{customdata[1]}</b><br>Value: %{customdata[0]:.3f}<extra></extra>'
))
fig.add_trace(go.Scatterpolar(
    r=nl, theta=cats, fill='toself', fillcolor='rgba(192,57,43,0.15)',
    line=dict(color=LOSS_COLOR_R, width=2.5), name='Losses',
    customdata=list(zip(rl, cats)),
    hovertemplate='<b>%{customdata[1]}</b><br>Value: %{customdata[0]:.3f}<extra></extra>'
))

buttons = []
for i, option in enumerate(year_options):
    nw, nl, rw, rl, cats = all_traces[i]
    buttons.append(dict(
        label=option, method='update',
        args=[{'r':[nw,nl], 'theta':[cats,cats],
               'customdata':[list(zip(rw,cats)), list(zip(rl,cats))]},
              {'title.text': all_titles[i]}]
    ))

fig.update_layout(
    polar=dict(
        bgcolor=LIGHT_GRAY,
        radialaxis=dict(visible=True, showticklabels=False, showline=False,
                        gridcolor='#CCCCCC', gridwidth=1, range=[0,1]),
        angularaxis=dict(tickfont=dict(size=12, color=TU_BLUE, family='Arial'),
                         linecolor='#CCCCCC', gridcolor='#CCCCCC')
    ),
    updatemenus=[dict(
        type='dropdown', direction='down',
        x=0.0, xanchor='left', y=1.12, yanchor='top',
        buttons=buttons, bgcolor='white', bordercolor='#CCCCCC',
        font=dict(color=TU_BLUE, size=10), showactive=True, active=0
    )],
    title=dict(text=all_titles[0], font=dict(size=14, color=TU_BLUE), x=0.5, xanchor='center'),
    legend=dict(font=dict(size=12, color=TU_BLUE), bgcolor='rgba(255,255,255,0.85)',
                bordercolor='#CCCCCC', borderwidth=1, x=1.1, y=1.0),
    paper_bgcolor=LIGHT_GRAY, height=720, margin=dict(t=110, b=20, l=40, r=80)
)
fig.show()
""")

# ══════════════════════════════════════════════════════════════════════════════
# CLOSING
# ══════════════════════════════════════════════════════════════════════════════

md("""
---
## Closing Note

This analysis represents the first full cycle of EPA-based performance evaluation
for Trinity University Football. The goal was never to replace the coach's eye —
it is to give it context. The numbers should confirm what great coaches already feel,
and they surface what is easy to miss in the flow of a season.

The next phase of this project will extend this framework to
**defensive EPA analysis** and, ultimately, **expand on offesnive analysis** — giving
us the ability to evaluate not just *what* we are calling, but *what plays* are
executing and at what efficiency.

---
*Analysis conducted using a custom EPA model built on multi-season TUFB
play-by-play data. Model baseline: +0.028 EPA/play. All visualizations
generated in Python (matplotlib, seaborn, plotly).*
""")

# ══════════════════════════════════════════════════════════════════════════════
# REPORT BUILDER — assembles and renders the notebook
# ══════════════════════════════════════════════════════════════════════════════

def build_report(
    excel_file: str  = str(PROCESSED_FILE),
    output_dir: str  = str(DOCS_DIR),
    report_name: str = 'TU_EPA_Offensive_Report',
):
    """
    Load the source data, assemble REPORT_CELLS into a Jupyter notebook,
    execute every code cell directly in the current Python process, then
    convert to a single self-contained HTML file.

    Parameters
    ----------
    excel_file  : Full path to TUFB_EPA_Analysis_FULL_DATA.xlsx on Drive.
    output_dir  : Directory on Google Drive where the HTML will be saved.
    report_name : Base filename (no extension) for both the .ipynb and .html.
    """
    import subprocess, sys, io, base64, traceback, os
    from IPython import get_ipython
    from IPython.core.interactiveshell import InteractiveShell

    # ── 1. Install dependencies if needed ─────────────────────────────────────
    for pkg in ['nbformat', 'nbconvert']:
        try:
            __import__(pkg.replace('-', '_'))
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                                   pkg, '-q', '--quiet'])

    import nbformat

    # ── 2. Build the notebook object ──────────────────────────────────────────
    nb = nbformat.v4.new_notebook()
    nb.metadata['celltoolbar'] = 'Tags'
    nb.metadata['kernelspec'] = {
        'display_name': 'Python 3', 'language': 'python', 'name': 'python3'
    }
    nb.metadata['language_info'] = {'name': 'python', 'version': '3'}

    for cell in REPORT_CELLS:
        if cell['type'] == 'markdown':
            nb.cells.append(nbformat.v4.new_markdown_cell(cell['source']))
        elif cell['type'] == 'code':
            c = nbformat.v4.new_code_cell(cell['source'])
            c.metadata['tags'] = ['hide-input']
            nb.cells.append(c)

    # ── 3. Execute each code cell in the live Colab kernel ────────────────────
    # Rather than spawning a new kernel (which has no Drive mount, no session
    # state), we run each cell's source directly via IPython's run_cell().
    # Outputs are captured by temporarily redirecting the IPython display
    # machinery and matplotlib's figure renderer.
    print("⚙️  Executing notebook cells in current session...")

    ip = get_ipython()
    if ip is None:
        # Fallback for non-interactive environments
        ip = InteractiveShell.instance()

    # ── Load source data and inject into IPython user namespace ──────────────
    # build_report() owns the data loading so it is fully self-contained.
    # If df / tu_df are already in the caller's namespace we reuse them to
    # avoid re-reading the file; otherwise we load from excel_file.
    import inspect
    caller_frame   = inspect.stack()[1].frame
    caller_ns      = {**caller_frame.f_globals, **caller_frame.f_locals}

    if 'df' in caller_ns and 'tu_df' in caller_ns:
        _df    = caller_ns['df']
        _tu_df = caller_ns['tu_df']
        print(f"   ✓ Reusing df and tu_df already in session.")
    else:
        print(f"   ↳ Loading data from {excel_file} ...")

        # Mount Google Drive if reading from Drive and not already mounted
        if str(excel_file).startswith('/content/drive') and not os.path.isdir('/content/drive/MyDrive'):
            print("   ↳ Mounting Google Drive...")
            from google.colab import drive as _drive
            _drive.mount('/content/drive')
            print("   ✓ Drive mounted.")
        else:
            print("   ✓ Drive already mounted.")

        import pandas as _pd
        _df = _pd.read_excel(excel_file)

        # Reshape: broadcast stable team_name / opp_name from offensive rows
        _game_names = (
            _df[_df["odk"] == "o"]
            .groupby("game_id")[["team_name", "opp_name"]]
            .first()
            .reset_index()
        )
        _df = _df.drop(columns=["team_name", "opp_name"])
        _df = _df.merge(_game_names, on="game_id", how="left")

        # Trinity-only subset
        _tu_df = (
            _df[_df['team_name'] == 'TU']
            .sort_values(['game_id', 'play'])
            .reset_index(drop=True)
        )
        print(f"   ✓ Loaded df ({len(_df):,} rows) and tu_df ({len(_tu_df):,} rows).")

    # Push into IPython namespace so all executed cells see them as globals
    ip.user_ns['df']    = _df
    ip.user_ns['tu_df'] = _tu_df
    print(f"   ✓ df and tu_df available in execution namespace.")

    import matplotlib
    import matplotlib.pyplot as plt
    matplotlib.use('Agg')   # non-interactive backend so figures are captured

    import plotly.graph_objects as _go

    # Intercept fig.show() so Plotly figures are captured as HTML
    # instead of being rendered to the Colab output widget (which
    # nbconvert can't see).
    _captured_plotly = []
    _original_show   = _go.Figure.show

    def _capture_show(self, *args, **kwargs):
        _captured_plotly.append(
            self.to_html(full_html=False, include_plotlyjs='cdn')
        )

    _go.Figure.show = _capture_show   # patch

    total_code_cells = sum(1 for c in nb.cells if c['cell_type'] == 'code')
    exec_count = 1

    try:
        for i, cell in enumerate(nb.cells):
            if cell['cell_type'] != 'code':
                continue

            outputs = []
            _captured_plotly.clear()
            plt.close('all')

            # ── Run cell in the live kernel ────────────────────────────────
            result = ip.run_cell(cell.source, store_history=False, silent=False)

            if result.error_in_exec is not None:
                tb = ''.join(traceback.format_exception(
                    type(result.error_in_exec),
                    result.error_in_exec,
                    result.error_in_exec.__traceback__
                ))
                print(f"\n❌  Error in cell {exec_count}:\n{tb}")
                raise result.error_in_exec

            # ── Capture matplotlib figures ─────────────────────────────────
            for fig_num in plt.get_fignums():
                fig = plt.figure(fig_num)
                buf = io.BytesIO()
                fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
                buf.seek(0)
                img_b64 = base64.b64encode(buf.read()).decode('utf-8')
                outputs.append(nbformat.v4.new_output(
                    output_type='display_data',
                    data={'image/png': img_b64, 'text/plain': '<Figure>'},
                    metadata={}
                ))
                plt.close(fig)

            # ── Capture Plotly figures (intercepted from fig.show()) ───────
            for fig_html in _captured_plotly:
                outputs.append(nbformat.v4.new_output(
                    output_type='display_data',
                    data={'text/html': fig_html, 'text/plain': '<Plotly Figure>'},
                    metadata={}
                ))

            # ── Capture plain text / DataFrame output ──────────────────────
            if result.result is not None and not isinstance(result.result, _go.Figure):
                try:
                    outputs.append(nbformat.v4.new_output(
                        output_type='execute_result',
                        execution_count=exec_count,
                        data={'text/plain': repr(result.result)},
                        metadata={}
                    ))
                except Exception:
                    pass

            cell.outputs         = outputs
            cell.execution_count = exec_count
            exec_count += 1
            print(f"   ✓ Cell {exec_count - 1}/{total_code_cells}")

    finally:
        # Always restore the original fig.show() even if we error out
        _go.Figure.show = _original_show

    print(f"✅  All {total_code_cells} cells executed.")

    # ── 4. Save the executed .ipynb (useful for debugging) ────────────────────
    os.makedirs(output_dir, exist_ok=True)
    ipynb_path = os.path.join(output_dir, f'{report_name}.ipynb')
    with open(ipynb_path, 'w') as f:
        nbformat.write(nb, f)
    print(f"📓  Notebook saved → {ipynb_path}")

    # ── 5. Build self-contained Reveal.js slideshow from scratch ────────────
    # We render each cell ourselves — custom builder, no external exporter needed.
    # Reveal.js is fetched reliably and we control the exact slide structure.
    # Plotly figures are already inline HTML. Matplotlib figures are base64 PNGs.
    # Every markdown cell starting with # or ## opens a new slide.

    import base64 as _b64

    # ── 5a. Collect rendered cell content ─────────────────────────────────────
    slides = []   # list of {'title': str, 'blocks': [html_str]}

    def _cell_to_html(cell):
        """Convert an executed notebook cell to an HTML string for the slide."""
        import re as _re
        chunks = []

        if cell.cell_type == 'markdown':
            # Convert basic markdown to HTML (headers, bold, italic, blockquote,
            # lists, tables, horizontal rules, inline code)
            src = cell.source

            # Fenced code blocks
            src = _re.sub(r'```[a-z]*\n(.*?)```', lambda m:
                f'<pre><code>{m.group(1)}</code></pre>', src, flags=_re.DOTALL)

            lines      = src.split('\n')
            html_lines = []
            in_table   = False
            in_list    = False

            for line in lines:
                # Horizontal rule
                if _re.match(r'^-{3,}$', line.strip()):
                    if in_list: html_lines.append('</ul>'); in_list = False
                    html_lines.append('<hr/>')
                    continue
                # Headers
                m = _re.match(r'^(#{1,4})\s+(.*)', line)
                if m:
                    if in_list: html_lines.append('</ul>'); in_list = False
                    level = len(m.group(1))
                    html_lines.append(f'<h{level}>{m.group(2)}</h{level}>')
                    continue
                # Blockquote
                if line.startswith('> '):
                    if in_list: html_lines.append('</ul>'); in_list = False
                    inner = line[2:]
                    inner = _re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', inner)
                    html_lines.append(f'<blockquote>{inner}</blockquote>')
                    continue
                # Table rows
                if '|' in line and line.strip().startswith('|'):
                    if in_list: html_lines.append('</ul>'); in_list = False
                    if not in_table:
                        html_lines.append('<table>')
                        in_table = True
                    cols = [c.strip() for c in line.strip().strip('|').split('|')]
                    # Separator row — skip
                    if all(_re.match(r'^[-:]+$', c) for c in cols if c):
                        continue
                    is_header = not any(
                        '|' in l for l in html_lines[-3:-1]
                        if '<tr>' in l
                    ) and '<table>' in html_lines[-1]
                    tag = 'th' if is_header else 'td'
                    row = ''.join(f'<{tag}>{c}</{tag}>' for c in cols)
                    html_lines.append(f'<tr>{row}</tr>')
                    continue
                else:
                    if in_table:
                        html_lines.append('</table>')
                        in_table = False
                # Bullet list
                if _re.match(r'^[-*]\s+', line):
                    if not in_list:
                        html_lines.append('<ul>')
                        in_list = True
                    inner = line[2:].strip()
                    inner = _re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', inner)
                    inner = _re.sub(r'\*(.+?)\*',   r'<em>\1</em>',       inner)
                    inner = _re.sub(r'`(.+?)`',        r'<code>\1</code>',   inner)
                    html_lines.append(f'<li>{inner}</li>')
                    continue
                else:
                    if in_list:
                        html_lines.append('</ul>')
                        in_list = False
                # Regular paragraph
                if line.strip():
                    line = _re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', line)
                    line = _re.sub(r'\*(.+?)\*',   r'<em>\1</em>',       line)
                    line = _re.sub(r'`(.+?)`',        r'<code>\1</code>',   line)
                    html_lines.append(f'<p>{line}</p>')

            if in_list:  html_lines.append('</ul>')
            if in_table: html_lines.append('</table>')
            chunks.append('\n'.join(html_lines))

        elif cell.cell_type == 'code':
            for output in cell.get('outputs', []):
                data = output.get('data', {})
                # Plotly — already full inline HTML
                if 'text/html' in data:
                    chunks.append(
                        f'''<div class="plotly-wrap">{data["text/html"]}</div>'''
                    )
                # Matplotlib PNG
                elif 'image/png' in data:
                    chunks.append(
                        f'''<img src="data:image/png;base64,{data["image/png"]}"
                              style="width:100%;height:100%;
                                      object-fit:contain;display:block;" />'''
                    )
                # Plain text (print output, describe(), etc.)
                elif 'text/plain' in data:
                    txt = data['text/plain']
                    chunks.append(f'<pre class="output">{txt}</pre>')
        return '\n'.join(chunks)

    # ── 5b. Partition cells into slides ───────────────────────────────────────
    # HIDDEN_SLIDE sentinel → skip slide entirely (setup, reshaping cells)
    # TITLE_SLIDE  sentinel → render as full maroon title card
    current_slide = None
    skip_current  = False

    for cell in nb.cells:
        src = cell.source.strip()

        is_slide_opener = (
            cell.cell_type == 'markdown' and
            (src.startswith('# ') or src.startswith('## ') or src.startswith('---\n#'))
        )

        if is_slide_opener or current_slide is None:
            if current_slide is not None and not skip_current:
                slides.append(current_slide)

            first_line   = src.lstrip('-').strip().split('\n')[0]
            title        = first_line.lstrip('#').strip()
            skip_current    = 'HIDDEN_SLIDE' in title
            is_title_slide  = 'TITLE_SLIDE'  in title

            if not skip_current:
                clean_title = title.replace('TITLE_SLIDE ', '').replace('HIDDEN_SLIDE ', '')
                current_slide = {
                    'title':    clean_title,
                    'blocks':   [_cell_to_html(cell)],
                    'is_title': is_title_slide,
                    'viz_only': False,
                }
            else:
                current_slide = {'title': title, 'blocks': [], 'is_title': False, 'viz_only': False}
        else:
            if not skip_current:
                block_html = _cell_to_html(cell)
                current_slide['blocks'].append(block_html)
                if cell.cell_type == 'code' and block_html.strip():
                    current_slide['viz_only'] = True

    if current_slide is not None and not skip_current:
        slides.append(current_slide)

    # ── 5c. Render slides to Reveal.js HTML ───────────────────────────────────
    def _make_slide(slide):
        body = '\n'.join(b for b in slide['blocks'] if b.strip())

        if slide.get('is_title'):
            body = body.replace('TITLE_SLIDE', '')
            return f"""
        <section class="title-slide">
          {body}
        </section>"""

        elif slide.get('viz_only'):
            blocks = [b for b in slide['blocks'] if b.strip()]
            header = blocks[0] if blocks else ''
            charts = '\n'.join(blocks[1:]) if len(blocks) > 1 else ''
            title  = slide.get('title', '')

            # Apply specialised plotly-wrap class based on slide title
            if '14c' in title or 'Drive Process' in title:
                charts = charts.replace('class="plotly-wrap"', 'class="plotly-wrap tall"')
            elif 'Fingerprint' in title or '15' in title:
                charts = charts.replace('class="plotly-wrap"', 'class="plotly-wrap fingerprint"')

            return f"""
                <section style="padding:0.5rem 1.1rem 0.4rem !important; overflow-y: auto !important; overflow-x: hidden !important;">
                  <div class="slide-header">{header}</div>
                  <div class="slide-chart">{charts}</div>
                </section>"""

        else:
            return f"""
        <section>
          <div class="slide-inner">
            {body}
          </div>
        </section>"""


    slides_html = '\n'.join(_make_slide(s) for s in slides)

    # ── 5d. Full HTML document with embedded Reveal.js from CDN with fallback─
    # Primary: jsDelivr (very reliable). The CSS/JS are small enough that
    # a one-time load caches in the browser for subsequent offline use.
    html_doc = f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>TU Offensive EPA Analysis</title>
  <link rel="stylesheet"
        href="https://cdn.jsdelivr.net/npm/reveal.js@5.1.0/dist/reveal.css"/>
  <style>
    /* ── Root colors ─────────────────────────────────────────────────────── */
    :root {{
      --tu-maroon: #96172E;
      --tu-light:  #f5e8ea;
    }}

    /* ── Maroon outer background ─────────────────────────────────────────── */
    .reveal-viewport, body {{ background: var(--tu-maroon) !important; }}

    /* ── Every slide section fills the full slide area ───────────────────── */
    /* Reveal sets width/height on .slides — we make every section fill it */
    .reveal .slides {{
      text-align: left;
    }}
    .reveal .slides > section {{
      /* Fill the full slide canvas */
      width:  100% !important;
      height: 100% !important;
      top: 0 !important;
      left: 0 !important;
      /* White card with maroon top stripe */
      background: #FFFFFF !important;
      border-top: 10px solid var(--tu-maroon) !important;
      border-radius: 8px !important;
      box-sizing: border-box !important;
      padding: 1rem 1.8rem 0.8rem !important;
      overflow: visible !important;
      /* Flex column so content stacks and fills height */
      display: flex !important;
      flex-direction: column !important;
      align-items: stretch !important;
      justify-content: flex-start !important;
    }}

    /* ── Title slide: full maroon, centred white text ────────────────────── */
    .reveal .slides > section.title-slide {{
      background: var(--tu-maroon) !important;
      border-top: none !important;
      justify-content: center !important;
      align-items: center !important;
      text-align: center !important;
    }}
    .reveal .slides > section.title-slide * {{
      color: #FFFFFF !important;
    }}
    .reveal .slides > section.title-slide h1 {{
      font-size: 3.2em !important;
      font-weight: 900 !important;
      border-bottom: 2px solid rgba(255,255,255,0.4) !important;
      padding-bottom: 0.4rem !important;
      margin-bottom: 0.5rem !important;
      white-space: normal !important;
    }}
    .reveal .slides > section.title-slide h2 {{
      font-size: 2em !important;
      font-weight: 400 !important;
      border-bottom: none !important;
      margin-bottom: 0.3rem !important;
    }}
    .reveal .slides > section.title-slide h3 {{
      font-size: 1.3em !important;
      font-weight: 300 !important;
      color: rgba(255,255,255,0.82) !important;
      margin-bottom: 1.5rem !important;
    }}
    .reveal .slides > section.title-slide hr {{
      border: none !important;
      border-top: 1px solid rgba(255,255,255,0.3) !important;
      width: 50% !important;
      margin: 1rem auto !important;
    }}
    .reveal .slides > section.title-slide p {{
      font-size: 1.1em !important;
      color: rgba(255,255,255,0.9) !important;
    }}

    /* ── Base typography ─────────────────────────────────────────────────── */
    .reveal {{
      font-family: 'Arial', sans-serif;
      font-size: 1.6rem;
      color: #1a1a1a;
    }}
    .reveal h1 {{
      color: var(--tu-maroon); font-size: 2.6em; font-weight: 900;
      text-transform: none; border-bottom: 2px solid var(--tu-maroon);
      padding-bottom: 0.2rem; margin: 0 0 0.6rem 0;
      white-space: nowrap; flex-shrink: 0;
    }}
    .reveal h2 {{
      color: var(--tu-maroon); font-size: 2.1em; font-weight: 800;
      text-transform: none; border-bottom: 2px solid #D4A0A8;
      padding-bottom: 0.15rem; margin: 0 0 0.5rem 0;
      white-space: nowrap; flex-shrink: 0;
    }}
    .reveal h3 {{
      color: #6B0E1E; font-size: 1.4em; font-weight: 700;
      text-transform: none; margin: 0.3rem 0; white-space: nowrap;
    }}
    .reveal p  {{ margin: 0.5rem 0; line-height: 1.7; font-size: 1.2em; flex-shrink: 0; }}
    .reveal ul {{ margin: 0.6rem 0 0.6rem 1.8rem; font-size: 1.2em; }}
    .reveal li {{ margin-bottom: 0.65rem; line-height: 1.7; white-space: normal; }}

    /* ── Blockquote So-what callout ──────────────────────────────────────── */
    .reveal blockquote {{
      border-left: 6px solid var(--tu-maroon);
      background: rgba(150,23,46,0.06);
      padding: 0.6rem 1.2rem;
      font-style: normal; font-weight: 600;
      color: #1a1a1a; width: 100%;
      margin: 0.4rem 0; box-shadow: none;
      font-size: 1.2em; line-height: 1.6;
      white-space: normal; flex-shrink: 0;
    }}
    .reveal blockquote strong {{ color: var(--tu-maroon); }}

    /* ── Tables ──────────────────────────────────────────────────────────── */
    .reveal table {{
      font-size: 0.82em; border-collapse: collapse;
      width: 98%; margin: 0.4rem 0;
    }}
    .reveal th {{
      background: var(--tu-maroon); color: white;
      padding: 0.35rem 0.8rem; white-space: nowrap;
    }}
    .reveal td {{ border: 1px solid #ddd; padding: 0.28rem 0.8rem; }}
    .reveal tr:nth-child(even) {{ background: #f5f0f0; }}

    /* ── Code ────────────────────────────────────────────────────────────── */
    .reveal code {{ background: #f0e8e8; padding: 0.08rem 0.3rem;
                   border-radius: 3px; font-size: 0.8em; color: #6B0E1E; }}
    .reveal pre  {{ background: #fafafa; padding: 0.5rem; font-size: 0.65em; }}
    .reveal hr   {{ border: none; border-top: 1px solid #D4A0A8; margin: 0.5rem 0; flex-shrink:0; }}

    /* ── Text slide inner scroll ─────────────────────────────────────────── */
    .slide-inner {{
      flex: 1 1 auto;
      overflow-y: auto;
      padding: 0;
      min-height: 0;
    }}

    /* ── Viz slide: header stays small, chart expands ───────────────────── */
    .slide-header {{
      flex-shrink: 0;
      margin-bottom: 0.15rem;
    }}
    .slide-header h2 {{
      margin-bottom: 0.05rem;
      padding-bottom: 0.05rem;
      border-bottom-width: 1px;
    }}
    .slide-header blockquote {{
      margin: 0.15rem 0 0 0;
      padding: 0.3rem 0.8rem;
      font-size: 0.88em;
    }}
    .slide-chart {{
      flex: 1 1 auto;
      min-height: 0;
      display: flex;
      flex-direction: column;
      align-items: stretch;
      justify-content: stretch;
      overflow: visible;
    }}

    /* ── Images fill the chart area completely ───────────────────────────── */
    .slide-chart img {{
      width: 100%;
      flex: 1 1 auto;
      min-height: 0;
      object-fit: contain;
      display: block;
    }}

    /* ── Plotly containers ───────────────────────────────────────────────── */
    .plotly-wrap {{
      flex: 1 1 auto;
      min-height: 0;
      width: 100%;
      overflow: visible;
    }}
    .plotly-wrap > div {{
      width:  100% !important;
      height: 100% !important;
    }}

    /* ── Navigation ──────────────────────────────────────────────────────── */
    .reveal .controls      {{ color: rgba(255,255,255,0.85); }}
    .reveal .progress      {{ background: rgba(255,255,255,0.15); height: 4px; }}
    .reveal .progress span {{ background: #FFFFFF; }}
    .reveal .slide-number  {{
      color: #FFFFFF; background: rgba(100,10,20,0.7);
      padding: 2px 8px; border-radius: 3px; font-size: 0.6rem;
    }}
    .output {{ display: none; }}  /* hide plain text outputs */
  </style>
</head>
<body>
<div class="reveal">
  <div class="slides">
{slides_html}
  </div>
</div>
<script src="https://cdn.jsdelivr.net/npm/reveal.js@5.1.0/dist/reveal.js"></script>
<script>
  Reveal.initialize({{
    controls:         true,
    controlsTutorial: false,
    progress:         true,
    slideNumber:      true,
    hash:             true,
    center:           false,
    transition:       'fade',
    transitionSpeed:  'fast',
    width:            1440,
    height:           900,
    margin:           0.04,
    minScale:         0.2,
    maxScale:         2.0,
    disableLayout:    false,
  }});
</script>
</body>
</html>'''

    html_path = os.path.join(output_dir, f'{report_name}.html')
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_doc)

    print(f"📄  Slideshow saved → {html_path}")
    print(f"    Slides: {len(slides)}")
    print(f"\n🏈  Open in a browser — arrow keys or click to navigate.")
    print(f"    F = fullscreen  |  ESC = slide overview  |  ? = keyboard shortcuts")


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == '__main__':
    build_report(report_name='TU_EPA_Offensive_Report_Slideshow_Final')

⚙️  Executing notebook cells in current session...
   ✓ Reusing df and tu_df already in session.
   ✓ df and tu_df available in execution namespace.
df:    20,344 rows × 51 columns
tu_df: 12,186 rows
   ✓ Cell 1/24
TU rows: 12,186

--- EP by odk ---
      count   mean    std    min    25%    50%    75%    max
odk                                                         
d    4803.0  2.069  1.141 -1.394  1.299  1.836  2.850  5.991
k       0.0    NaN    NaN    NaN    NaN    NaN    NaN    NaN
o    5140.0  2.556  1.332 -1.360  1.555  2.367  3.395  6.790

--- EPA by odk ---
      count   mean    std     min    25%    50%    75%    max
odk                                                          
d    4803.0 -0.173  1.267 -10.409 -0.648 -0.294  0.351  6.771
k     130.0  2.846  2.835  -7.000  3.000  3.000  3.000  7.000
o    5140.0  0.057  1.391 -10.073 -0.557 -0.060  0.653  6.315
   ✓ Cell 2/24


   ✓ Cell 3/24
   ✓ Cell 4/24


   ✓ Cell 5/24


   ✓ Cell 6/24


   ✓ Cell 7/24


...

   ✓ Cell 20/24


   ✓ Cell 21/24


   ✓ Cell 22/24


   ✓ Cell 23/24
   ✓ Cell 24/24
✅  All 24 cells executed.
📓  Notebook saved → /root/work/repo/docs/TU_EPA_Offensive_Report_Slideshow_Final.ipynb


📄  Slideshow saved → /root/work/repo/docs/TU_EPA_Offensive_Report_Slideshow_Final.html
    Slides: 27

🏈  Open in a browser — arrow keys or click to navigate.
    F = fullscreen  |  ESC = slide overview  |  ? = keyboard shortcuts
